# Post-Rerank Knowledge Graph Augmentation for Telecom Standards QA
### A controlled quality–cost ablation study: complete, runnable experimental notebook
**Author:** Raju KSR  ·  **Current VS Code configuration:** OpenAI API  ·  **Platform:** VS Code or Google Colab

This single notebook contains the whole experiment of Chapters 3–4 of the thesis: data preparation, independent benchmark
validation, retrieval and reranking, knowledge-graph construction, the controlled comparison of five configurations,
evaluation, statistics, figures and an evidence package. In VS Code, the checked-in project files are authoritative;
the embedded Part 2 cells only create missing files unless `KGRAG_REBUILD_EMBEDDED=1` is explicitly set.

---
## What question does it answer?
Does appending knowledge-graph (KG) triples **after** the reranker improve answers to 3GPP questions, and at what cost,
compared with plain RAG and with putting the KG **before** the reranker?

| Config | What the generator sees |
|---|---|
| **0** | Question only (closed-book reference) |
| **A** | Top-5 vector-retrieval passages (BGE + FAISS) |
| **B** | Top-30 candidates → `bge-reranker-base` → ≤ 5 passages (**control**) |
| **C** | **B's passages unchanged**, plus KG triples seeded only from those passages (**proposed method**) |
| **D** | KG expanded from all 30 candidates, triples reranked jointly with passages (pre-rerank KG) |
| controls | **RANDOM** triples and **UNFILTERED** (no fidelity filter) triples, to show whether the *right* triples matter |

Context budgets: **matched-context** (graph tokens ≤ g × B's evidence tokens, g ∈ {10, 20, 30 %}, ±5 %) and
**overhead-inclusive** (the graph is added on top).

## Data
| Dataset | Role | Licence |
|---|---|---|
| **TeleQnA** (Standards specifications + overview) | question source | MIT |
| **TSpec-LLM**, Release 17, Markdown | retrieval corpus | CC-BY-NC-4.0 (needs a Hugging Face token) |
| **Official 3GPP archive** | spec-version verification only (never used as a corpus) | public |

Raw data is downloaded at run time and stays in your Drive. It is never packaged or redistributed.

## Models
| Role | Model | Used for |
|---|---|---|
| Sampled VS Code run | `gpt-4o-mini` | generator, verifier, evaluator and KG extraction; roles share one model |

No model is trained or fine-tuned. Retrieval uses `BAAI/bge-base-en-v1.5`; reranking `BAAI/bge-reranker-base`;
independent pooling for benchmark validation uses BM25 + `intfloat/e5-base-v2` (the BGE models are deliberately **not** used there).

## How to run
1. To use the G4 GPU while staying in VS Code, install Google's [Colab extension](https://marketplace.visualstudio.com/items?itemName=Google.colab), then choose **Select Kernel → Colab → New Colab Server**, sign in with the account that has G4 access, and select G4 if offered. The local project kernel is the CPU fallback.
2. For a Colab kernel, add `OPENAI_API_KEY` to Colab Secrets and enable notebook access; the remote kernel cannot read your local `.env`. For a local kernel, set it in the ignored project `.env`. Never put keys in notebook source.
3. Upload `kg-rag-thesis-openai-source.zip` to the root of MyDrive so Part 1 syncs the current code/configs; keep the existing raw data under `MyDrive/kg-rag-thesis/data/raw/`. The five Release 17 Markdown files mean no Hugging Face token is needed for this sampled run.
4. Confirm the selected kernel sees the GPU, run Parts 1–3, then the **OpenAI local results run · exploratory only** cell instead of the full thesis stages.
4. The full thesis workflow still has blind-audit gates; the sampled OpenAI cell intentionally skips those gates and is not
   thesis-validated evidence.

The full workflow saves to `MyDrive/kg-rag-thesis/` in Colab or the project folder in VS Code. `Tracker` prints a 12-gate table; a gate turns **VALIDATED** only when its
acceptance criteria were met on real data.

## Map of this notebook
| Part | Content |
|---|---|
| 1 | Setup: Drive, packages, secrets |
| 2 | Embedded project code (configs, source modules, unit tests): collapsed, nothing to edit |
| 3 | Load the code |
| 4 | **The experiment**: eleven stages, each with its purpose, thesis section, outputs and pass criteria |
| 5 | Final evidence package |

# Part 1 · Setup
Selects the Colab Drive path or the local VS Code project, loads local credentials without displaying them, and defines
`put()`, which preserves existing project files by default. **Run this cell every time the session starts.**

In [ ]:
# ===== Part 1 | Setup (run every session)
import os, sys, subprocess
from pathlib import Path

IN_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG'))
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/kg-rag-thesis')
    ROOT.mkdir(parents=True, exist_ok=True)
    ZIP = Path('/content/drive/MyDrive/kg-rag-thesis-openai-source.zip')
    if ZIP.is_file():
        import zipfile
        with zipfile.ZipFile(ZIP) as archive:
            archive.extractall(ROOT.parent)
    if not (ROOT / 'src' / 'kgrag' / 'smoke.py').is_file():
        raise FileNotFoundError(f'Upload the current source archive to {ZIP.parent}: {ZIP.name}')
    for name in ('OPENAI_API_KEY', 'GEMINI_API_KEY', 'HF_TOKEN'):
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            pass
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'kg-rag-thesis', *cwd.parents]
    ROOT = next((p for p in candidates if (p / 'configs' / 'study.yaml').is_file()), None)
    if ROOT is None:
        raise FileNotFoundError(f'Could not locate project root from {cwd}')
    env_file = ROOT / '.env'
    if env_file.is_file():
        for raw_line in env_file.read_text(encoding='utf-8').splitlines():
            line = raw_line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            name, value = line.split('=', 1)
            name, value = name.strip(), value.strip().strip(chr(34)).strip(chr(39))
            if name in {'OPENAI_API_KEY', 'GEMINI_API_KEY', 'HF_TOKEN'} and value:
                os.environ.setdefault(name, value)
    print('VS Code mode: active kernel; automatic installation disabled.')
    print('Credentials available:', {name: bool(os.environ.get(name)) for name in ('OPENAI_API_KEY', 'HF_TOKEN')})

if IN_COLAB or os.environ.get('KGRAG_INSTALL_DEPS') == '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)], check=True)

sys.path.insert(0, str(ROOT / 'src'))
REBUILD_EMBEDDED = os.environ.get('KGRAG_REBUILD_EMBEDDED') == '1'
def put(rel, text):
    """Create missing embedded files; overwrite only when explicitly requested."""
    f = ROOT / rel
    if f.exists() and not REBUILD_EMBEDDED:
        return
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(text, encoding='utf-8')
print('Ready. Project folder:', ROOT)

Mounted at /content/drive


FileNotFoundError: Upload the current source archive to /content/drive/MyDrive: kg-rag-thesis-openai-source.zip

# Part 2 · Embedded project code
The cells below are **collapsed** (double-click a title to read the code). Each one writes a single file into
`MyDrive/kg-rag-thesis/`. They are the only place where the logic lives; the stages in Part 4 call them. Run all of Part 2
(Runtime → *Run selected cells*, or just run them in order); it takes about a minute and is safe to repeat.

### 2.1 Configuration: every thesis parameter in three files
* `study.yaml`: corpus, chunking, retrieval (k = 30), reranker (≤ 5 passages), KG schema (15 predicates, 10 entity types),
  budgets, ablation grid (triples {5,10,20} × hops {1,2} × allocation {10,20,30 %} = 18 variants), seeds.
* `models.yaml`: provider roles, decoding settings, API budget cap and dated price table used for cost reporting.
* `statistical_plan.yaml`: the pre-specified test families (exact McNemar, Wilcoxon, paired bootstrap with 10,000 resamples,
  seed 42, Holm correction within families 2–7; the single confirmatory test is B vs C unadjusted).

In [ ]:
#@title configs/study.yaml
put('configs/study.yaml', r'''# =====================================================================
# Study configuration — POST-RERANK KG AUGMENTATION FOR TELECOM STANDARDS QA
# Every value below is taken from thesis Chapters 3–4 unless marked
# "implementation choice" (documented in reports/methodology_compliance_matrix.csv).
# This file is hashed into every log record; do NOT edit after the test split
# has been evaluated.
# =====================================================================
study_title: "Post-Rerank Knowledge Graph Augmentation for Telecom Standards Question Answering: A Controlled Quality-Cost Ablation Study"
seed: 42                                   # Sec. 3.7 / App. 6

corpus:                                    # Sec. 3.3, Table 3.1, Sec. 4.4
  release: "17"
  hf_repo: "rasoul-nikbakht/TSpec-LLM"
  hf_folder: "3GPP-clean/Rel-17"
  specs:                                   # initial five specifications (Table 4.3)
    "23501": "23_series"
    "23502": "23_series"
    "24501": "24_series"
    "38300": "38_series"
    "38331": "38_series"
  expansion_specs:                         # Sec. 3.3: used ONLY if < 300 questions qualify
    "23401": "23_series"
    "33501": "33_series"
  official_archive: "https://www.3gpp.org/ftp/Specs/archive/"
  chunking:                                # Sec. 3.3 / 4.4.3
    cap_tokens: 500
    target_min_tokens: 300
    min_section_body_tokens: 60            # implementation choice: tiny sections merged forward
    tokenizer_preliminary: "cl100k_base"   # chunk sizing only; budgets use generator tokenizer
    exclude_flags_from_retrieval: ["change_history"]

benchmark:                                 # Sec. 3.3 (a)-(d), Tables 3.2-3.4
  teleqna_url: "https://raw.githubusercontent.com/netop-team/TeleQnA/main/TeleQnA.zip"
  teleqna_password: "teleqnadataset"       # published in the TeleQnA README
  categories: ["Standards specifications", "Standards overview"]
  pooling:
    depth_per_method: 50                   # top-50 BM25 + top-50 E5
    dense_model: "intfloat/e5-base-v2"     # different family from BGE (3.3a)
    forbidden_model_substrings: ["bge"]
  verification:
    window_size: 25                        # implementation choice: RRF-ordered windows of the pool
    max_windows: 4                         # covers the full pool (<=100 passages)
    non_3gpp_tag_pattern: "\\[(IEEE|ETSI(?! *3GPP)|ITU|IETF|O-RAN|ORAN|Wi-?Fi|Bluetooth|LoRa|DVB)[^\\]]*\\]"
  near_duplicate_cosine: 0.90              # X6 rule (char 3-5 gram TF-IDF)
  min_questions: 300
  target_questions: 500
  audit:
    n_included: 100
    n_excluded: 100
    precision_min: 0.90
    kappa_min: 0.70
  split:
    dev_fraction: 0.20
    strata: ["question_type", "hop_type"]

retrieval:                                 # Sec. 3.4 Component 2, 4.7.1
  embedding_model: "BAAI/bge-base-en-v1.5"
  query_instruction: "Represent this sentence for searching relevant passages: "
  top_k: 30
  batch_size: 64
  index: "faiss.IndexFlatIP"               # exact inner product on L2-normalised vectors

reranking:                                 # Sec. 3.4 Component 3, 4.7.2
  model: "BAAI/bge-reranker-base"
  max_passages: 5
  min_passages: 1                          # implementation choice: never an empty context
  max_length: 512
  threshold_rule: "maximise mean per-question F1 of selected passages vs gold passages on DEVELOPMENT split; ties -> lowest threshold"

knowledge_graph:                           # Sec. 3.3, 3.4 Comp. 1 & 4, 4.6
  prompt_version: "kg-extract-v1.0"
  verify_prompt_version: "kg-verify-v1.0"
  entity_types: ["NetworkFunction", "Interface", "Message", "Procedure", "Timer",
                 "Requirement", "InformationElement", "State", "Service", "Other"]
  predicates: ["SENDS", "RECEIVES", "USES_INTERFACE", "CONNECTS", "PART_OF", "TRIGGERS",
               "INITIATES", "STARTS_TIMER", "STOPS_TIMER", "CONTAINS", "PROVIDES",
               "REQUIRES", "SELECTS", "STORES", "IS_A"]
  triple_audit:
    n_triples: 200                         # Sec. 3.6 Triple fidelity
    stratify_by: "predicate"
  relevance_scoring: "cosine(question, rendered statement) with BGE query/passage encoders; ties -> hop asc, triple_id"

experiment:                                # Sec. 3.5, Table 4.5
  configurations: ["0", "A", "B", "C", "D"]
  triple_budgets: [5, 10, 20]
  hop_depths: [1, 2]
  graph_allocations: [0.10, 0.20, 0.30]
  default_variant: {triples: 10, hops: 1, alloc: 0.20}
  n_top_variants: 3
  variant_selection_rule: "development accuracy (desc); ties -> lower mean realised graph tokens; then smaller (alloc, triples, hops)"
  budget_tolerance: 0.05                   # +-5% of T_q
  refill_unused_graph_allocation: false    # Sec. 3.5 worked example (see compliance matrix C-17)
  question_order_seed: 42
  n_runs: 1                                # set to 3 by the determinism pilot (Sec. 3.7) BEFORE the test run
  run_seeds: [42, 43, 44]
  workers: 8
  latency_cache_policy: "retrieval and reranking recomputed per configuration call (no cache); B ranking cache used only for invariance comparison"

pricing_assumption_date: "2026-10-31"
''')

In [ ]:
#@title configs/models.yaml
put('configs/models.yaml', r'''# Model roles (Sec. 3.3b, 3.6, 4.9): four DIFFERENT models for four roles.
# Chosen option: "Balanced" (user decision, 2026-10-31). All hosted via the Gemini API,
# key read from the Colab secret GEMINI_API_KEY. No OpenAI models are used.
# The notebook's first cell checks that each model id is callable with the key and
# records the served `model_version` returned by the API in every log record.
provider: "google-gemini-api"
api_key_secret: "GEMINI_API_KEY"

roles:
  generator:                  # answers for configurations 0/A/B/C/D
    model: "gemini-3.8-flash"
    temperature: 0.0
    thinking_level: "low"
    max_output_tokens: 2048
  verifier:                   # answerability verification (3.3b), triple fidelity (3.6), judged triple relevance
    model: "gemini-3.5-flash"
    temperature: 0.0
    thinking_level: "low"
    max_output_tokens: 4096
  evaluator:                  # RAGAs-protocol faithfulness / answer relevance / context P&R
    model: "gemini-3.1-flash-lite"
    temperature: 0.0
    thinking_level: "low"
    max_output_tokens: 4096
  kg_extractor:               # KG triple extraction (3.4 Component 1)
    model: "gemini-3-flash"
    temperature: 0.0
    thinking_level: "low"
    max_output_tokens: 4096

local_models:
  embedding: "BAAI/bge-base-en-v1.5"
  reranker: "BAAI/bge-reranker-base"
  pooling_dense: "intfloat/e5-base-v2"
  answer_relevance_embedding: "intfloat/e5-base-v2"   # RAGAs answer-relevance cosine (not the experimental retriever)

# USD per 1M tokens. ASSUMPTIONS for cost reporting (Sec. 3.6) — confirm against
# https://ai.google.dev/gemini-api/docs/pricing on the day of the run and edit before Gate 10.
pricing_usd_per_million:
  "gemini-3.8-flash":      {input: 0.75, output: 3.75, source: "ai.google.dev pricing via anotherwrapper.com, 2026-10-11"}
  "gemini-3.5-flash":      {input: 0.75, output: 4.50, source: "cloudprice.net listing, 2026-10-31 (verify)"}
  "gemini-3.1-flash-lite": {input: 0.25, output: 1.50, source: "cloudprice.net listing, 2026-10-31 (verify)"}
  "gemini-3-flash":        {input: 0.50, output: 3.00, source: "cloudprice.net listing, 2026-10-31 (verify)"}
# Thinking tokens are billed as output tokens.
''')

In [ ]:
#@title configs/statistical_plan.yaml
put('configs/statistical_plan.yaml', r'''# Statistical plan — Sec. 3.7, Tables 3.7-3.8, Appendix 6. Frozen before the test split is evaluated.
alpha: 0.05
two_sided: true
bootstrap_resamples: 10000
bootstrap_seed: 42
reference_configuration: "B"
confirmatory:
  comparison: ["B", "C_default_matched"]
  metric: "correct"
  test: "mcnemar_exact"
  adjustment: "none"
families:
  "2_variants":   {metric: correct, test: mcnemar_exact, adjustment: holm, comparisons: "B vs each selected variant (top-3 + default, deduplicated), matched"}
  "3_controls":   {metric: correct, test: mcnemar_exact, adjustment: holm, comparisons: ["C_default_matched vs RANDOM_default_matched", "C_default_matched vs UNFILTERED_default_matched"]}
  "4_placement":  {metric: correct, test: mcnemar_exact, adjustment: holm, comparisons: ["C_default_matched vs D_default_matched"]}
  "5_subgroups":  {metric: correct, test: mcnemar_exact, adjustment: holm, subgroups: ["single-hop", "multi-hop", "retriever-found", "independent-only"]}
  "6_secondary":  {test: wilcoxon, adjustment: holm, metrics: ["faithfulness_full", "faithfulness_passages", "answer_relevance", "context_precision", "context_recall", "citation_validity", "citation_support"]}
  "7_cost":       {test: wilcoxon, adjustment: holm, metrics: ["latency_total_s"], note: "reported as a separate Holm family"}
descriptive_only: ["input_tokens", "output_tokens", "graph_tokens", "cost_usd", "peak_graph_memory_mb"]
repeated_runs: {binary: "majority of 3", continuous: "mean over runs"}
missing_values: "no imputation; pairwise exclusion with n reported"
interpretation: "improvement only if (adjusted) p < 0.05 AND bootstrap 95% CI excludes 0; non-significance is not equivalence"
''')

### 2.2 Core infrastructure
* `core.py`: project paths, hashing, atomic and append-only writes (so a crash never corrupts a result) and the **gate tracker**.
* `env.py`: detects and records the software/hardware environment (thesis Tables 4.1–4.2).
* `llm.py`: the Gemini client: structured JSON output, retries, token and cost accounting, and a local tokenizer
  calibrated against Gemini's `count_tokens` so context budgets are counted with the *generator's* tokenizer.

In [ ]:
#@title kgrag/core.py: paths, IO, gate tracker
put('src/kgrag/core.py', r'''"""Shared infrastructure: configuration, paths, hashing, atomic/append-only IO, stage tracker."""
from __future__ import annotations

import datetime as _dt
import hashlib
import json
import os
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Iterator

import yaml

_LOCK = threading.Lock()


def utc_now() -> str:
    return _dt.datetime.now(_dt.timezone.utc).isoformat()


def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


def sha256_text(s: str) -> str:
    return sha256_bytes(s.encode("utf-8"))


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def stable_hash(obj: Any, n: int = 12) -> str:
    """Deterministic short hash of any JSON-serialisable object."""
    return sha256_text(json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str))[:n]


# ----------------------------------------------------------------------------- config
@dataclass
class Project:
    root: Path
    study: dict
    models: dict
    stats: dict

    @classmethod
    def load(cls, root: str | Path) -> "Project":
        root = Path(root).resolve()
        ld = lambda n: yaml.safe_load((root / "configs" / n).read_text(encoding="utf-8"))
        p = cls(root, ld("study.yaml"), ld("models.yaml"), ld("statistical_plan.yaml"))
        for d in p.dirs().values():
            d.mkdir(parents=True, exist_ok=True)
        return p

    def dirs(self) -> dict[str, Path]:
        r = self.root
        return {k: r / v for k, v in {
            "raw_q": "data/raw/teleqna", "raw_md": "data/raw/tspec", "interim": "data/interim",
            "processed": "data/processed", "manifests": "data/manifests", "logs": "logs",
            "dev": "results/development", "test": "results/heldout", "reports": "reports",
            "figures": "artifacts/figures", "tables": "artifacts/tables", "diagrams": "artifacts/diagrams",
            "audit": "reports/audit", "cache": "data/interim/cache",
        }.items()}

    def path(self, key: str, name: str = "") -> Path:
        d = self.dirs()[key]
        return d / name if name else d

    @property
    def seed(self) -> int:
        return int(self.study["seed"])

    def config_hash(self) -> str:
        return stable_hash({"study": self.study, "models": self.models, "stats": self.stats})


# ----------------------------------------------------------------------------- IO
def atomic_write_text(path: str | Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)


def write_json(path: str | Path, obj: Any) -> None:
    atomic_write_text(path, json.dumps(obj, indent=2, ensure_ascii=False, default=str))


def read_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_jsonl(path: str | Path, rows: Iterable[dict]) -> int:
    lines = [json.dumps(r, ensure_ascii=False, default=str) for r in rows]
    atomic_write_text(path, "\n".join(lines) + ("\n" if lines else ""))
    return len(lines)


def append_jsonl(path: str | Path, row: dict) -> None:
    """Thread-safe append of one record (raw immutable logs are append-only)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    line = json.dumps(row, ensure_ascii=False, default=str) + "\n"
    with _LOCK:
        with open(path, "a", encoding="utf-8") as f:
            f.write(line)
            f.flush()
            os.fsync(f.fileno())


def iter_jsonl(path: str | Path) -> Iterator[dict]:
    path = Path(path)
    if not path.exists():
        return
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue  # a torn final line from an interrupted run is ignored, never repaired


def read_jsonl(path: str | Path) -> list[dict]:
    return list(iter_jsonl(path))


# ----------------------------------------------------------------------------- stage tracker / gates
STATUSES = ["NOT STARTED", "IN PROGRESS", "BLOCKED", "IMPLEMENTED", "TESTED", "VALIDATED"]
GATES = {
    1: "Environment and dependencies verified",
    2: "Dataset files obtained and versions confirmed",
    3: "Corpus preprocessing validated",
    4: "Independent answerability verification completed",
    5: "Manual audit meets criteria and gold benchmark frozen",
    6: "BGE retrieval and cross-encoder reranking validated",
    7: "Knowledge Graph extracted and fidelity audited",
    8: "Configurations 0-D implemented and invariance checks passed",
    9: "Development ablation completed and parameters frozen",
    10: "Held-out experimental runs completed",
    11: "Metrics and statistical tests validated",
    12: "Final artifacts, diagrams and reproducibility package verified",
}


class Tracker:
    """Gate tracker persisted at reports/stage_tracker.json (Master prompt Stage 22)."""

    def __init__(self, project: Project):
        self.path = project.path("reports", "stage_tracker.json")
        self.state = read_json(self.path) if self.path.exists() else {
            str(g): {"title": t, "status": "NOT STARTED", "evidence": [], "updated": None} for g, t in GATES.items()}

    def set(self, gate: int, status: str, evidence: list[str] | str | None = None, note: str = "") -> None:
        assert status in STATUSES, status
        rec = self.state[str(gate)]
        rec["status"] = status
        if evidence:
            rec["evidence"] = sorted(set(rec["evidence"] + ([evidence] if isinstance(evidence, str) else evidence)))
        if note:
            rec["note"] = note
        rec["updated"] = utc_now()
        write_json(self.path, self.state)

    def require(self, gate: int) -> None:
        st = self.state[str(gate)]["status"]
        if st != "VALIDATED":
            raise RuntimeError(f"Gate {gate} ({GATES[gate]}) is '{st}', not VALIDATED. "
                               "Do not advance (Master prompt Stage 22).")

    def table(self) -> str:
        rows = [f"| {g} | {r['title']} | {r['status']} | {r.get('note', '')} |" for g, r in self.state.items()]
        return "| Gate | Title | Status | Note |\n|---|---|---|---|\n" + "\n".join(rows)
''')

In [ ]:
#@title kgrag/env.py: environment capture
put('src/kgrag/env.py', r'''"""Environment capture (Sec. 4.3, Tables 4.1-4.2). Values are DETECTED; unavailable ones are marked."""
from __future__ import annotations

import importlib
import os
import platform
import subprocess
import sys

from .core import Project, utc_now, write_json

PACKAGES = ["numpy", "pandas", "scipy", "statsmodels", "networkx", "faiss", "torch", "transformers",
            "sentence_transformers", "rank_bm25", "tiktoken", "google.genai", "huggingface_hub", "matplotlib", "yaml"]


def _version(mod: str) -> str:
    try:
        m = importlib.import_module(mod)
        return str(getattr(m, "__version__", "installed (no __version__)"))
    except Exception:
        return "NOT INSTALLED"


def _cpu_model() -> str:
    try:
        for line in open("/proc/cpuinfo", encoding="utf-8"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "UNAVAILABLE"


def _ram_gb() -> str:
    try:
        for line in open("/proc/meminfo", encoding="utf-8"):
            if line.startswith("MemTotal"):
                return f"{int(line.split()[1]) / 1024 / 1024:.1f} GB"
    except OSError:
        pass
    return "UNAVAILABLE"


def _gpu() -> tuple[str, str]:
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=20).stdout.strip()
        if out:
            name, mem = [x.strip() for x in out.splitlines()[0].split(",")]
            return name, mem
    except Exception:
        pass
    return "NONE DETECTED", "n/a"


def capture(project: Project) -> dict:
    gpu, vram = _gpu()
    env = {
        "captured_utc": utc_now(),
        "execution_platform": "Google Colab" if "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content") else platform.node(),
        "processor": _cpu_model(), "cpu_count": os.cpu_count(), "system_ram": _ram_gb(),
        "gpu": gpu, "gpu_memory": vram, "operating_system": platform.platform(),
        "python_version": sys.version.split()[0],
        "model_execution": "Hosted (Gemini API) for LLM roles; local GPU/CPU for BGE, reranker, E5",
        "packages": {p: _version(p) for p in PACKAGES},
        "config_hash": project.config_hash(),
    }
    write_json(project.path("reports", "environment.json"), env)
    try:
        frz = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=120).stdout
        (project.root / "requirements.lock.txt").write_text(frz, encoding="utf-8")
    except Exception:
        pass
    return env


def hardware_table_md(env: dict) -> str:
    rows = [("Execution platform", env["execution_platform"]), ("Processor", env["processor"]),
            ("System RAM", env["system_ram"]), ("GPU", env["gpu"]), ("GPU memory", env["gpu_memory"]),
            ("Operating system", env["operating_system"]), ("Python version", env["python_version"]),
            ("Model execution method", env["model_execution"])]
    return "| Hardware parameter | Specification |\n|---|---|\n" + "\n".join(f"| {a} | {b} |" for a, b in rows)
''')

In [ ]:
#@title kgrag/llm.py: Gemini client, cost accounting, tokenizer
put('src/kgrag/llm.py', r'''"""Gemini API client with structured JSON output, retries, usage and cost accounting.

All LLM roles go through `LLMClient.generate_json`. A `FakeLLM` with the same interface is used in unit tests,
so no test ever calls a hosted model and no synthetic output can enter a results file.
"""
from __future__ import annotations

import json
import os
import random
import re
import threading
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Protocol

from .core import append_jsonl, iter_jsonl, sha256_text


@dataclass
class LLMResult:
    data: dict | None
    text: str
    model: str
    model_version: str | None
    input_tokens: int | None
    output_tokens: int | None          # candidates + thoughts (both billed as output)
    thought_tokens: int | None
    latency_s: float
    error: str | None = None
    attempts: int = 1
    cost_usd: float | None = None

    @property
    def ok(self) -> bool:
        return self.error is None and self.data is not None


class LLMClient(Protocol):
    def generate_json(self, role: str, prompt: str, schema: dict | None = None,
                      system: str | None = None, seed: int | None = None) -> LLMResult: ...

    def count_tokens(self, role: str, text: str) -> int: ...

    def model_of(self, role: str) -> str: ...


def parse_json_loose(text: str) -> dict | None:
    """Parse a JSON object from model text; tolerate ```json fences. Never invents content."""
    if not text:
        return None
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t)
    try:
        v = json.loads(t)
        return v if isinstance(v, dict) else {"_list": v}
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", t, re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except json.JSONDecodeError:
                return None
    return None


class GeminiClient:
    """google-genai SDK wrapper. Requires `pip install google-genai`."""

    def __init__(self, models_cfg: dict, api_key: str | None = None, max_retries: int = 6):
        from google import genai  # lazy import
        from google.genai import types
        self._types = types
        key = api_key or os.environ.get(models_cfg.get("api_key_secret", "GEMINI_API_KEY"))
        if not key:
            raise RuntimeError("GEMINI_API_KEY not found (Colab: key icon -> add secret, enable notebook access)")
        self.client = genai.Client(api_key=key)
        self.cfg = models_cfg
        self.max_retries = max_retries

    def model_of(self, role: str) -> str:
        return self.cfg["roles"][role]["model"]

    def price(self, model: str) -> dict | None:
        return self.cfg.get("pricing_usd_per_million", {}).get(model)

    def _config(self, role: str, schema: dict | None, system: str | None, seed: int | None):
        r = self.cfg["roles"][role]
        kw: dict[str, Any] = dict(temperature=r.get("temperature", 0.0),
                                  max_output_tokens=r.get("max_output_tokens", 2048))
        if seed is not None:
            kw["seed"] = int(seed)
        if system:
            kw["system_instruction"] = system
        kw["response_mime_type"] = "application/json"
        if schema is not None:
            kw["response_json_schema"] = schema
        if r.get("thinking_level"):
            kw["thinking_config"] = self._types.ThinkingConfig(thinking_level=r["thinking_level"])
        try:
            return self._types.GenerateContentConfig(**kw)
        except Exception:  # older SDK without response_json_schema / thinking_level
            kw.pop("thinking_config", None)
            if "response_json_schema" in kw:
                kw["response_schema"] = kw.pop("response_json_schema")
            return self._types.GenerateContentConfig(**kw)

    def generate_json(self, role: str, prompt: str, schema: dict | None = None,
                      system: str | None = None, seed: int | None = None) -> LLMResult:
        model = self.model_of(role)
        cfg = self._config(role, schema, system, seed)
        last_err, t0 = None, time.perf_counter()
        for attempt in range(1, self.max_retries + 1):
            t0 = time.perf_counter()
            try:
                resp = self.client.models.generate_content(model=model, contents=prompt, config=cfg)
                lat = time.perf_counter() - t0
                text = resp.text or ""
                um = getattr(resp, "usage_metadata", None)
                inp = getattr(um, "prompt_token_count", None) if um else None
                cand = getattr(um, "candidates_token_count", None) if um else None
                th = getattr(um, "thoughts_token_count", None) if um else None
                out = (cand or 0) + (th or 0) if (cand is not None or th is not None) else None
                data = parse_json_loose(text)
                pr = self.price(model)
                cost = (inp * pr["input"] + out * pr["output"]) / 1e6 if (pr and inp is not None and out is not None) else None
                return LLMResult(data, text, model, getattr(resp, "model_version", None), inp, out, th, lat,
                                 None if data is not None else "unparseable_json", attempt, cost)
            except Exception as e:  # retry on rate limits / transient server errors
                last_err = f"{type(e).__name__}: {e}"
                transient = any(s in last_err for s in ("429", "RESOURCE_EXHAUSTED", "500", "502", "503", "504",
                                                        "UNAVAILABLE", "DEADLINE", "Timeout", "timed out", "Connection"))
                if not transient or attempt == self.max_retries:
                    break
                time.sleep(min(60, 2 ** attempt) + random.random())
        return LLMResult(None, "", model, None, None, None, None, time.perf_counter() - t0, last_err, self.max_retries)

    def count_tokens(self, role: str, text: str) -> int:
        for attempt in range(1, self.max_retries + 1):
            try:
                return int(self.client.models.count_tokens(model=self.model_of(role), contents=text).total_tokens)
            except Exception as e:
                if attempt == self.max_retries:
                    raise
                time.sleep(min(30, 2 ** attempt) + random.random())
        raise RuntimeError("unreachable")

    def list_models(self) -> list[str]:
        return [m.name.split("/")[-1] for m in self.client.models.list()]


class FakeLLM:
    """Deterministic test double. `responder(role, prompt) -> dict`. Token count = whitespace words."""

    def __init__(self, responder: Callable[[str, str], dict], models: dict[str, str] | None = None):
        self.responder = responder
        self.models = models or {"generator": "fake-gen", "verifier": "fake-ver", "evaluator": "fake-eval",
                                 "kg_extractor": "fake-kg"}
        self.calls: list[tuple[str, str]] = []

    def model_of(self, role: str) -> str:
        return self.models[role]

    def generate_json(self, role, prompt, schema=None, system=None, seed=None) -> LLMResult:
        self.calls.append((role, prompt))
        d = self.responder(role, prompt)
        return LLMResult(d, json.dumps(d), self.models[role], "fake-1", len(prompt.split()), 10, 0, 0.001,
                         None if d is not None else "unparseable_json")

    def count_tokens(self, role, text):
        return len(text.split())


def local_gemini_tokenizer(model: str, fallbacks: tuple[str, ...] = ("gemini-3-flash", "gemini-2.5-flash")):
    """google-genai LocalTokenizer (offline SentencePiece). Returns (callable, tokenizer_name) or (None, reason)."""
    try:
        from google.genai.local_tokenizer import LocalTokenizer
    except Exception as e:
        return None, f"LocalTokenizer unavailable: {e}"
    for name in (model, *fallbacks):
        try:
            tok = LocalTokenizer(model_name=name)
            tok.count_tokens("probe")
            return (lambda s, _t=tok: int(_t.count_tokens(s).total_tokens)), name
        except Exception:
            continue
    return None, "no LocalTokenizer model name accepted"


def calibrate_local_tokenizer(llm: LLMClient, local_fn, texts: list[str], role: str = "generator") -> dict:
    """Compare local counts with the hosted count_tokens endpoint on real texts. Use local only if exact on >=99%."""
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(16) as ex:
        api = list(ex.map(lambda t: llm.count_tokens(role, t), texts))
    loc = [local_fn(t) for t in texts]
    exact = sum(a == b for a, b in zip(api, loc)) / max(1, len(texts))
    rel = max((abs(a - b) / a for a, b in zip(api, loc) if a), default=0.0)
    return {"n": len(texts), "exact_match_rate": exact, "max_relative_error": rel, "use_local": exact >= 0.99}


class TokenCounter:
    """Generator-tokenizer token counts (Sec. 3.5: T_q counted with the generation model's tokenizer).

    Counts come from the Gemini count_tokens endpoint, or from the SDK's local tokenizer when calibration shows it is
    identical (>=99% exact on real corpus texts). Cached on disk by text hash.
    """

    def __init__(self, llm: LLMClient, cache_path: str | Path | None, role: str = "generator", local_fn=None,
                 method: str = "api"):
        self.llm, self.role, self.local_fn, self.method = llm, role, local_fn, method if local_fn else "api"
        self.cache_path = Path(cache_path) if cache_path else None
        self._c: dict[str, int] = {}
        self._lock = threading.Lock()
        if self.cache_path:
            for r in iter_jsonl(self.cache_path):
                self._c[r["h"]] = r["n"]
        self.model = llm.model_of(role)

    def __call__(self, text: str) -> int:
        if not text:
            return 0
        if self.local_fn is not None:
            return self.local_fn(text)
        h = sha256_text(self.model + "\x00" + text)[:24]
        with self._lock:
            if h in self._c:
                return self._c[h]
        n = self.llm.count_tokens(self.role, text)
        with self._lock:
            self._c[h] = n
        if self.cache_path:
            append_jsonl(self.cache_path, {"h": h, "n": n})
        return n

    def prefetch(self, texts: list[str], workers: int = 16) -> None:
        if self.local_fn is not None:
            return
        from concurrent.futures import ThreadPoolExecutor
        todo = list({t for t in texts if t})
        with ThreadPoolExecutor(workers) as ex:
            list(ex.map(self, todo))
''')

### 2.3 Data
* `teleqna.py`: downloads and parses TeleQnA (Standards categories only), keeps the keyed option and source tags.
* `tspec.py`: downloads TSpec-LLM Release-17 Markdown, builds the corpus manifest, checks versions against the official 3GPP archive.
* `chunker.py`: clause-aware chunking (your validated **v2.4** logic): splits at clause headings, repairs tables/lists, merges tiny sections.
* `corpus.py`: writes `passages.jsonl`, corpus statistics and the corpus-quality report.

In [ ]:
#@title kgrag/data/teleqna.py
put('src/kgrag/data/teleqna.py', r'''"""TeleQnA acquisition and parsing (Sec. 3.3, Table 3.1). Ported from notebook cells 03-04."""
from __future__ import annotations

import io
import json
import re
import zipfile
from pathlib import Path

from ..core import Project, sha256_file, utc_now, write_json, write_jsonl

TAG = re.compile(r"\[([^\]]+)\]\s*$")


def download(project: Project) -> Path:
    import requests
    import pyzipper
    cfg = project.study["benchmark"]
    raw = project.path("raw_q")
    zpath, jpath = raw / "TeleQnA.zip", raw / "TeleQnA.json"
    if not jpath.exists():
        with requests.get(cfg["teleqna_url"], stream=True, timeout=(25, 180),
                          headers={"User-Agent": "Academic-Research"}) as r:
            r.raise_for_status()
            with zpath.open("wb") as f:
                for part in r.iter_content(1 << 20):
                    f.write(part)
        pwd = cfg["teleqna_password"].encode()
        with pyzipper.AESZipFile(zpath) as a:
            names = [n for n in a.namelist() if n.lower().endswith((".json", ".txt")) and not n.startswith("__MACOSX")]
            pref = next((x for x in names if "teleqna" in x.lower()), names[0])
            try:
                payload = a.read(pref, pwd=pwd)
            except Exception:
                payload = zipfile.ZipFile(zpath).read(pref, pwd=pwd)
        jpath.write_text(json.dumps(json.loads(payload.decode("utf-8-sig")), ensure_ascii=False), encoding="utf-8")
    write_json(project.path("manifests", "teleqna_manifest.json"), {
        "source_url": cfg["teleqna_url"], "repository": "https://github.com/netop-team/TeleQnA",
        "licence": "MIT", "access": "password-protected archive (password in README)",
        "downloaded_utc": utc_now(), "zip_sha256": sha256_file(zpath) if zpath.exists() else None,
        "json_sha256": sha256_file(jpath)})
    return jpath


def parse(records: dict | list) -> list[dict]:
    it = records.items() if isinstance(records, dict) else [(f"index_{i}", v) for i, v in enumerate(records)]
    rows = []
    for key, item in it:
        if not isinstance(item, dict):
            continue
        d = {str(k).strip().lower(): v for k, v in item.items()}
        options = {m.group(1): str(v) for k, v in d.items() if (m := re.fullmatch(r"option\s*(\d+)", k))}
        ans = str(d.get("answer", ""))
        m = re.match(r"^\s*option\s*(\d+)\s*(?::|$)", ans, re.I)
        aid = m.group(1) if m else None
        q = str(d.get("question", ""))
        tag = TAG.search(q)
        rows.append({"question_id": str(key), "question": q, "options": options, "answer_text": ans,
                     "answer_option_id": aid, "explanation": str(d.get("explanation", "")),
                     "category": str(d.get("category", "")).strip(), "source_tag": tag.group(1) if tag else None,
                     "answer_parsed": bool(aid and aid in options)})
    return rows


def load_candidates(project: Project, jpath: Path | None = None) -> list[dict]:
    jpath = jpath or project.path("raw_q", "TeleQnA.json")
    rows = parse(json.loads(Path(jpath).read_text(encoding="utf-8-sig")))
    cats = {c.casefold() for c in project.study["benchmark"]["categories"]}
    cand = [r for r in rows if r["category"].casefold() in cats]
    write_jsonl(project.path("interim", "teleqna_candidates.jsonl"), cand)
    write_json(project.path("manifests", "teleqna_counts.json"), {
        "total_questions": len(rows),
        "by_category": {c: sum(r["category"] == c for r in rows) for c in sorted({r["category"] for r in rows})},
        "candidates": len(cand), "unparsed_answers_in_candidates": sum(not r["answer_parsed"] for r in cand)})
    return cand
''')

In [ ]:
#@title kgrag/data/tspec.py
put('src/kgrag/data/tspec.py', r'''"""TSpec-LLM Release-17 acquisition, corpus manifest and official-3GPP-archive version check (Sec. 3.3, 4.4.1)."""
from __future__ import annotations

import re
from pathlib import Path

from ..core import Project, sha256_file, utc_now, write_json

VERSION_CHARS = "0123456789abcdefghijklmnopqrstuvwxyz"  # 3GPP: a=10, b=11 ... (TS 21.900)


def version_from_filename(name: str) -> str | None:
    """'23501-ha0.md' -> '17.10.0' (3GPP file-naming convention: major/technical/editorial digits)."""
    m = re.match(r"^\d{5}-([0-9a-z])([0-9a-z])([0-9a-z])", name.lower())
    if not m:
        return None
    return ".".join(str(VERSION_CHARS.index(c)) for c in m.groups())


def download(project: Project, specs: dict[str, str] | None = None, token: str | None = None) -> dict:
    from huggingface_hub import HfApi, hf_hub_download
    c = project.study["corpus"]
    specs = specs or c["specs"]
    api, out = HfApi(), {}
    for series in sorted(set(specs.values())):
        folder = f"{c['hf_folder']}/{series}"
        paths = [o.path for o in api.list_repo_tree(c["hf_repo"], path_in_repo=folder, repo_type="dataset",
                                                     recursive=False, token=token)
                 if getattr(o, "path", "").lower().endswith(".md")]
        for spec, s in specs.items():
            if s != series:
                continue
            choices = sorted(p for p in paths if re.match(fr"^{spec}-[^/]+\.md$", Path(p).name, re.I))
            if not choices:
                out[spec] = {"status": "MISSING_IN_DATASET"}
                continue
            pick = choices[-1]  # latest Rel-17 version in TSpec-LLM; verified against archive below
            hf_hub_download(c["hf_repo"], pick, repo_type="dataset", token=token, local_dir=str(project.path("raw_md")))
            out[spec] = {"status": "DOWNLOADED", "hf_path": pick, "alternatives": choices}
    return out


def verify_against_archive(spec: str, version: str, timeout: int = 30) -> dict:
    """Check that <spec>-<code>.zip exists in https://www.3gpp.org/ftp/Specs/archive/<series>/<xx.yyy>/."""
    import requests
    series = f"{spec[:2]}_series"
    dotted = f"{spec[:2]}.{spec[2:]}"
    url = f"https://www.3gpp.org/ftp/Specs/archive/{series}/{dotted}/"
    code = "".join(VERSION_CHARS[int(x)] for x in version.split("."))
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent": "Academic-Research"})
        r.raise_for_status()
        listed = sorted(set(re.findall(fr"{spec}-([0-9a-z]{{3}})\.zip", r.text)))
        rel17 = [x for x in listed if x[0] == "h"]
        return {"archive_url": url, "file_expected": f"{spec}-{code}.zip", "present_in_archive": code in listed,
                "release17_versions_in_archive": rel17, "latest_release17_in_archive": rel17[-1] if rel17 else None,
                "checked_utc": utc_now()}
    except Exception as e:
        return {"archive_url": url, "present_in_archive": None, "error": f"{type(e).__name__}: {e}",
                "checked_utc": utc_now()}


def build_manifest(project: Project, specs: dict[str, str] | None = None, check_archive: bool = True) -> dict:
    c = project.study["corpus"]
    specs = specs or c["specs"]
    files = list(project.path("raw_md").rglob("*.md"))
    entries = []
    for spec, series in specs.items():
        matches = [p for p in files if re.match(fr"^{spec}-", p.name)]
        if len(matches) != 1:
            entries.append({"spec_id": spec, "specification": f"TS {spec[:2]}.{spec[2:]}",
                            "status": "MISSING" if not matches else "AMBIGUOUS", "files": [str(m) for m in matches]})
            continue
        p = matches[0]
        ver = version_from_filename(p.name)
        cover = None
        for line in p.read_text(encoding="utf-8", errors="replace").splitlines()[:600]:
            m = re.search(r"3GPP TS \d+\.\d+ V(\d+\.\d+\.\d+)", line)
            if m:
                cover = m.group(1)
                break
        e = {"spec_id": spec, "specification": f"TS {spec[:2]}.{spec[2:]}", "release": c["release"],
             "version_from_filename": ver, "version_on_cover": cover,
             "filename_cover_consistent": (ver == cover) if cover else None,
             "release_consistent": bool(ver and ver.split(".")[0] == c["release"]),
             "local_path": str(p.relative_to(project.root)), "file_bytes": p.stat().st_size, "sha256": sha256_file(p),
             "source_url": f"https://huggingface.co/datasets/{c['hf_repo']}/blob/main/{c['hf_folder']}/{series}/{p.name}",
             "licence": "CC-BY-NC-4.0 (TSpec-LLM); 3GPP terms of use", "access": "Hugging Face account + accepted dataset conditions",
             "registered_utc": utc_now()}
        if check_archive and ver:
            e["official_archive"] = verify_against_archive(spec, ver)
        e["status"] = ("VERIFIED" if e.get("official_archive", {}).get("present_in_archive") and e["release_consistent"]
                       else "PROVISIONAL")
        entries.append(e)
    man = {"created_utc": utc_now(), "release": c["release"], "documents": entries,
           "mixed_releases": len({e.get("version_from_filename", "x").split(".")[0] for e in entries if e.get("version_from_filename")}) > 1}
    write_json(project.path("manifests", "corpus_manifest.json"), man)
    write_json(project.root / "configs" / "corpus_manifest.json", man)
    return man
''')

In [ ]:
#@title kgrag/data/chunker.py (v2.4)
put('src/kgrag/data/chunker.py', r'''"""Clause-aware chunking of TSpec-LLM Markdown (Sec. 3.3, 4.4.2-4.4.3). Port of notebook CELL 12 (chunker v2.4).

Rules derived from the REAL file structure observed in the Colab diagnostics:
* Real headings are Markdown lines '### 4.2.1 General' (3-6 '#'), numeric clause id required.
* Levels 1-2 may be plain text; accepted only if strictly increasing vs. the previous heading (monotonic),
  capitalised, short, with no trailing page number.
* Contents-page lines end in a page number ('1 Scope 24') and are never headings.
* Cross-references ('38.214 \\[19\\]') are body text; hashed ids that look like a spec number are rejected.
* ASN.1 names ('#### -- *BWP*') are kept in the text but do not start a new clause id.
* Everything before the first '1 Scope' heading (cover, foreword, contents) is dropped and counted.
* Tables and code blocks are split only at row/line boundaries (header repeated); flagged when oversize.
* Sections with a tiny body are merged forward into the next section of the same top-level clause.
Token counting is injected (tiktoken in Colab, whitespace in tests).
"""
from __future__ import annotations

import hashlib
import re
from dataclasses import dataclass, field
from typing import Callable

Counter_ = Callable[[str], int]

HASH_HEAD = re.compile(r"^#{3,6}\s+((?:[A-Z]|\d{1,2})(?:\.\d{1,3}[a-z]?){0,6})\s+(\S.*?)\s*$")
ANNEX_HEAD = re.compile(r"^(?:#{1,8}\s+(Annex\s+[A-Z])\b(.*)|(Annex\s+[A-Z])\s*(\((?:normative|informative)\).*))$")
ASN1_HEAD = re.compile(r"^#{1,8}\s+--")
PLAIN_HEAD = re.compile(r"^ ?(\d{1,2}(?:\.\d{1,3}[a-z]?){0,2})\s+([A-Z][^|+\\]{1,100}?)\s*$")
TRAILING_PAGE = re.compile(r"\s\d{1,4}\s*$")
VERSION = re.compile(r"V(\d+\.\d+\.\d+)(?:\s*\((\d{4}-\d{2})\))?")
TABLE_LINE = re.compile(r"^\s*[|+]")
NORMATIVE = re.compile(r"\b(shall not|shall|should|may)\b", re.I)
SCOPE_HEAD = re.compile(r"^[#\s]*1\s+Scope\s*$")


def _tuple(cid: str) -> tuple[int, ...]:
    return tuple(int(x) for x in re.findall(r"\d+", cid))


def is_spec_ref(cid: str) -> bool:
    parts = cid.split(".")
    return (len(parts) == 2 and parts[0].isdigit() and parts[1][:3].isdigit() and len(parts[1]) == 3
            and int(parts[0]) >= 21)


def parse_version(lines: list[str]) -> tuple[str, str]:
    for line in lines[:600]:
        m = VERSION.search(line) if "3GPP TS" in line else None
        if m:
            return m.group(1), (m.group(2) or "not stated on cover")
    return "unknown", "unknown"


@dataclass
class Section:
    clause_id: str
    heading: str
    lines: list[str] = field(default_factory=list)
    start_line: int = 0
    clause_ids: list[str] = field(default_factory=list)


def split_sections(lines: list[str]) -> tuple[list[Section], dict]:
    sections: list[Section] = []
    stats = {"dropped_front_matter_lines": 0, "plain_headings_accepted": 0, "plain_headings_rejected_nonmonotonic": 0,
             "spec_ref_headings_rejected": 0, "asn1_subsections": 0, "toc_like_lines_skipped": 0,
             "normative_terms_in_dropped_front": 0, "scope_anchor_found": False}
    last: tuple[int, ...] = ()
    cur: Section | None = None
    parent = ""
    body_start = next((i for i, l in enumerate(lines) if SCOPE_HEAD.match(l)), None)
    stats["scope_anchor_found"] = body_start is not None
    for n, raw in enumerate(lines):
        line = raw.rstrip("\n")
        if body_start is not None and n < body_start:
            stats["dropped_front_matter_lines"] += 1
            stats["normative_terms_in_dropped_front"] += len(NORMATIVE.findall(line))
            continue
        m = HASH_HEAD.match(line)
        a = ANNEX_HEAD.match(line)
        if a and TRAILING_PAGE.search(line):  # contents-page annex entry
            a = None
            stats["toc_like_lines_skipped"] += 1
        cid = head = None
        if m:
            if is_spec_ref(m.group(1)):
                stats["spec_ref_headings_rejected"] += 1
            else:
                cid, head = m.group(1), m.group(2)
                last = _tuple(cid) or last
        elif a:
            cid = a.group(1) or a.group(3)
            head = (a.group(2) or a.group(4) or "").strip(" :-")
            last = (999,)  # annexes follow the main body
        elif ASN1_HEAD.match(line) and cur is not None:
            cid, head = parent, line.lstrip("# ").strip()
            stats["asn1_subsections"] += 1
        else:
            p = PLAIN_HEAD.match(line)
            if (p and not TRAILING_PAGE.search(line) and not line.rstrip().endswith((";", ",", ":"))
                    and not TABLE_LINE.match(line)):
                t = _tuple(p.group(1))
                if t and t > last and t[0] <= (last[0] if last else 0) + 1 and last != (999,):
                    cid, head = p.group(1), p.group(2)
                    last = t
                    stats["plain_headings_accepted"] += 1
                elif t:
                    stats["plain_headings_rejected_nonmonotonic"] += 1
            elif p and TRAILING_PAGE.search(line):
                stats["toc_like_lines_skipped"] += 1
        if cid:
            if not ASN1_HEAD.match(line):
                parent = cid
            cur = Section(cid, head, [line], n, [cid])
            sections.append(cur)
        elif cur is None:
            stats["dropped_front_matter_lines"] += 1
            stats["normative_terms_in_dropped_front"] += len(NORMATIVE.findall(line))
        else:
            cur.lines.append(line)
    return sections, stats


def _blocks(lines: list[str]) -> list[tuple[str, bool]]:
    """Split on blank lines; a block is (text, splittable). Tables / indented code are not splittable."""
    out, buf = [], []
    for l in lines + [""]:
        if l.strip():
            buf.append(l)
        elif buf:
            txt = "\n".join(buf)
            out.append((txt, not any(TABLE_LINE.match(x) or x.startswith("    ") for x in buf)))
            buf = []
    return out


def split_prose(text: str, cap: int, count: Counter_) -> list[str]:
    parts, cur = [], ""
    for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9(])", text):
        cand = (cur + " " + s).strip()
        if cur and count(cand) > cap:
            parts.append(cur)
            cur = s
        else:
            cur = cand
    if cur:
        parts.append(cur)
    return parts


def split_unsplittable(text: str, cap: int, count: Counter_) -> list[tuple[str, list[str]]]:
    lines = text.split("\n")
    is_table = sum(1 for l in lines if TABLE_LINE.match(l)) >= max(3, len(lines) // 2)
    header: list[str] = []
    units: list[list[str]] = []
    if is_table:
        buf: list[str] = []
        for l in lines:
            buf.append(l)
            if l.lstrip().startswith("+"):
                units.append(buf)
                buf = []
        if buf:
            units.append(buf)
        for k, u in enumerate(units):
            if any(x.lstrip().startswith("+=") for x in u):
                header = [x for uu in units[: k + 1] for x in uu]
                units = units[k + 1:]
                break
        else:
            header, units = (units[0], units[1:]) if units else ([], [])
        if count("\n".join(header)) > cap // 2:
            units, header = ([header] if header else []) + units, []
    else:
        units = [[l] for l in lines]
    base_flag = "table_split" if is_table else "code_split"
    parts: list[tuple[str, list[str]]] = []
    cur: list[str] = []

    def emit():
        nonlocal cur
        if cur:
            body = "\n".join(header + cur)
            fl = [base_flag]
            if count(body) > cap:
                fl.append("oversize_unsplittable")
            parts.append((body, fl))
            cur = []

    for u in units:
        if cur and count("\n".join(header + cur + u)) > cap:
            emit()
        cur = cur + u
    emit()
    if len(parts) == 1 and "oversize_unsplittable" not in parts[0][1]:
        parts[0] = (parts[0][0], [])
    return parts or [(text, ["oversize_unsplittable"])]


def pack_section(sec: Section, cap: int, count: Counter_) -> list[tuple[str, list[str]]]:
    chunks: list[tuple[str, list[str]]] = []
    cur, flags = "", []

    def flush():
        nonlocal cur, flags
        if cur.strip():
            chunks.append((cur.strip(), flags))
        cur, flags = "", []

    for text, splittable in _blocks(sec.lines):
        n = count(text)
        pieces = [text]
        if n > cap and splittable:
            pieces = split_prose(text, cap, count)
        elif n > cap:
            flush()
            chunks.extend(split_unsplittable(text, cap, count))
            continue
        for piece in pieces:
            cand = (cur + "\n\n" + piece) if cur else piece
            if cur and count(cand) > cap:
                flush()
                cur = piece
            else:
                cur = cand
    flush()
    return chunks


def make_passages(lines: list[str], spec: str, release: str, source_file: str, count: Counter_,
                  cap: int = 500, min_body: int = 60) -> tuple[list[dict], dict]:
    version, vdate = parse_version(lines)
    sections, stats = split_sections(lines)
    merged: list[Section] = []
    carry: Section | None = None
    top = lambda c: c.split(".")[0]
    for i, s in enumerate(sections):
        if carry is not None:
            s = Section(s.clause_id, s.heading, carry.lines + s.lines, carry.start_line, carry.clause_ids + s.clause_ids)
            carry = None
        nxt = sections[i + 1] if i + 1 < len(sections) else None
        if nxt is not None and count("\n".join(s.lines)) < min_body and top(nxt.clause_id) == top(s.clause_id):
            carry = s
            continue
        merged.append(s)
    if carry is not None:
        merged.append(carry)
    passages: list[dict] = []
    for s in merged:
        for k, (text, flags) in enumerate(pack_section(s, cap, count)):
            key = f"{spec}|{version}|{s.clause_id}|{s.start_line}|{k}"
            pid = f"TS{spec}-{s.clause_id}-{hashlib.sha1(key.encode()).hexdigest()[:8]}"
            if "change history" in s.heading.lower():
                flags = flags + ["change_history"]
            passages.append({"passage_id": pid, "specification": f"TS {spec[:2]}.{spec[2:]}", "spec_id": spec,
                             "release": release, "version": version, "version_date": vdate,
                             "clause_id": s.clause_id, "clause_ids": list(dict.fromkeys(s.clause_ids)),
                             "heading": s.heading, "part": k, "text": text, "token_count": count(text),
                             "chunk_flags": flags, "source_file": source_file, "source_start_line": s.start_line + 1})
    stats.update({"sections": len(sections), "passages": len(passages), "version": version, "version_date": vdate})
    return passages, stats


def quality_checks(passages: list[dict], source_text: str | None = None) -> dict:
    ids = [p["passage_id"] for p in passages]
    toks = sorted(p["token_count"] for p in passages)
    q = lambda f: toks[int(f * (len(toks) - 1))] if toks else 0
    rep = {"n_passages": len(passages), "duplicate_ids": len(ids) - len(set(ids)),
           "empty_passages": sum(1 for p in passages if not p["text"].strip()),
           "missing_clause_id": sum(1 for p in passages if not p["clause_id"]),
           "spec_number_like_clause_ids": sum(1 for p in passages if is_spec_ref(p["clause_id"])),
           "lt_100_tokens": sum(1 for t in toks if t < 100), "in_300_500": sum(1 for t in toks if 300 <= t <= 500),
           "gt_500_tokens": sum(1 for t in toks if t > 500),
           "oversize_unsplittable": sum(1 for p in passages if "oversize_unsplittable" in p["chunk_flags"]),
           "token_p05": q(0.05), "token_p50": q(0.5), "token_p95": q(0.95),
           "normative_terms_in_passages": len(NORMATIVE.findall(" ".join(p["text"] for p in passages)))}
    if source_text is not None:
        rep["normative_terms_in_source"] = len(NORMATIVE.findall(source_text))
    return rep
''')

In [ ]:
#@title kgrag/data/corpus.py
put('src/kgrag/data/corpus.py', r'''"""Build passages.jsonl + corpus statistics + quality report from the manifest (Stage 4 outputs)."""
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

from ..core import Project, read_json, read_jsonl, sha256_file, utc_now, write_jsonl, atomic_write_text
from .chunker import make_passages, quality_checks, split_sections

XREF = re.compile(r"(?:clause|subclause|section)\s+(\d+(?:\.\d+){0,6})", re.I)


def tiktoken_counter(name: str = "cl100k_base"):
    import tiktoken
    enc = tiktoken.get_encoding(name)
    return lambda s: len(enc.encode(s, disallowed_special=()))


def build(project: Project, count=None) -> pd.DataFrame:
    count = count or tiktoken_counter(project.study["corpus"]["chunking"]["tokenizer_preliminary"])
    ch = project.study["corpus"]["chunking"]
    man = read_json(project.path("manifests", "corpus_manifest.json"))
    all_p, stats = [], []
    for d in man["documents"]:
        if not d.get("local_path"):
            continue
        p = project.root / d["local_path"]
        raw = p.read_text(encoding="utf-8", errors="replace")
        lines = raw.splitlines()
        passages, st = make_passages(lines, d["spec_id"], project.study["corpus"]["release"], p.name, count,
                                     cap=ch["cap_tokens"], min_body=ch["min_section_body_tokens"])
        q = quality_checks(passages, raw)
        clause_set = {c for x in passages for c in x["clause_ids"]}
        xrefs = [m.group(1) for x in passages for m in XREF.finditer(x["text"])]
        broken = sorted({r for r in xrefs if r not in clause_set and not any(c.startswith(r + ".") for c in clause_set)})
        ver_ok = st["version"] == d.get("version_on_cover") == d.get("version_from_filename")
        stats.append({"specification": d["specification"], "file": p.name, "sha256": sha256_file(p),
                      "version": st["version"], "version_date": st["version_date"], "version_consistent": ver_ok,
                      **{k: v for k, v in st.items() if k not in ("version", "version_date")}, **q,
                      "normative_lost": q["normative_terms_in_source"] - st["normative_terms_in_dropped_front"]
                      - q["normative_terms_in_passages"],
                      "internal_xrefs": len(xrefs), "unresolved_internal_xrefs": len(broken),
                      "unresolved_xref_examples": ";".join(broken[:10])})
        all_p += passages
    write_jsonl(project.path("interim", "passages.jsonl"), all_p)
    df = pd.DataFrame(stats)
    df.to_csv(project.root / "results" / "corpus_statistics.csv", index=False)
    _report(project, df, all_p)
    return df


def _report(project: Project, df: pd.DataFrame, passages: list[dict]) -> None:
    pdf = pd.DataFrame(passages)
    flags = pdf["chunk_flags"].astype(str).value_counts().to_dict()
    n_ret = int((~pdf["chunk_flags"].apply(lambda f: "change_history" in f)).sum())
    md = [f"# Corpus quality report\n\nGenerated {utc_now()} from `data/interim/passages.jsonl` (computed, not estimated).\n",
          f"* Total passages: **{len(pdf)}**; retrieval corpus (change-history annexes excluded): **{n_ret}**",
          f"* Duplicate passage IDs: **{int(pdf.passage_id.duplicated().sum())}**",
          f"* Empty passages: **{int((pdf.text.str.strip() == '').sum())}**",
          f"* Share 300-500 tokens (cl100k, preliminary): **{pdf.token_count.between(300, 500).mean():.3f}**; "
          f"share < 100: **{(pdf.token_count < 100).mean():.3f}**; > 500: **{(pdf.token_count > 500).mean():.3f}** "
          "(tables/code never split mid-row; see flags)",
          f"* Chunk flags: `{flags}`\n",
          "## Per specification\n", df.drop(columns=["unresolved_xref_examples"], errors="ignore").to_markdown(index=False),
          "\n\n`normative_lost` = source terms - terms in dropped front matter - terms in passages; values <= 0 mean no loss "
          "(negative values come from repeated table headers).\n",
          "Unresolved internal cross-references usually point to clauses of other specifications cited inline; they are "
          "reported, not repaired."]
    atomic_write_text(project.path("reports", "corpus_quality_report.md"), "\n".join(md))


def load_passages(project: Project, retrieval_only: bool = True) -> list[dict]:
    ps = read_jsonl(project.path("interim", "passages.jsonl"))
    if retrieval_only:
        ex = set(project.study["corpus"]["chunking"]["exclude_flags_from_retrieval"])
        ps = [p for p in ps if not ex.intersection(p["chunk_flags"])]
    return ps
''')

### 2.4 Independent benchmark validation (Sec. 3.3)
* `pooling.py`: BM25 + E5 top-50 candidate pooling, deliberately independent of the retriever being evaluated.
* `verifier.py`: Gemini answerability verifier. It must return a **verbatim quote** (checked in code) and passes a *separate* entailment check.
* `register.py`: inclusion/exclusion register with reason codes **X1–X7**, near-duplicate removal, the question-flow table (Table 3.4).
* `audit.py`: builds the **blind** audit sheets and scores them (precision, Cohen's κ, Wilson CI).
* `freeze.py`: freezes the gold set and makes the stratified 20 % development / 80 % held-out split, with hashes.

In [ ]:
#@title kgrag/validation/pooling.py
put('src/kgrag/validation/pooling.py', r'''"""Stage 5a: retriever-independent candidate pooling (Sec. 3.3a). BM25 + intfloat/e5-base-v2, top-50 each.

BGE models and the experimental reranker are NEVER used here. The pool is the union of both top-50 lists; reciprocal
rank fusion only orders verification windows, it never decides support.
"""
from __future__ import annotations

import re

import numpy as np

from ..core import Project, write_jsonl


def lexical_tokens(s: str) -> list[str]:
    return re.findall(r"[A-Za-z0-9]+(?:[._-][A-Za-z0-9]+)*", s.lower())


def top_k(scores: np.ndarray, k: int) -> np.ndarray:
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(-scores[idx], kind="stable")]


def rrf(a: int | None, b: int | None, k: int = 60) -> float:
    return sum(1.0 / (k + r) for r in (a, b) if r is not None)


def assert_not_forbidden(name: str, forbidden=("bge",)) -> None:
    if any(f in name.lower() for f in forbidden):
        raise ValueError(f"{name} is forbidden for independent answerability validation (Sec. 3.3a)")


def pool_for_question(bm25_scores: np.ndarray, dense_scores: np.ndarray, pids: list[str], depth: int = 50) -> list[dict]:
    bm = {int(i): r for r, i in enumerate(top_k(bm25_scores, depth), 1)}
    de = {int(i): r for r, i in enumerate(top_k(dense_scores, depth), 1)}
    rows = [{"passage_id": pids[i], "bm25_rank": bm.get(i), "e5_rank": de.get(i),
             "bm25_score": float(bm25_scores[i]), "e5_score": float(dense_scores[i]),
             "sources": [n for n, d in (("bm25", bm), ("e5", de)) if i in d], "rrf": rrf(bm.get(i), de.get(i))}
            for i in set(bm) | set(de)]
    rows.sort(key=lambda r: (-r["rrf"], r["passage_id"]))
    return rows


def build_pools(project: Project, questions: list[dict], passages: list[dict], dense_encoder) -> list[dict]:
    from rank_bm25 import BM25Okapi
    from ..retrieval.encoders import cached_passage_embeddings
    cfg = project.study["benchmark"]["pooling"]
    assert_not_forbidden(dense_encoder.name, tuple(cfg["forbidden_model_substrings"]))
    pids = [p["passage_id"] for p in passages]
    bm25 = BM25Okapi([lexical_tokens(p["text"]) for p in passages])
    P = cached_passage_embeddings(dense_encoder, passages, project.path("interim", "e5_passage_embeddings.npy"),
                                  project.path("interim", "e5_passage_ids.json"))
    qtexts = [q["question"] + " " + " ".join(q["options"].values()) for q in questions]  # options give the keyed terms
    Q = dense_encoder.encode_queries(qtexts)
    out = []
    for q, qt, qv in zip(questions, qtexts, Q):
        pool = pool_for_question(bm25.get_scores(lexical_tokens(qt)), P @ qv, pids, cfg["depth_per_method"])
        out.append({"question_id": q["question_id"], "pool": pool, "n_pool": len(pool),
                    "dense_model": dense_encoder.name, "lexical_model": "rank_bm25.BM25Okapi"})
    write_jsonl(project.path("interim", "candidate_pools.jsonl"), out)
    return out
''')

In [ ]:
#@title kgrag/validation/verifier.py
put('src/kgrag/validation/verifier.py', r'''"""Stage 5b/5c: independent answerability verification (Sec. 3.3b, Tables 3.2-3.3).

Verifier = models.yaml role `verifier` (different from generator, RAGAs evaluator and KG extractor).
Acceptance requires ALL of:
  1. verifier verdict "supports" with cited passage(s) taken from the question's independent pool;
  2. every quotation found verbatim (whitespace/markdown-escape normalised) in its cited passage (script check);
  3. a SEPARATE entailment call confirming the verified quotations entail the keyed option (exact string presence
     does not establish entailment — Master prompt Stage 5);
  4. no ambiguity / general-knowledge flag.
"""
from __future__ import annotations

import re
from concurrent.futures import ThreadPoolExecutor, as_completed

from ..core import Project, append_jsonl, iter_jsonl, sha256_text, utc_now
from ..llm import LLMClient

PROMPT_VERSION = "answerability-v2.0-gemini"
QTYPES = ["factual_lookup", "cross_section_dependency", "procedure_association", "multi_hop_reasoning"]

VERIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "verdict": {"type": "string", "enum": ["supports", "partial", "no_support"]},
        "evidence": {"type": "array", "items": {"type": "object", "properties": {
            "passage_id": {"type": "string"}, "clause_id": {"type": "string"}, "quote": {"type": "string"}},
            "required": ["passage_id", "quote"]}},
        "ambiguous": {"type": "boolean"},
        "general_knowledge": {"type": "boolean"},
        "out_of_corpus": {"type": "boolean"},
        "question_type": {"type": "string", "enum": QTYPES},
        "reason": {"type": "string"},
    },
    "required": ["verdict", "evidence", "ambiguous", "general_knowledge", "out_of_corpus", "question_type", "reason"],
}

ENTAIL_SCHEMA = {"type": "object", "properties": {
    "entailment": {"type": "string", "enum": ["entails", "partial", "not_entailed"]},
    "reason": {"type": "string"}}, "required": ["entailment", "reason"]}

SYSTEM = ("You are an independent verification assistant for a telecommunications-standards benchmark. "
          "You judge ONLY from the 3GPP Release 17 passages supplied. Never use outside knowledge to declare support.")

VERIFY_TEMPLATE = """Decide whether the PASSAGES below (3GPP Release 17 text) contain explicit evidence that the KEYED OPTION is the correct answer to the QUESTION.

QUESTION ({qid}): {question}
OPTIONS:
{options}
KEYED OPTION: option {aid}: {akey}

Rules:
- verdict "supports": the passages explicitly establish the keyed option (and rule out choosing another). List EVERY passage needed (several passages if the answer combines clauses).
- verdict "partial": the passages support only part of the keyed answer, or the key cannot be derived explicitly.
- verdict "no_support": nothing in the passages establishes the keyed option.
- For every evidence item copy a quotation VERBATIM (exact characters, at least 6 words) from that passage, and give its passage_id exactly as shown.
- ambiguous = true if, given the evidence, more than one option is defensible.
- general_knowledge = true ONLY if the question is answerable from widely known general knowledge without any 3GPP text.
- out_of_corpus = true if the question clearly depends on a specification, organisation or release that these 3GPP Release 17 passages do not cover.
- question_type: factual_lookup (single fact/definition), procedure_association (which procedure/step/entity in a procedure), cross_section_dependency (needs two or more clauses of one specification), multi_hop_reasoning (chains facts across clauses or specifications).
- reason: one or two sentences.

PASSAGES:
{passages}"""

ENTAIL_TEMPLATE = """QUESTION: {question}
OPTIONS:
{options}
KEYED OPTION: option {aid}: {akey}

VERIFIED QUOTATIONS (copied verbatim from 3GPP Release 17):
{quotes}

Do these quotations, taken together and without outside knowledge, entail that the keyed option is the correct answer?
"entails" = fully establish it; "partial" = support only part; "not_entailed" = do not establish it."""

_ESC = re.compile(r"\\([\\`*_{}\[\]()#+\-.!|<>~])")


def normalise(s: str) -> str:
    s = _ESC.sub(r"\1", s or "")
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    s = s.replace("–", "-").replace("—", "-").replace(" ", " ")
    s = re.sub(r"[*|+]+", " ", s)  # table borders / emphasis are layout, not content
    return re.sub(r"\s+", " ", s).strip()


def quote_in_passage(quote: str, passage_text: str, min_words: int = 6) -> bool:
    q = normalise(quote)
    return len(q.split()) >= min_words and q in normalise(passage_text)


def render_options(opts: dict) -> str:
    return "\n".join(f"option {k}: {v}" for k, v in sorted(opts.items(), key=lambda kv: int(kv[0])))


def render_passages(pids: list[str], passages: dict) -> str:
    out = []
    for pid in pids:
        p = passages[pid]
        out.append(f"<passage id=\"{pid}\" spec=\"{p['specification']}\" clause=\"{p['clause_id']}\" heading=\"{p['heading'][:80]}\">\n{p['text']}\n</passage>")
    return "\n\n".join(out)


def classify(evidence_pids: list[str], passages: dict, verifier_type: str | None) -> tuple[str, str]:
    """Documented hop/type rule (Sec. 3.3): >1 clause or specification -> multi-hop."""
    clauses = {(passages[p]["spec_id"], passages[p]["clause_id"]) for p in evidence_pids}
    specs = {s for s, _ in clauses}
    if len(clauses) > 1:
        qtype = "multi_hop_reasoning" if (len(specs) > 1 or verifier_type == "multi_hop_reasoning") else "cross_section_dependency"
        return "multi-hop", qtype
    qtype = verifier_type if verifier_type in ("factual_lookup", "procedure_association") else "factual_lookup"
    return "single-hop", qtype


def verify_question(llm: LLMClient, q: dict, pool: list[dict], passages: dict, cfg: dict) -> dict:
    """Run windows over the RRF-ordered pool; returns a complete, auditable record (never raises)."""
    rec = {"question_id": q["question_id"], "prompt_version": PROMPT_VERSION, "verifier_model": llm.model_of("verifier"),
           "started_utc": utc_now(), "calls": [], "status": "ok"}
    tag_re = re.compile(cfg["non_3gpp_tag_pattern"], re.I)
    if tag_re.search(q["question"]):
        rec.update(decision="exclude", reason_code="X2", rule="non-3GPP source tag in question text (no LLM call)",
                   verifier_verdict="not_run", quote_check="n/a")
        return rec
    if not pool:
        rec.update(decision="exclude", reason_code="X1", rule="empty independent pool", verifier_verdict="not_run", quote_check="n/a")
        return rec
    pool_ids = [r["passage_id"] for r in pool]
    in_pool = set(pool_ids)
    W, maxw = cfg["window_size"], cfg["max_windows"]
    opts, aid = render_options(q["options"]), q["answer_option_id"]
    akey = q["options"].get(aid, "")
    carry: list[str] = []
    best = None
    for w in range(min(maxw, (len(pool_ids) + W - 1) // W)):
        window = list(dict.fromkeys(carry + pool_ids[w * W:(w + 1) * W]))
        prompt = VERIFY_TEMPLATE.format(qid=q["question_id"], question=q["question"], options=opts, aid=aid,
                                        akey=akey, passages=render_passages(window, passages))
        r = llm.generate_json("verifier", prompt, VERIFY_SCHEMA, SYSTEM)
        rec["calls"].append({"stage": "verify", "window": w, "n_passages": len(window), "ok": r.ok, "error": r.error,
                             "input_tokens": r.input_tokens, "output_tokens": r.output_tokens, "cost_usd": r.cost_usd,
                             "model_version": r.model_version, "latency_s": r.latency_s,
                             "prompt_sha": sha256_text(prompt)[:16], "response": r.data})
        if not r.ok:
            rec.update(status="error", error=r.error)
            return rec
        d = r.data
        checks = []
        for e in d.get("evidence", []) or []:
            pid = e.get("passage_id", "")
            ok = pid in in_pool and pid in passages and quote_in_passage(e.get("quote", ""), passages[pid]["text"])
            checks.append({**e, "in_pool": pid in in_pool, "quote_found": ok})
        d["_checks"] = checks
        if d["verdict"] == "supports":
            best = (w, d)
            break
        if d["verdict"] == "partial":
            best = best or (w, d)
            carry = [c["passage_id"] for c in checks if c["quote_found"]][:5]
    if best is None:
        last = rec["calls"][-1]["response"]
        rec.update(verifier_verdict="no_support", quote_check="n/a", decision="exclude",
                   reason_code="X2" if last.get("out_of_corpus") else "X1",
                   verifier_reason=last.get("reason"))
        return rec
    _, d = best
    good = [c for c in d["_checks"] if c["quote_found"]]
    rec.update(verifier_verdict=d["verdict"], verifier_reason=d.get("reason"),
               quote_check="pass" if good and len(good) == len(d["_checks"]) else "fail",
               evidence=d["_checks"], verifier_question_type=d.get("question_type"),
               flags={k: d.get(k) for k in ("ambiguous", "general_knowledge", "out_of_corpus")})
    if d["verdict"] == "partial":
        rec.update(decision="exclude", reason_code="X3")
        return rec
    if rec["quote_check"] != "pass":
        rec.update(decision="exclude", reason_code="X7", rule="quotation not found verbatim in cited passage")
        return rec
    if d.get("out_of_corpus"):
        rec.update(decision="exclude", reason_code="X2")
        return rec
    if d.get("ambiguous"):
        rec.update(decision="exclude", reason_code="X4")
        return rec
    if d.get("general_knowledge"):
        rec.update(decision="exclude", reason_code="X5")
        return rec
    quotes = "\n".join(f"[{c['passage_id']} | {passages[c['passage_id']]['specification']} clause "
                       f"{passages[c['passage_id']]['clause_id']}] \"{c['quote']}\"" for c in good)
    eprompt = ENTAIL_TEMPLATE.format(question=q["question"], options=opts, aid=aid, akey=akey, quotes=quotes)
    er = llm.generate_json("verifier", eprompt, ENTAIL_SCHEMA, SYSTEM)
    rec["calls"].append({"stage": "entailment", "ok": er.ok, "error": er.error, "input_tokens": er.input_tokens,
                         "output_tokens": er.output_tokens, "cost_usd": er.cost_usd, "model_version": er.model_version,
                         "latency_s": er.latency_s, "response": er.data})
    if not er.ok:
        rec.update(status="error", error=er.error)
        return rec
    rec["entailment"] = er.data["entailment"]
    if er.data["entailment"] != "entails":
        rec.update(decision="exclude", reason_code="X3" if er.data["entailment"] == "partial" else "X7",
                   rule="separate entailment check failed")
        return rec
    gold = list(dict.fromkeys(c["passage_id"] for c in good))
    hop, qtype = classify(gold, passages, d.get("question_type"))
    rec.update(decision="include", reason_code=None, reference_passage_ids=gold,
               reference_clauses=[f"{passages[p]['specification']} {passages[p]['clause_id']}" for p in gold],
               hop_type=hop, question_type=qtype,
               candidate_sources=sorted({s for r in pool if r["passage_id"] in gold for s in r["sources"]}))
    return rec


def run(project: Project, llm: LLMClient, questions: list[dict], pools: dict[str, list[dict]], passages: dict,
        out_name: str = "verifier_results.jsonl", workers: int | None = None, limit: int | None = None) -> str:
    """Resumable: questions with an 'ok' record for this PROMPT_VERSION are skipped; error records are retried."""
    out = project.path("interim", out_name)
    done = {r["question_id"] for r in iter_jsonl(out) if r.get("status") == "ok" and r.get("prompt_version") == PROMPT_VERSION}
    todo = [q for q in questions if q["question_id"] not in done][: limit or None]
    cfg = project.study["benchmark"]["verification"]
    with ThreadPoolExecutor(workers or project.study["experiment"]["workers"]) as ex:
        futs = [ex.submit(verify_question, llm, q, pools.get(q["question_id"], []), passages, cfg) for q in todo]
        for f in as_completed(futs):
            append_jsonl(out, f.result())
    return str(out)


def latest_records(project: Project, name: str = "verifier_results.jsonl") -> dict[str, dict]:
    recs: dict[str, dict] = {}
    for r in iter_jsonl(project.path("interim", name)):
        if r.get("prompt_version") == PROMPT_VERSION and (r.get("status") == "ok" or r["question_id"] not in recs):
            recs[r["question_id"]] = r
    return recs
''')

In [ ]:
#@title kgrag/validation/register.py
put('src/kgrag/validation/register.py', r'''"""Inclusion/exclusion register (Table 3.2), X6 near-duplicates, question-flow table (Table 3.4)."""
from __future__ import annotations

import re

import numpy as np
import pandas as pd

from ..core import Project, write_json

REASONS = {"X1": "No supporting evidence in corpus", "X2": "Out-of-corpus specification or release",
           "X3": "Partial support only", "X4": "Ambiguous question", "X5": "General-knowledge answer",
           "X6": "Duplicate or near-duplicate", "X7": "Failed verification"}


def _qnum(qid: str) -> int:
    m = re.search(r"(\d+)$", qid)
    return int(m.group(1)) if m else 10**9


def char_tfidf(texts: list[str]) -> np.ndarray:
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=True).fit_transform(texts)
        return X
    except ImportError:  # minimal fallback
        vocab: dict[str, int] = {}
        rows = []
        for t in texts:
            t = f" {t.lower()} "
            grams = [t[i:i + n] for n in (3, 4, 5) for i in range(len(t) - n + 1)]
            rows.append(grams)
            for g in grams:
                vocab.setdefault(g, len(vocab))
        M = np.zeros((len(texts), len(vocab)), dtype="float32")
        for i, gs in enumerate(rows):
            for g in gs:
                M[i, vocab[g]] += 1
        M /= np.linalg.norm(M, axis=1, keepdims=True) + 1e-9
        return M


def near_duplicates(df_inc: pd.DataFrame, threshold: float) -> dict[str, str]:
    """Return {qid: earlier_qid} for included questions duplicating an EARLIER included question (X6 rule):
    char 3-5-gram TF-IDF cosine >= threshold on question text, OR identical gold-passage set and keyed answer text."""
    d = df_inc.assign(_n=df_inc.question_id.map(_qnum)).sort_values("_n").reset_index(drop=True)
    X = char_tfidf(d.question.tolist())
    S = (X @ X.T)
    S = S.toarray() if hasattr(S, "toarray") else np.asarray(S)
    dup: dict[str, str] = {}
    seen_keys: dict[tuple, str] = {}
    for i, r in d.iterrows():
        key = (tuple(sorted(r.reference_passage_ids)), re.sub(r"\s+", " ", str(r.answer_key_text)).strip().lower())
        earlier = [j for j in range(i) if d.question_id[j] not in dup and S[i, j] >= threshold]
        if earlier:
            dup[r.question_id] = d.question_id[earlier[0]]
        elif key in seen_keys:
            dup[r.question_id] = seen_keys[key]
        else:
            seen_keys[key] = r.question_id
    return dup


def build(project: Project, candidates: list[dict], records: dict[str, dict],
          manual: pd.DataFrame | None = None) -> pd.DataFrame:
    rows = []
    for q in candidates:
        r = records.get(q["question_id"])
        if r is None:
            rows.append({"question_id": q["question_id"], "category": q["category"], "decision": "pending",
                         "reason_code": None})
            continue
        rows.append({
            "question_id": q["question_id"], "category": q["category"], "source_tag": q.get("source_tag"),
            "question": q["question"], "answer_option_id": q["answer_option_id"],
            "answer_key_text": q["options"].get(q["answer_option_id"], ""),
            "candidate_sources": ";".join(r.get("candidate_sources", []) or []),
            "verifier_verdict": r.get("verifier_verdict"), "quote_check": r.get("quote_check"),
            "entailment": r.get("entailment"),
            "reference_passage_ids": r.get("reference_passage_ids", []) or [],
            "reference_clauses": r.get("reference_clauses", []) or [],
            "manual_review": "no", "manual_verdict": None,
            "decision": r.get("decision", "pending") if r.get("status") == "ok" else "pending",
            "reason_code": r.get("reason_code"),
            "hop_type": r.get("hop_type"), "question_type": r.get("question_type"),
            "retriever_found": None, "split": None,
            "verifier_model": r.get("verifier_model"), "prompt_version": r.get("prompt_version"),
            "verifier_cost_usd": sum((c.get("cost_usd") or 0) for c in r.get("calls", [])),
        })
    df = pd.DataFrame(rows)
    if manual is not None and len(manual):  # manual-validation corrections (Sec. 3.3c)
        for _, m in manual.iterrows():
            i = df.index[df.question_id == m.question_id]
            if not len(i):
                continue
            i = i[0]
            df.at[i, "manual_review"] = "yes"
            df.at[i, "manual_verdict"] = m.manual_answerable
            if df.at[i, "decision"] == "include" and m.manual_answerable == "N":
                df.at[i, "decision"], df.at[i, "reason_code"] = "exclude", "X7"
            elif df.at[i, "decision"] == "exclude" and m.manual_answerable == "Y" and m.get("manual_passage_ids"):
                df.at[i, "decision"], df.at[i, "reason_code"] = "include", None
                df.at[i, "reference_passage_ids"] = m.manual_passage_ids
                df.at[i, "hop_type"], df.at[i, "question_type"] = m.manual_hop_type, m.manual_question_type
    inc = df[df.decision == "include"]
    if len(inc):
        for qid, first in near_duplicates(inc, project.study["benchmark"]["near_duplicate_cosine"]).items():
            j = df.index[df.question_id == qid][0]
            df.at[j, "decision"], df.at[j, "reason_code"] = "exclude", "X6"
            df.at[j, "duplicate_of"] = first
    return df


def flow_table(df: pd.DataFrame, records: dict[str, dict]) -> pd.DataFrame:
    """Table 3.4 rows computed from the register."""
    pooled = sum(1 for r in records.values() if r.get("verifier_verdict") not in ("not_run", None) or r.get("calls"))
    sup = sum(1 for r in records.values() if r.get("verifier_verdict") == "supports" and r.get("quote_check") == "pass")
    rows = [("TeleQnA Standards specifications and overview", len(df)),
            ("After candidate pooling and non-3GPP-tag screening (verifier run)", pooled),
            ("Verifier verdict 'supports' and quote check passed", sup),
            ("Included after separate entailment check and flags", int(((df.decision == "include") | (df.reason_code == "X6")).sum())),
            ("Final retained set (after X6 and manual corrections)", int((df.decision == "include").sum())),
            ("Pending (not yet verified / API error)", int((df.decision == "pending").sum()))]
    rows += [(f"Excluded {c} — {t}", int((df.reason_code == c).sum())) for c, t in REASONS.items()]
    return pd.DataFrame(rows, columns=["stage", "questions"])


def save(project: Project, df: pd.DataFrame, flow: pd.DataFrame) -> None:
    out = df.drop(columns=["question", "answer_key_text"], errors="ignore")  # IDs + codes only (Sec. 3.8 licensing)
    for c in ("reference_passage_ids", "reference_clauses"):
        if c in out:
            out[c] = out[c].apply(lambda v: ";".join(v) if isinstance(v, list) else v)
    out.to_csv(project.path("reports", "inclusion_exclusion_log.csv"), index=False)
    flow.to_csv(project.path("reports", "question_flow_table.csv"), index=False)
    write_json(project.path("reports", "question_flow_table.json"), flow.to_dict("records"))
''')

In [ ]:
#@title kgrag/validation/audit.py
put('src/kgrag/validation/audit.py', r'''"""Blind manual audits (Sec. 3.3c answerability; Sec. 3.6 triple fidelity) and their scoring.

The reviewer sheet NEVER contains the model verdict. The key (audit_id -> verdict) is written to a separate file.
Human labels are never generated by code: unscored sheets keep the gate BLOCKED.
"""
from __future__ import annotations

import math

import numpy as np
import pandas as pd

from ..core import Project, write_json, utc_now


def cohen_kappa(a: list, b: list) -> float | None:
    a, b = list(a), list(b)
    if not a:
        return None
    cats = sorted(set(a) | set(b))
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pe = sum((a.count(c) / n) * (b.count(c) / n) for c in cats)
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)


def wilson_ci(k: int, n: int, z: float = 1.959964) -> tuple[float | None, float | None]:
    if n == 0:
        return None, None
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return max(0.0, c - h), min(1.0, c + h)


def stratified_sample(df: pd.DataFrame, n: int, strata: str, seed: int) -> pd.DataFrame:
    """Proportional allocation (largest remainder) within `strata`; all rows if fewer than n (thesis exception)."""
    if len(df) <= n:
        return df.copy()
    sizes = df[strata].value_counts()
    raw = sizes / sizes.sum() * n
    alloc = np.floor(raw).astype(int)
    for k in (raw - alloc).sort_values(ascending=False).index[: n - alloc.sum()]:
        alloc[k] += 1
    parts = [df[df[strata] == s].sample(n=int(alloc[s]), random_state=seed) for s in sizes.index if alloc[s] > 0]
    return pd.concat(parts)


# ----------------------------------------------------------------------------- answerability audit
def make_question_audit(project: Project, register: pd.DataFrame, candidates: dict, passages: dict,
                        records: dict, pools: dict | None = None, round_no: int = 1) -> tuple[str, str]:
    a = project.study["benchmark"]["audit"]
    seed = project.seed + 1000 * round_no
    done = register[register.decision.isin(["include", "exclude"])]
    inc = stratified_sample(done[done.decision == "include"], a["n_included"], "category", seed)
    exc = stratified_sample(done[done.decision == "exclude"], a["n_excluded"], "category", seed)
    sample = pd.concat([inc, exc]).sample(frac=1, random_state=seed).reset_index(drop=True)
    rows, key = [], []
    for i, r in sample.iterrows():
        q, rec = candidates[r.question_id], records.get(r.question_id, {})
        pids = list(r.reference_passage_ids) if r.decision == "include" else \
            [e["passage_id"] for e in rec.get("evidence", []) or [] if e.get("passage_id") in passages][:3]
        if not pids:
            pids = [x["passage_id"] for x in (pools or {}).get(r.question_id, [])][:3]  # RRF top-3 of the pool
        ev = "\n\n".join(f"[{p}] {passages[p]['specification']} clause {passages[p]['clause_id']} "
                         f"({passages[p]['heading'][:60]}):\n{passages[p]['text']}" for p in pids if p in passages)
        aid = f"QA{round_no}-{i + 1:03d}"
        rows.append({"audit_id": aid, "question_id": r.question_id, "category": r.category, "question": q["question"],
                     "options": "\n".join(f"{k}: {v}" for k, v in sorted(q["options"].items())),
                     "keyed_option": f"{q['answer_option_id']}: {q['options'].get(q['answer_option_id'], '')}",
                     "teleqna_explanation": q["explanation"], "candidate_evidence": ev,
                     "official_archive": "https://www.3gpp.org/ftp/Specs/archive/",
                     "manual_answerable": "", "manual_passage_ids": "", "manual_hop_type": "",
                     "manual_question_type": "", "manual_reason_code_if_N": "", "reviewer_notes": ""})
        key.append({"audit_id": aid, "question_id": r.question_id, "verifier_decision": r.decision,
                    "reason_code": r.reason_code})
    sheet = project.path("audit", f"question_audit_round{round_no}_BLIND.csv")
    keyf = project.path("audit", f"question_audit_round{round_no}_KEY_do_not_open.csv")
    pd.DataFrame(rows).to_csv(sheet, index=False)
    pd.DataFrame(key).to_csv(keyf, index=False)
    return str(sheet), str(keyf)


def score_question_audit(project: Project, round_no: int = 1) -> dict:
    a = project.study["benchmark"]["audit"]
    sheet = pd.read_csv(project.path("audit", f"question_audit_round{round_no}_BLIND.csv"), dtype=str).fillna("")
    key = pd.read_csv(project.path("audit", f"question_audit_round{round_no}_KEY_do_not_open.csv"), dtype=str)
    m = sheet.merge(key, on="audit_id", how="inner")
    m["manual_answerable"] = m.manual_answerable.str.strip().str.upper().str[:1]
    filled = m[m.manual_answerable.isin(["Y", "N"])]
    res = {"round": round_no, "scored_utc": utc_now(), "n_sheet": len(m), "n_labelled": len(filled)}
    if len(filled) < len(m):
        res.update(status="INCOMPLETE", note="every row needs manual_answerable = Y or N")
        write_json(project.path("audit", f"question_audit_round{round_no}_score.json"), res)
        return res
    v = (filled.verifier_decision == "include").astype(int).tolist()
    h = (filled.manual_answerable == "Y").astype(int).tolist()
    tp = sum(x and y for x, y in zip(v, h))
    fp = sum(x and not y for x, y in zip(v, h))
    fn = sum((not x) and y for x, y in zip(v, h))
    prec = tp / (tp + fp) if tp + fp else None
    rec = tp / (tp + fn) if tp + fn else None
    kap = cohen_kappa(v, h)
    res.update(tp=tp, fp=fp, fn=fn, tn=len(v) - tp - fp - fn, precision=prec, recall=rec, cohen_kappa=kap,
               precision_ci95=wilson_ci(tp, tp + fp), thresholds={"precision": a["precision_min"], "kappa": a["kappa_min"]},
               note_recall="recall is computed on the stratified sample (included and excluded sampled at different rates)")
    res["status"] = "PASS" if (prec is not None and kap is not None and prec >= a["precision_min"]
                               and kap >= a["kappa_min"]) else "FAIL"
    corr = filled[["question_id", "manual_answerable", "manual_passage_ids", "manual_hop_type", "manual_question_type"]].copy()
    corr["manual_passage_ids"] = corr.manual_passage_ids.apply(lambda s: [x.strip() for x in s.replace(",", ";").split(";") if x.strip()])
    corr.to_json(project.path("audit", f"question_audit_round{round_no}_corrections.json"), orient="records", indent=1)
    write_json(project.path("audit", f"question_audit_round{round_no}_score.json"), res)
    return res


# ----------------------------------------------------------------------------- triple audit
def make_triple_audit(project: Project, triples: pd.DataFrame, passages: dict) -> tuple[str, str]:
    """Sample 200 extracted triples stratified by predicate, verifier-blind."""
    n = project.study["knowledge_graph"]["triple_audit"]["n_triples"]
    s = stratified_sample(triples, n, project.study["knowledge_graph"]["triple_audit"]["stratify_by"], project.seed)
    s = s.sample(frac=1, random_state=project.seed).reset_index(drop=True)
    rows, key = [], []
    for i, t in s.iterrows():
        aid = f"TA-{i + 1:03d}"
        p = passages.get(t.source_passage_id, {})
        rows.append({"audit_id": aid, "triple": f"{t.subject} --{t.predicate}--> {t.object}",
                     "subject_type": t.subject_type, "object_type": t.object_type, "supporting_quote": t.quote,
                     "source": f"{p.get('specification')} clause {p.get('clause_id')} [{t.source_passage_id}]",
                     "source_passage_text": p.get("text", ""), "manual_entailed": "", "reviewer_notes": ""})
        key.append({"audit_id": aid, "triple_id": t.triple_id, "predicate": t.predicate,
                    "verifier_verdict": t.get("fidelity_verdict")})
    sheet = project.path("audit", "triple_audit_BLIND.csv")
    keyf = project.path("audit", "triple_audit_KEY_do_not_open.csv")
    pd.DataFrame(rows).to_csv(sheet, index=False)
    pd.DataFrame(key).to_csv(keyf, index=False)
    return str(sheet), str(keyf)


def score_triple_audit(project: Project) -> dict:
    sheet = pd.read_csv(project.path("audit", "triple_audit_BLIND.csv"), dtype=str).fillna("")
    key = pd.read_csv(project.path("audit", "triple_audit_KEY_do_not_open.csv"), dtype=str)
    m = sheet.merge(key, on="audit_id")
    m["manual_entailed"] = m.manual_entailed.str.strip().str.upper().str[:1]
    f = m[m.manual_entailed.isin(["Y", "N"])]
    res = {"scored_utc": utc_now(), "n_sheet": len(m), "n_labelled": len(f)}
    if len(f) < len(m):
        res.update(status="INCOMPLETE")
        write_json(project.path("audit", "triple_audit_score.json"), res)
        return res
    acc = f[f.verifier_verdict == "entailed"]
    k = int((acc.manual_entailed == "Y").sum())
    lo, hi = wilson_ci(k, len(acc))
    v = (f.verifier_verdict == "entailed").astype(int).tolist()
    h = (f.manual_entailed == "Y").astype(int).tolist()
    res.update(n_verifier_accepted_in_sample=len(acc), fidelity_precision=k / len(acc) if len(acc) else None,
               fidelity_precision_wilson95=(lo, hi), remaining_error_rate=1 - k / len(acc) if len(acc) else None,
               verifier_agreement=sum(x == y for x, y in zip(v, h)) / len(v), verifier_kappa=cohen_kappa(v, h),
               all_triples_manual_precision=sum(h) / len(h),
               by_predicate=f.groupby("predicate").apply(lambda g: {"n": len(g), "manual_Y": int((g.manual_entailed == "Y").sum())},
                                                         include_groups=False).to_dict(),
               status="SCORED")
    write_json(project.path("audit", "triple_audit_score.json"), res)
    return res
''')

In [ ]:
#@title kgrag/benchmark/freeze.py
put('src/kgrag/benchmark/freeze.py', r'''"""Stage 6: freeze the gold benchmark and the stratified 20/80 development/test split (Sec. 3.3, 4.5)."""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..core import Project, read_json, sha256_file, utc_now, write_json, write_jsonl


def stratified_split(df: pd.DataFrame, strata: list[str], dev_fraction: float, seed: int) -> tuple[list[str], list[str]]:
    """Per stratum: shuffle with the fixed seed, round(dev_fraction*n) go to development (>=1 if n>=2)."""
    rng = np.random.default_rng(seed)
    dev, test = [], []
    for _, g in df.sort_values("question_id").groupby(strata, dropna=False, sort=True):
        ids = g.question_id.tolist()
        rng.shuffle(ids)
        k = int(round(dev_fraction * len(ids)))
        if len(ids) >= 2:
            k = max(1, min(k, len(ids) - 1))
        dev += ids[:k]
        test += ids[k:]
    return sorted(dev), sorted(test)


def freeze(project: Project, register: pd.DataFrame, candidates: dict, passages: dict, force: bool = False) -> dict:
    lock = project.path("processed", "FROZEN.json")
    if lock.exists() and not force:
        raise RuntimeError("Benchmark already frozen (data/processed/FROZEN.json). The test set must not change "
                           "after tuning begins (Sec. 3.3). Use force=True only before ANY experiment has run.")
    inc = register[register.decision == "include"].copy()
    cfg = project.study["benchmark"]
    dev, test = stratified_split(inc, cfg["split"]["strata"], cfg["split"]["dev_fraction"], project.seed)
    gq, ge = [], []
    for _, r in inc.sort_values("question_id").iterrows():
        q = candidates[r.question_id]
        gq.append({"question_id": r.question_id, "category": r.category, "question": q["question"],
                   "options": q["options"], "answer_option_id": q["answer_option_id"],
                   "hop_type": r.hop_type, "question_type": r.question_type,
                   "split": "development" if r.question_id in set(dev) else "test"})
        ge.append({"question_id": r.question_id, "reference_passage_ids": list(r.reference_passage_ids),
                   "reference_clauses": [f"{passages[p]['specification']} {passages[p]['clause_id']}"
                                         for p in r.reference_passage_ids if p in passages],
                   "n_required": len(r.reference_passage_ids)})
    P = lambda n: project.path("processed", n)
    write_jsonl(P("gold_questions.jsonl"), gq)
    write_jsonl(P("gold_evidence.jsonl"), ge)
    write_json(P("development_ids.json"), dev)
    write_json(P("test_ids.json"), test)
    man = {"frozen_utc": utc_now(), "seed": project.seed, "n_included": len(inc), "n_development": len(dev),
           "n_test": len(test), "strata": cfg["split"]["strata"],
           "by_stratum": inc.groupby(cfg["split"]["strata"]).size().reset_index(name="n").to_dict("records"),
           "below_minimum": len(inc) < cfg["min_questions"],
           "sha256": {n: sha256_file(P(n)) for n in ("gold_questions.jsonl", "gold_evidence.jsonl",
                                                     "development_ids.json", "test_ids.json")},
           "passages_sha256": sha256_file(project.path("interim", "passages.jsonl")),
           "corpus_manifest_sha256": sha256_file(project.path("manifests", "corpus_manifest.json"))}
    write_json(lock, man)
    return man


def verify_frozen(project: Project) -> bool:
    man = read_json(project.path("processed", "FROZEN.json"))
    return all(sha256_file(project.path("processed", n)) == h for n, h in man["sha256"].items())


def load(project: Project) -> tuple[list[dict], dict[str, dict], list[str], list[str]]:
    from ..core import read_jsonl
    if not verify_frozen(project):
        raise RuntimeError("Frozen benchmark files changed after freezing — hashes do not match FROZEN.json")
    gq = read_jsonl(project.path("processed", "gold_questions.jsonl"))
    ge = {r["question_id"]: r for r in read_jsonl(project.path("processed", "gold_evidence.jsonl"))}
    return gq, ge, read_json(project.path("processed", "development_ids.json")), read_json(project.path("processed", "test_ids.json"))
''')

### 2.5 Retrieval and reranking (Sec. 3.4)
* `encoders.py`: sentence-transformer encoders and a FAISS inner-product index (exact NumPy fallback).
* `dense.py`: BGE + FAISS top-30 retrieval; derives the *retriever-found* labels.
* `reranker.py`: `bge-reranker-base` cross-encoder; keeps ≤ 5 passages above a threshold fixed **on development questions only**.

In [ ]:
#@title kgrag/retrieval/encoders.py
put('src/kgrag/retrieval/encoders.py', r'''"""Dense encoders and a FAISS (or exact NumPy fallback) inner-product index."""
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Protocol

import numpy as np


class Encoder(Protocol):
    name: str

    def encode_queries(self, texts: list[str]) -> np.ndarray: ...

    def encode_passages(self, texts: list[str]) -> np.ndarray: ...


class STEncoder:
    """sentence-transformers encoder with model-specific prefixes (E5: 'query: '/'passage: '; BGE: query instruction)."""

    def __init__(self, name: str, query_prefix: str = "", passage_prefix: str = "", device: str | None = None,
                 batch_size: int = 64, max_seq_length: int = 512):
        import torch
        from sentence_transformers import SentenceTransformer
        self.name = name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(name, device=self.device)
        self.model.max_seq_length = max_seq_length
        self.qp, self.pp, self.bs = query_prefix, passage_prefix, batch_size
        self.revision = getattr(getattr(self.model, "model_card_data", None), "base_model_revision", None)

    def encode_queries(self, texts):
        return np.asarray(self.model.encode([self.qp + t for t in texts], batch_size=self.bs,
                                            normalize_embeddings=True, show_progress_bar=False), dtype="float32")

    def encode_passages(self, texts, show_progress=True):
        return np.asarray(self.model.encode([self.pp + t for t in texts], batch_size=self.bs,
                                            normalize_embeddings=True, show_progress_bar=show_progress), dtype="float32")


def e5(device=None) -> STEncoder:
    return STEncoder("intfloat/e5-base-v2", "query: ", "passage: ", device)


def bge(instruction: str, device=None) -> STEncoder:
    return STEncoder("BAAI/bge-base-en-v1.5", instruction, "", device)


class IPIndex:
    """Exact inner-product search. Uses faiss.IndexFlatIP when available, NumPy otherwise (identical results)."""

    def __init__(self, vectors: np.ndarray):
        self.vectors = np.ascontiguousarray(vectors, dtype="float32")
        try:
            import faiss
            self.index = faiss.IndexFlatIP(self.vectors.shape[1])
            self.index.add(self.vectors)
            self.backend = f"faiss.IndexFlatIP (faiss {faiss.__version__})"
        except ImportError:
            self.index, self.backend = None, "numpy exact inner product"

    def search(self, q: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
        q = np.ascontiguousarray(np.atleast_2d(q), dtype="float32")
        k = min(k, len(self.vectors))
        if self.index is not None:
            return self.index.search(q, k)
        s = q @ self.vectors.T
        idx = np.argsort(-s, axis=1, kind="stable")[:, :k]
        return np.take_along_axis(s, idx, 1), idx

    def save(self, path: str | Path) -> None:
        np.save(Path(path), self.vectors)
        if self.index is not None:
            import faiss
            faiss.write_index(self.index, str(Path(path).with_suffix(".faiss")))


def cached_passage_embeddings(encoder: Encoder, passages: list[dict], cache_npy: Path, ids_json: Path) -> np.ndarray:
    pids = [p["passage_id"] for p in passages]
    if cache_npy.exists() and ids_json.exists() and json.loads(ids_json.read_text()) == pids:
        return np.load(cache_npy)
    t0 = time.time()
    emb = encoder.encode_passages([p["text"] for p in passages])
    np.save(cache_npy, emb)
    ids_json.write_text(json.dumps(pids))
    (cache_npy.with_suffix(".timing.json")).write_text(json.dumps({"encode_seconds": time.time() - t0,
                                                                     "n": len(pids), "model": encoder.name}))
    return emb
''')

In [ ]:
#@title kgrag/retrieval/dense.py
put('src/kgrag/retrieval/dense.py', r'''"""Stage 7: experimental semantic retrieval — BAAI/bge-base-en-v1.5 + FAISS, top-30 (Sec. 3.4 Comp. 2, 4.7.1)."""
from __future__ import annotations

import time

import numpy as np

from ..core import Project, write_jsonl, write_json
from .encoders import IPIndex, cached_passage_embeddings


class Retriever:
    def __init__(self, encoder, passages: list[dict], vectors: np.ndarray, top_k: int = 30):
        self.encoder, self.passages, self.top_k = encoder, passages, top_k
        self.pids = [p["passage_id"] for p in passages]
        self.index = IPIndex(vectors)

    @classmethod
    def build(cls, project: Project, encoder, passages: list[dict]) -> "Retriever":
        V = cached_passage_embeddings(encoder, passages, project.path("interim", "bge_passage_embeddings.npy"),
                                      project.path("interim", "bge_passage_ids.json"))
        r = cls(encoder, passages, V, project.study["retrieval"]["top_k"])
        r.index.save(project.path("interim", "bge_index.npy"))
        write_json(project.path("manifests", "retrieval_index.json"), {
            "embedding_model": encoder.name, "n_passages": len(passages), "dim": int(V.shape[1]),
            "index_backend": r.index.backend, "top_k": r.top_k, "normalised": True})
        return r

    def retrieve(self, question: str) -> tuple[list[dict], float]:
        """Fresh (uncached) query encoding + search; returns ranked candidates and latency in seconds."""
        t0 = time.perf_counter()
        qv = self.encoder.encode_queries([question])
        scores, idx = self.index.search(qv, self.top_k)
        lat = time.perf_counter() - t0
        out = []
        for rank, (s, i) in enumerate(zip(scores[0], idx[0]), 1):
            p = self.passages[int(i)]
            out.append({"passage_id": p["passage_id"], "score": float(s), "rank": rank,
                        "specification": p["specification"], "clause_id": p["clause_id"]})
        return out, lat


def question_text(q: dict) -> str:
    """Query = question + options (identical for all retrieval configurations)."""
    return q["question"] + "\n" + "\n".join(f"option {k}: {v}" for k, v in sorted(q["options"].items()))


def run_and_label(project: Project, retriever: Retriever, questions: list[dict], gold: dict[str, dict]) -> dict:
    """Retrieve top-30 for every frozen question; record retriever-found vs independent-only (Sec. 3.3d)."""
    rows, groups = [], {}
    for q in questions:
        cands, lat = retriever.retrieve(question_text(q))
        ref = set(gold[q["question_id"]]["reference_passage_ids"])
        found = any(c["passage_id"] in ref for c in cands)
        groups[q["question_id"]] = "retriever-found" if found else "independent-only"
        rows.append({"question_id": q["question_id"], "candidates": cands, "retrieval_latency_s": lat,
                     "gold_in_top30": found})
    write_jsonl(project.path("interim", "bge_top30.jsonl"), rows)
    write_json(project.path("processed", "retriever_groups.json"), groups)  # diagnostic label; never filters questions
    return groups
''')

In [ ]:
#@title kgrag/reranking/reranker.py
put('src/kgrag/reranking/reranker.py', r'''"""Stage 8: cross-encoder reranking with BAAI/bge-reranker-base; <=5 passages above a development-fixed threshold."""
from __future__ import annotations

import math
import time

import numpy as np


class CrossEncoderReranker:
    def __init__(self, name: str = "BAAI/bge-reranker-base", max_length: int = 512, device: str | None = None,
                 batch_size: int = 32):
        import torch
        from sentence_transformers import CrossEncoder
        self.name = name
        self.model = CrossEncoder(name, max_length=max_length,
                                  device=device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.bs = batch_size

    def score(self, query: str, texts: list[str]) -> tuple[list[float], float]:
        t0 = time.perf_counter()
        # sentence-transformers applies Sigmoid by default for single-label cross-encoders -> scores in (0, 1)
        probs = self.model.predict([(query, t) for t in texts], batch_size=self.bs, show_progress_bar=False) if texts else []
        lat = time.perf_counter() - t0
        vals = [float(x) for x in np.ravel(probs)]
        if vals and (min(vals) < 0 or max(vals) > 1):  # older/newer API returned logits
            vals = [1.0 / (1.0 + math.exp(-v)) for v in vals]
        return vals, lat


def select(scored: list[dict], threshold: float, max_k: int = 5, min_k: int = 1) -> list[dict]:
    """Order by reranker score (ties: original retrieval rank); keep items >= threshold, at most max_k, at least min_k."""
    order = sorted(scored, key=lambda r: (-r["rerank_score"], r["rank"]))
    keep = [r for r in order if r["rerank_score"] >= threshold][:max_k]
    if len(keep) < min_k:
        keep = order[:min_k]
    return [{**r, "final_rank": i + 1} for i, r in enumerate(keep)]


def rerank(reranker, query: str, candidates: list[dict], passages: dict, threshold: float,
           max_k: int = 5, min_k: int = 1) -> tuple[list[dict], list[dict], float]:
    scores, lat = reranker.score(query, [passages[c["passage_id"]]["text"] for c in candidates])
    scored = [{**c, "rerank_score": s} for c, s in zip(candidates, scores)]
    return select(scored, threshold, max_k, min_k), scored, lat


def choose_threshold(dev_scored: dict[str, list[dict]], gold: dict[str, set], max_k: int = 5, min_k: int = 1) -> dict:
    """Development-only threshold: maximise mean per-question F1 of selected passages vs gold passages.
    Candidate thresholds = every distinct score observed (plus 0); ties -> lowest threshold (keeps more evidence)."""
    cands = sorted({0.0} | {round(r["rerank_score"], 6) for v in dev_scored.values() for r in v})
    best = None
    for t in cands:
        f1s = []
        for qid, scored in dev_scored.items():
            sel = {r["passage_id"] for r in select(scored, t, max_k, min_k)}
            g = gold[qid]
            tp = len(sel & g)
            p = tp / len(sel) if sel else 0
            rc = tp / len(g) if g else 0
            f1s.append(0 if tp == 0 else 2 * p * rc / (p + rc))
        m = float(np.mean(f1s)) if f1s else 0.0
        if best is None or m > best["mean_f1"] + 1e-12:
            best = {"threshold": t, "mean_f1": m}
    best.update(n_dev_questions=len(dev_scored), n_candidate_thresholds=len(cands),
                rule="max mean per-question F1 (selected vs gold) on development split; ties -> lowest threshold")
    return best
''')

### 2.6 Knowledge graph (Sec. 3.4, 3.6)
* `extract.py`: LLM triple extraction with one fixed prompt and a closed schema; every triple carries its source passage and supporting quote.
* `fidelity.py`: a **different** model re-checks that each triple is entailed by its passage.
* `normalize.py`: entity normalisation, abbreviation mapping learned from the corpus itself, de-duplication that keeps provenance.
* `graph.py`: NetworkX multigraph; query-time triple selection for C (seeds from B's passages), D (seeds from all 30), RANDOM and UNFILTERED.

In [ ]:
#@title kgrag/knowledge_graph/extract.py
put('src/kgrag/knowledge_graph/extract.py', r'''"""Stage 9: LLM-based triple extraction with a fixed prompt and predefined schema (Sec. 3.4 Comp. 1, 4.6.1).

Model = models.yaml role `kg_extractor`. One call per passage (provenance stays one-to-one). Every triple must carry a
verbatim supporting quotation; the script marks provenance invalid when the quote is not in the passage.
"""
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed

from ..core import Project, append_jsonl, iter_jsonl, utc_now
from ..llm import LLMClient
from ..validation.verifier import quote_in_passage

PROMPT_VERSION = "kg-extract-v1.0"

SYSTEM = ("You extract a knowledge graph from 3GPP Release 17 specification text. Use ONLY facts stated in the given "
          "passage. Never add background knowledge.")

TEMPLATE = """Extract subject-predicate-object triples from the PASSAGE.

Entity types: {etypes}
Predicates (use exactly one of these):
- SENDS (entity sends a Message), RECEIVES (entity receives a Message), USES_INTERFACE (entity/message uses an Interface or reference point),
- CONNECTS (Interface connects two entities; subject = Interface), PART_OF (step/message/procedure is part of a Procedure),
- TRIGGERS (message/event/procedure triggers a Procedure or Message), INITIATES (entity initiates a Procedure),
- STARTS_TIMER / STOPS_TIMER (entity/message/procedure starts/stops a Timer), CONTAINS (Message contains an InformationElement),
- PROVIDES (NetworkFunction provides a Service or function), REQUIRES (normative requirement: subject shall/should/may do or have object),
- SELECTS (entity selects another entity), STORES (entity stores data), IS_A (subtype relation).
Rules:
- Use the specification's own names; prefer the abbreviation when the passage gives one (e.g. "AMF").
- quote: copy VERBATIM from the passage (exact characters, at least 6 words) the sentence fragment that states the triple.
- For REQUIRES keep the normative word (shall / should / may / shall not) in `modality`.
- Return at most 12 of the most informative triples. Return an empty list if the passage states no such relation (e.g. a pure table of values or a change log).

PASSAGE [{pid}] {spec} clause {clause} - {heading}:
{text}"""

SCHEMA = {"type": "object", "properties": {"triples": {"type": "array", "items": {"type": "object", "properties": {
    "subject": {"type": "string"}, "subject_type": {"type": "string"},
    "predicate": {"type": "string"}, "object": {"type": "string"}, "object_type": {"type": "string"},
    "modality": {"type": "string"}, "quote": {"type": "string"}},
    "required": ["subject", "subject_type", "predicate", "object", "object_type", "quote"]}}}, "required": ["triples"]}


def extract_passage(llm: LLMClient, p: dict, kg_cfg: dict) -> dict:
    prompt = TEMPLATE.format(etypes=", ".join(kg_cfg["entity_types"]), pid=p["passage_id"], spec=p["specification"],
                             clause=p["clause_id"], heading=p["heading"][:100], text=p["text"])
    r = llm.generate_json("kg_extractor", prompt, SCHEMA, SYSTEM)
    rec = {"passage_id": p["passage_id"], "prompt_version": PROMPT_VERSION, "model": r.model,
           "model_version": r.model_version, "input_tokens": r.input_tokens, "output_tokens": r.output_tokens,
           "cost_usd": r.cost_usd, "latency_s": r.latency_s, "utc": utc_now(),
           "status": "ok" if r.ok else "error", "error": r.error, "triples": []}
    if not r.ok:
        return rec
    preds, etypes = set(kg_cfg["predicates"]), set(kg_cfg["entity_types"])
    for i, t in enumerate(r.data.get("triples", []) or []):
        pred = str(t.get("predicate", "")).strip().upper().replace(" ", "_")
        st, ot = t.get("subject_type", "Other"), t.get("object_type", "Other")
        rec["triples"].append({
            "local_id": i, "subject": str(t.get("subject", "")).strip(), "predicate": pred,
            "object": str(t.get("object", "")).strip(), "subject_type": st if st in etypes else "Other",
            "object_type": ot if ot in etypes else "Other", "modality": t.get("modality"), "quote": t.get("quote", ""),
            "schema_valid": pred in preds and bool(t.get("subject")) and bool(t.get("object")),
            "quote_found": quote_in_passage(t.get("quote", ""), p["text"])})
    return rec


def run(project: Project, llm: LLMClient, passages: list[dict], workers: int | None = None,
        limit: int | None = None) -> str:
    out = project.path("interim", "kg_extraction.jsonl")
    done = {r["passage_id"] for r in iter_jsonl(out) if r.get("status") == "ok" and r.get("prompt_version") == PROMPT_VERSION}
    todo = [p for p in passages if p["passage_id"] not in done][: limit or None]
    kg_cfg = project.study["knowledge_graph"]
    with ThreadPoolExecutor(workers or project.study["experiment"]["workers"]) as ex:
        for f in as_completed([ex.submit(extract_passage, llm, p, kg_cfg) for p in todo]):
            append_jsonl(out, f.result())
    return str(out)
''')

In [ ]:
#@title kgrag/knowledge_graph/fidelity.py
put('src/kgrag/knowledge_graph/fidelity.py', r'''"""Stage 10: independent triple-fidelity verification (Sec. 3.6) by the `verifier` role (not the extractor,
generator or evaluator). One call per source passage judges all of that passage's triples."""
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed

from ..core import Project, append_jsonl, iter_jsonl, utc_now
from ..llm import LLMClient

PROMPT_VERSION = "kg-verify-v1.0"
SYSTEM = "You check whether knowledge-graph triples are entailed by their source text from a 3GPP specification."
TEMPLATE = """SOURCE PASSAGE [{pid}] {spec} clause {clause}:
{text}

For each numbered triple decide whether the SOURCE PASSAGE alone entails it (subject, relation and object all correct;
direction correct; no information added). Verdict "entailed" or "not_entailed".

TRIPLES:
{triples}"""
SCHEMA = {"type": "object", "properties": {"judgements": {"type": "array", "items": {"type": "object", "properties": {
    "n": {"type": "integer"}, "verdict": {"type": "string", "enum": ["entailed", "not_entailed"]},
    "reason": {"type": "string"}}, "required": ["n", "verdict"]}}}, "required": ["judgements"]}

PRED_TEXT = {"SENDS": "sends", "RECEIVES": "receives", "USES_INTERFACE": "uses interface", "CONNECTS": "connects",
             "PART_OF": "is part of", "TRIGGERS": "triggers", "INITIATES": "initiates", "STARTS_TIMER": "starts timer",
             "STOPS_TIMER": "stops timer", "CONTAINS": "contains", "PROVIDES": "provides", "REQUIRES": "requires",
             "SELECTS": "selects", "STORES": "stores", "IS_A": "is a"}


def statement(s: str, p: str, o: str) -> str:
    return f"{s} {PRED_TEXT.get(p, p.lower())} {o}"


def verify_passage(llm: LLMClient, p: dict, instances: list[dict]) -> list[dict]:
    lines = "\n".join(f"{i + 1}. ({t['subject']}) --{t['predicate']}--> ({t['object']})" + (f" [modality: {t['modality']}]" if t.get("modality") else "")
                      for i, t in enumerate(instances))
    r = llm.generate_json("verifier", TEMPLATE.format(pid=p["passage_id"], spec=p["specification"], clause=p["clause_id"],
                                                      text=p["text"], triples=lines), SCHEMA, SYSTEM)
    verdicts = {}
    if r.ok:
        for j in r.data.get("judgements", []):
            verdicts[int(j.get("n", 0))] = (j.get("verdict"), j.get("reason"))
    out = []
    for i, t in enumerate(instances):
        v = verdicts.get(i + 1)
        out.append({"instance_id": t["instance_id"], "passage_id": p["passage_id"], "prompt_version": PROMPT_VERSION,
                    "model": r.model, "model_version": r.model_version,
                    "fidelity_verdict": v[0] if v else None, "reason": v[1] if v else None,
                    "status": "ok" if (r.ok and v) else "error", "error": r.error,
                    "batch_cost_usd": r.cost_usd if i == 0 else 0.0, "utc": utc_now()})
    return out


def run(project: Project, llm: LLMClient, instances: list[dict], passages: dict, workers: int | None = None) -> str:
    out = project.path("interim", "kg_fidelity.jsonl")
    done = {r["instance_id"] for r in iter_jsonl(out) if r.get("status") == "ok" and r.get("prompt_version") == PROMPT_VERSION}
    by_p: dict[str, list[dict]] = {}
    for t in instances:
        if t["instance_id"] not in done:
            by_p.setdefault(t["passage_id"], []).append(t)
    with ThreadPoolExecutor(workers or project.study["experiment"]["workers"]) as ex:
        for f in as_completed([ex.submit(verify_passage, llm, passages[pid], ts) for pid, ts in by_p.items()]):
            for row in f.result():
                append_jsonl(out, row)
    return str(out)
''')

In [ ]:
#@title kgrag/knowledge_graph/normalize.py
put('src/kgrag/knowledge_graph/normalize.py', r'''"""Entity normalisation, abbreviation mapping from the corpus itself, de-duplication with provenance (Sec. 4.6.2).

The alias table is built ONLY from the Abbreviations clauses of the frozen corpus (no external dictionaries).
"""
from __future__ import annotations

import collections
import re

from ..core import stable_hash

ABBR_LINE = re.compile(r"^\s*\|?\s*([A-Z][A-Za-z0-9\-/]{1,15})\s*\|?\s{1,}\|?\s*([A-Z][A-Za-z0-9 ,\-/()']{3,120}?)\s*\|?\s*$")
_ESC = re.compile(r"\\([\\`*_{}\[\]()#+\-.!|<>~])")


def build_alias_table(passages: list[dict]) -> dict[str, str]:
    """{long form lower -> ABBREVIATION} from clauses headed 'Abbreviations' in the corpus."""
    table: dict[str, str] = {}
    for p in passages:
        if "abbreviation" not in p["heading"].lower():
            continue
        for line in p["text"].splitlines():
            line = _ESC.sub(r"\1", line).replace("*", "")
            m = ABBR_LINE.match(line)
            if m and is_abbreviation(m.group(1)):
                abbr, long = m.group(1).strip(), m.group(2).strip()
                if len(long.split()) >= 2:
                    table.setdefault(long.lower(), abbr)
    return table


def is_abbreviation(s: str) -> bool:
    return bool(re.fullmatch(r"[A-Z0-9][A-Za-z0-9\-/]*", s)) and sum(c.isupper() for c in s) >= 2


def canonical(name: str, aliases: dict[str, str]) -> tuple[str, str]:
    """Return (key, display). Abbreviations are case-sensitive keys; other names are case-folded."""
    s = _ESC.sub(r"\1", name or "").replace("*", "").replace("`", "")
    s = re.sub(r"\s+", " ", s).strip(" .,:;\"'")
    s = re.sub(r"^(the|a|an)\s+", "", s, flags=re.I)
    m = re.fullmatch(r"(.+?)\s*\(([A-Z][A-Za-z0-9\-/]{1,15})\)", s)  # "Access and Mobility ... (AMF)"
    if m and is_abbreviation(m.group(2)):
        s = m.group(2)
    m = re.fullmatch(r"([A-Z][A-Za-z0-9\-/]{1,15})\s*\((.+)\)", s)    # "AMF (Access and Mobility ...)"
    if m and is_abbreviation(m.group(1)):
        s = m.group(1)
    if s.lower() in aliases:
        s = aliases[s.lower()]
    key = s if is_abbreviation(s) else s.casefold()
    return key, s


def deduplicate(instances: list[dict], aliases: dict[str, str]) -> list[dict]:
    """Merge extraction instances with the same (subject, predicate, object) keys; keep ALL provenance entries."""
    groups: dict[tuple, list[dict]] = collections.defaultdict(list)
    for t in instances:
        sk, sd = canonical(t["subject"], aliases)
        ok, od = canonical(t["object"], aliases)
        if not sk or not ok or sk == ok:
            continue
        groups[(sk, t["predicate"], ok)].append({**t, "_sd": sd, "_od": od})
    out = []
    for (sk, pred, ok), inst in groups.items():
        sd = collections.Counter(i["_sd"] for i in inst).most_common(1)[0][0]
        od = collections.Counter(i["_od"] for i in inst).most_common(1)[0][0]
        st = collections.Counter(i["subject_type"] for i in inst).most_common(1)[0][0]
        ot = collections.Counter(i["object_type"] for i in inst).most_common(1)[0][0]
        prov = sorted(({"passage_id": i["passage_id"], "quote": i["quote"], "specification": i["specification"],
                        "clause_id": i["clause_id"], "version": i["version"], "release": i["release"],
                        "instance_id": i["instance_id"], "fidelity_verdict": i.get("fidelity_verdict")}
                       for i in inst), key=lambda d: d["instance_id"])
        out.append({"triple_id": "T" + stable_hash([sk, pred, ok], 12), "subject": sd, "subject_key": sk,
                    "subject_type": st, "predicate": pred, "object": od, "object_key": ok, "object_type": ot,
                    "provenance": prov, "n_sources": len(prov)})
    return sorted(out, key=lambda t: t["triple_id"])
''')

In [ ]:
#@title kgrag/knowledge_graph/graph.py
put('src/kgrag/knowledge_graph/graph.py', r'''"""Validated KG assembly (NetworkX), statistics, and query-time triple selection for C, D and the two controls.

Only triples with >=1 provenance instance that is schema-valid, quote-verified (provenance) AND verifier-entailed
(fidelity) enter the validated graph (Sec. 3.3: unsupported relationships excluded).
"""
from __future__ import annotations

import collections
import pickle
import random
import re
import time
import tracemalloc

import networkx as nx
import numpy as np

from ..core import Project, iter_jsonl, read_jsonl, sha256_text, write_json, write_jsonl
from .fidelity import statement
from .normalize import build_alias_table, canonical, deduplicate, is_abbreviation


def assemble(project: Project, passages: dict, release: str = "17") -> tuple[list[dict], list[dict], dict]:
    """Returns (all_triples_with_verdicts, validated_triples, stats)."""
    fid = {}
    for r in iter_jsonl(project.path("interim", "kg_fidelity.jsonl")):
        if r.get("status") == "ok":
            fid[r["instance_id"]] = r["fidelity_verdict"]
    instances, n_ext, n_err = [], 0, 0
    for rec in iter_jsonl(project.path("interim", "kg_extraction.jsonl")):
        if rec.get("status") != "ok":
            n_err += 1
            continue
        p = passages.get(rec["passage_id"])
        if p is None:
            continue
        for t in rec["triples"]:
            n_ext += 1
            iid = f"{rec['passage_id']}#{t['local_id']}"
            instances.append({**t, "instance_id": iid, "passage_id": p["passage_id"], "specification": p["specification"],
                              "clause_id": p["clause_id"], "version": p["version"], "release": p["release"],
                              "provenance_valid": bool(t["quote_found"] and p["release"] == release),
                              "fidelity_verdict": fid.get(iid)})
    aliases = build_alias_table(list(passages.values()))
    all_tr = deduplicate([i for i in instances if i["schema_valid"]], aliases)
    by_inst = {i["instance_id"]: i for i in instances}
    validated = []
    for t in all_tr:
        good = [pv for pv in t["provenance"] if by_inst[pv["instance_id"]]["provenance_valid"]
                and pv["fidelity_verdict"] == "entailed"]
        t["fidelity_verdict"] = "entailed" if good else ("not_entailed" if all(pv["fidelity_verdict"] for pv in t["provenance"]) else "unverified")
        t["provenance_valid"] = any(by_inst[pv["instance_id"]]["provenance_valid"] for pv in t["provenance"])
        if good:
            validated.append({**t, "provenance": good, "n_sources": len(good),
                              "source_passage_id": good[0]["passage_id"], "quote": good[0]["quote"]})
    for t in all_tr:  # flattened fields for the audit sampler
        t["source_passage_id"], t["quote"] = t["provenance"][0]["passage_id"], t["provenance"][0]["quote"]
    stats = {"extraction_records_error": n_err, "instances_extracted": n_ext,
             "instances_schema_valid": sum(i["schema_valid"] for i in instances),
             "instances_quote_verified": sum(i["provenance_valid"] for i in instances),
             "instances_verifier_entailed": sum(i["fidelity_verdict"] == "entailed" for i in instances),
             "instances_unverified": sum(i["fidelity_verdict"] is None for i in instances),
             "unique_triples_before_validation": len(all_tr), "validated_triples": len(validated),
             "alias_table_size": len(aliases)}
    write_jsonl(project.path("interim", "kg_triples_all.jsonl"), all_tr)
    write_jsonl(project.path("processed", "kg_triples_validated.jsonl"), validated)
    write_json(project.path("processed", "kg_alias_table.json"), aliases)
    G = to_networkx(validated)
    with open(project.path("processed", "kg_validated.gpickle"), "wb") as f:
        pickle.dump(G, f)
    stats.update(graph_statistics(G, validated))
    write_json(project.path("reports", "kg_statistics.json"), stats)
    return all_tr, validated, stats


def to_networkx(triples: list[dict]) -> nx.MultiDiGraph:
    G = nx.MultiDiGraph()
    for t in triples:
        G.add_node(t["subject_key"], name=t["subject"], type=t["subject_type"])
        G.add_node(t["object_key"], name=t["object"], type=t["object_type"])
        G.add_edge(t["subject_key"], t["object_key"], key=t["triple_id"], predicate=t["predicate"],
                   sources=[p["passage_id"] for p in t["provenance"]])
    return G


def graph_statistics(G: nx.MultiDiGraph, triples: list[dict]) -> dict:
    deg = [d for _, d in G.degree()]
    return {"nodes": G.number_of_nodes(), "edges": G.number_of_edges(),
            "relation_type_frequencies": dict(collections.Counter(t["predicate"] for t in triples).most_common()),
            "entity_type_frequencies": dict(collections.Counter(d.get("type") for _, d in G.nodes(data=True)).most_common()),
            "weakly_connected_components": nx.number_weakly_connected_components(G) if G.number_of_nodes() else 0,
            "mean_degree": float(np.mean(deg)) if deg else 0.0, "max_degree": int(max(deg)) if deg else 0,
            "source_linked_triples": sum(1 for t in triples if t.get("provenance"))}


# ============================================================================ query-time
GRAPH_HEADER = "Knowledge-graph statements (each derived from the cited source clause):"
STOP_NAMES = {"data", "information", "procedure", "message", "network", "request", "response", "function",
              "service", "user", "value", "parameter", "timer", "entity", "node", "list", "set", "case"}


def render_statement(t: dict, label: str) -> str:
    p = t["provenance"][0]
    return f"[{label}] {statement(t['subject'], t['predicate'], t['object'])}. (Source: {p['specification']} clause {p['clause_id']}, {p['passage_id']})"


class KGIndex:
    """In-memory validated KG with entity matcher and statement embeddings (BGE passage encoder)."""

    def __init__(self, triples: list[dict], encoder=None, emb: np.ndarray | None = None):
        tracemalloc.start()
        t0 = time.perf_counter()
        self.triples = triples
        self.by_id = {t["triple_id"]: t for t in triples}
        self.ids = [t["triple_id"] for t in triples]
        self.pos = {tid: i for i, tid in enumerate(self.ids)}
        self.incident: dict[str, list[str]] = collections.defaultdict(list)
        names: dict[str, set] = collections.defaultdict(set)
        for t in triples:
            self.incident[t["subject_key"]].append(t["triple_id"])
            self.incident[t["object_key"]].append(t["triple_id"])
            names[t["subject_key"]].add(t["subject"])
            names[t["object_key"]].add(t["object"])
        self.matcher: dict[str, str] = {}
        for key, surf in names.items():
            for s in surf | {key}:
                k = s if is_abbreviation(s) else s.casefold()
                if len(k) >= 2 and k.casefold() not in STOP_NAMES:
                    self.matcher.setdefault(k, key)
        self.max_ngram = max((len(k.split()) for k in self.matcher), default=1)
        self.encoder = encoder
        if emb is None and encoder is not None and triples:
            emb = encoder.encode_passages([statement(t["subject"], t["predicate"], t["object"]) for t in triples],
                                          show_progress=False)
        self.emb = emb
        self.build_seconds = time.perf_counter() - t0
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        self.peak_memory_mb = peak / 2**20

    def seeds(self, text: str) -> set[str]:
        words = re.findall(r"[A-Za-z0-9][A-Za-z0-9\-/]*", text)
        found = set()
        for n in range(1, self.max_ngram + 1):
            for i in range(len(words) - n + 1):
                span = " ".join(words[i:i + n])
                k = span if is_abbreviation(span) else span.casefold()
                if k in self.matcher:
                    found.add(self.matcher[k])
        return found

    def expand(self, seeds: set[str], hops: int) -> dict[str, int]:
        """triple_id -> hop distance (1 or 2), deterministic."""
        dist: dict[str, int] = {}
        frontier = set(seeds)
        visited = set(seeds)
        for h in range(1, hops + 1):
            nxt = set()
            for node in sorted(frontier):
                for tid in self.incident.get(node, []):
                    if tid not in dist:
                        dist[tid] = h
                    t = self.by_id[tid]
                    for other in (t["subject_key"], t["object_key"]):
                        if other not in visited:
                            nxt.add(other)
            visited |= nxt
            frontier = nxt
        return dist

    def relevance(self, qvec: np.ndarray, tids: list[str]) -> np.ndarray:
        if self.emb is None or not tids:
            return np.zeros(len(tids))
        return self.emb[[self.pos[t] for t in tids]] @ np.ravel(qvec)

    def select(self, *, qvec, seed_text: str, hops: int, triple_budget: int | None, token_cap: int | None,
               count, mode: str = "selective", rng_seed: int = 0, n_random: int | None = None,
               label_start: int = 1) -> dict:
        """mode: selective (Configuration C/D), unfiltered (control), random (control).
        Stops at the first statement that would exceed `token_cap` (the binding limit is logged)."""
        t0 = time.perf_counter()
        seeds = self.seeds(seed_text)
        dist = self.expand(seeds, hops)
        cands = sorted(dist)
        if mode == "selective":
            sc = self.relevance(qvec, cands)
            order = [c for _, c in sorted(zip(-sc, cands), key=lambda x: (x[0], dist[x[1]], x[1]))]
            limit = triple_budget
        elif mode == "unfiltered":
            order = sorted(cands, key=lambda c: (dist[c], c))
            limit = None  # until the token allocation is reached (Sec. 3.5)
        elif mode == "random":
            rng = random.Random(rng_seed)
            order = rng.sample(self.ids, min(len(self.ids), max(0, n_random or 0) * 3 + 50))
            limit = n_random
        else:
            raise ValueError(mode)
        if hasattr(count, "prefetch"):  # API-count mode: count the likely statements in parallel first
            k = (limit or 40) + 5
            count.prefetch([render_statement(self.by_id[t], f"G{label_start + i}") for i, t in enumerate(order[:k])])
        chosen, toks, stop_reason = [], 0, "exhausted_candidates"
        for tid in order:
            if limit is not None and len(chosen) >= limit:
                stop_reason = "triple_budget"
                break
            text = render_statement(self.by_id[tid], f"G{label_start + len(chosen)}")
            n = count(text)
            if token_cap is not None and toks + n > token_cap:
                stop_reason = "token_cap"
                break
            chosen.append({"triple_id": tid, "text": text, "tokens": n, "hop": dist.get(tid),
                           "source_passage_ids": [p["passage_id"] for p in self.by_id[tid]["provenance"]]})
            toks += n
        return {"items": chosen, "graph_tokens": toks, "n_seeds": len(seeds), "n_candidates": len(cands),
                "stop_reason": stop_reason, "lookup_latency_s": time.perf_counter() - t0}


def load(project: Project, encoder=None) -> KGIndex:
    tr = read_jsonl(project.path("processed", "kg_triples_validated.jsonl"))
    cache = project.path("interim", "kg_statement_emb.npy")
    key = project.path("interim", "kg_statement_emb.key")
    sig = sha256_text("|".join(t["triple_id"] for t in tr))
    emb = np.load(cache) if (cache.exists() and key.exists() and key.read_text() == sig) else None
    kg = KGIndex(tr, encoder, emb)
    if emb is None and kg.emb is not None:
        np.save(cache, kg.emb)
        key.write_text(sig)
    return kg
''')

### 2.7 Experiment, evaluation, statistics, reporting
* `context.py`: builds the evidence context for each configuration and budget mode; enforces the matched-context rule.
* `generator.py`: one generation interface: same model, prompt and decoding for every configuration.
* `runner.py`: runs configurations over the questions in parallel (8 workers), resumable, with invariance checks and a determinism pilot.
* `selection.py`: development-only choice of the top-3 ablation variants + the default, then **freezes** them before the test set is touched.
* `metrics.py`: script-computed measures (accuracy, Recall@k, MRR, full coverage, citation validity, latency, tokens, cost).
* `ragas_eval.py`: RAGAs-protocol metrics (faithfulness, answer relevance, context precision/recall) with fixed evaluator prompts.
* `triple_relevance.py`: judged relevance of the appended triples.
* `tests.py` / `analysis.py`: Appendix 6 statistics and every pre-specified test family.
* `figures.py`: all Chapter 5 figures and tables, computed only from saved logs, plus a results manifest.

In [ ]:
#@title kgrag/experiments/context.py
put('src/kgrag/experiments/context.py', r'''"""Evidence-context construction for configurations 0/A/B/C/D and controls, under the two budget modes (Sec. 3.5).

Evidence tokens = sum of generator-tokenizer counts of the rendered evidence items (passages [P#], graph statements
[G#] and the graph header). Instructions, question and options are excluded (identical across configurations).
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field

SENT = re.compile(r"(?<=[.!?;:])\s+(?=[A-Z0-9(\[])|\n+")


def render_passage(p: dict, label: str, text: str | None = None) -> str:
    return f"[{label}] {p['specification']} clause {p['clause_id']} - {p['heading'][:90]}\n{text if text is not None else p['text']}"


@dataclass
class Item:
    label: str
    kind: str                 # "P" passage | "G" graph statement | "H" graph header
    ref: str                  # passage_id / triple_id / ""
    text: str
    tokens: int
    trimmed: bool = False
    source_passage_ids: list = field(default_factory=list)
    score: float | None = None


@dataclass
class Context:
    items: list[Item]
    T_q: int | None = None
    graph_cap: int | None = None
    checks: dict = field(default_factory=dict)
    meta: dict = field(default_factory=dict)

    @property
    def passage_items(self):
        return [i for i in self.items if i.kind == "P"]

    @property
    def graph_items(self):
        return [i for i in self.items if i.kind == "G"]

    @property
    def evidence_tokens(self) -> int:
        return sum(i.tokens for i in self.items)

    @property
    def graph_tokens(self) -> int:
        return sum(i.tokens for i in self.items if i.kind in ("G", "H"))

    def text(self, passages_only: bool = False) -> str:
        its = [i for i in self.items if (i.kind == "P" or not passages_only)]
        return "\n\n".join(i.text for i in its)

    def labels(self) -> dict[str, Item]:
        return {i.label: i for i in self.items if i.kind in ("P", "G")}


def trim_to(p: dict, label: str, budget: int, count) -> Item | None:
    """Longest sentence-boundary prefix whose rendered item fits `budget` tokens (None if not even one sentence)."""
    sents = [s for s in SENT.split(p["text"]) if s.strip()]
    lo, hi, best = 1, len(sents), None
    while lo <= hi:  # binary search over prefix length (token count is monotone in prefix length)
        mid = (lo + hi) // 2
        txt = render_passage(p, label, " ".join(sents[:mid]))
        n = count(txt)
        if n <= budget:
            best, lo = Item(label, "P", p["passage_id"], txt, n, trimmed=mid < len(sents),
                            source_passage_ids=[p["passage_id"]]), mid + 1
        else:
            hi = mid - 1
    return best


def passage_items(pids: list[str], passages: dict, count, budget: int | None = None,
                  scores: list[float] | None = None) -> list[Item]:
    """Passages in the given order; if `budget` is set, whole passages while they fit, then the next one trimmed at a
    sentence boundary, then stop (lower-ranked passages omitted). Never reorders."""
    out, used = [], 0
    for k, pid in enumerate(pids):
        lab = f"P{k + 1}"
        txt = render_passage(passages[pid], lab)
        n = count(txt)
        sc = scores[k] if scores else None
        if budget is None or used + n <= budget:
            out.append(Item(lab, "P", pid, txt, n, source_passage_ids=[pid], score=sc))
            used += n
            continue
        it = trim_to(passages[pid], lab, budget - used, count)
        if it is not None:
            it.score = sc
            out.append(it)
        break
    return out


def graph_items(sel: dict, header_tokens: int, header_text: str) -> list[Item]:
    if not sel["items"]:
        return []
    its = [Item("", "H", "", header_text, header_tokens)]
    for k, g in enumerate(sel["items"]):
        its.append(Item(f"G{k + 1}", "G", g["triple_id"], g["text"], g["tokens"], source_passage_ids=g["source_passage_ids"]))
    return its


def check_matched(ctx: Context, tol: float, b_ids: list[str] | None = None) -> dict:
    T = ctx.T_q or 0
    tot = ctx.evidence_tokens
    chk = {"T_q": T, "evidence_tokens": tot, "ratio_to_T_q": (tot / T) if T else None,
           "within_tolerance": (abs(tot - T) <= tol * T) if T else tot == 0,
           "not_over_budget": tot <= (1 + tol) * T}
    if ctx.graph_cap is not None:
        chk["graph_tokens"] = ctx.graph_tokens
        chk["graph_cap_ok"] = ctx.graph_tokens <= ctx.graph_cap
    if b_ids is not None:
        c_ids = [i.ref for i in ctx.passage_items]
        chk["passage_prefix_of_B"] = c_ids == b_ids[: len(c_ids)]
    return chk
''')

In [ ]:
#@title kgrag/generation/generator.py
put('src/kgrag/generation/generator.py', r'''"""Stage 14: common generation interface (Sec. 3.4 Comp. 5, 4.9). Same model, prompt, decoding for every configuration.

Selected generator: models.yaml role `generator` (gemini-3.8-flash, temperature 0, thinking_level low) — selection
rationale recorded in reports/model_selection.md. Malformed / missing option => incorrect. Exceptions are logged.
"""
from __future__ import annotations

import re

from ..core import sha256_text
from ..llm import LLMClient

PROMPT_VERSION = "gen-v1.0"
SYSTEM = ("You are a 3GPP telecommunications standards expert answering multiple-choice questions. "
          "When evidence is supplied, base your answer on it and cite it; do not invent clause numbers.")
TEMPLATE = """EVIDENCE:
{evidence}

QUESTION: {question}
OPTIONS:
{options}

Instructions:
1. Select exactly one option by its number.
2. Give a concise technical justification (at most 120 words).
3. List the labels of the evidence items you relied on (e.g. "P1", "G2"). Use only labels that appear in EVIDENCE.
   If no evidence is supplied, return an empty list.
Return JSON with keys: answer_option, justification, citations."""
NO_EVIDENCE = "No evidence is supplied for this question. Answer from your own knowledge."


def schema(option_ids: list[str]) -> dict:
    return {"type": "object", "properties": {
        "answer_option": {"type": "string", "enum": option_ids},
        "justification": {"type": "string"},
        "citations": {"type": "array", "items": {"type": "string"}}},
        "required": ["answer_option", "justification", "citations"]}


def build_prompt(q: dict, evidence_text: str) -> str:
    opts = "\n".join(f"{k}: {v}" for k, v in sorted(q["options"].items(), key=lambda kv: int(kv[0])))
    return TEMPLATE.format(evidence=evidence_text or NO_EVIDENCE, question=q["question"], options=opts)


TEMPLATE_HASH = sha256_text(SYSTEM + TEMPLATE + NO_EVIDENCE)[:16]


def parse_option(data: dict | None, option_ids: list[str]) -> str | None:
    if not data:
        return None
    v = str(data.get("answer_option", "")).strip()
    m = re.search(r"(\d+)", v)
    v = m.group(1) if m else v
    return v if v in option_ids else None


def generate(llm: LLMClient, q: dict, evidence_text: str, seed: int) -> dict:
    ids = sorted(q["options"], key=int)
    prompt = build_prompt(q, evidence_text)
    r = llm.generate_json("generator", prompt, schema(ids), SYSTEM, seed=seed)
    opt = parse_option(r.data, ids)
    cits = [str(c).strip().strip("[]") for c in (r.data or {}).get("citations", []) or []] if r.data else []
    return {"parsed_option": opt, "answer_valid": opt is not None, "justification": (r.data or {}).get("justification", ""),
            "citations": cits, "raw_text": r.text, "generation_error": r.error, "generator_model": r.model,
            "generator_model_version": r.model_version, "input_tokens": r.input_tokens, "output_tokens": r.output_tokens,
            "thought_tokens": r.thought_tokens, "generation_latency_s": r.latency_s, "generation_cost_usd": r.cost_usd,
            "prompt_sha": sha256_text(prompt)[:16], "template_hash": TEMPLATE_HASH, "prompt_version": PROMPT_VERSION}
''')

In [ ]:
#@title kgrag/experiments/runner.py
put('src/kgrag/experiments/runner.py', r'''"""Stages 11-15: configurations 0/A/B/C/D + controls, budget modes, invariance checks, resumable logging.

Invariants enforced at run time (and unit-tested):
* C never re-ranks: its passage list is an in-order prefix of B's cached final list (`passage_prefix_of_B`).
* Retrieval/reranking recomputed for C must reproduce B's cached ranking (`ranking_invariance_ok`).
* KG seeds for C come ONLY from B's selected passages; D seeds come from all 30 candidates.
* Matched mode: total evidence tokens checked against T_q +-5%; graph tokens <= g*T_q.
"""
from __future__ import annotations

import math
import random
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

from ..core import Project, append_jsonl, iter_jsonl, read_json, stable_hash, utc_now, write_json
from ..generation import generator
from ..knowledge_graph.graph import GRAPH_HEADER, render_statement
from ..reranking.reranker import rerank as do_rerank
from ..retrieval.dense import question_text
from .context import Context, Item, check_matched, graph_items, passage_items, render_passage, trim_to


@dataclass(frozen=True)
class Condition:
    config: str                    # 0 | A | B | C | D | RANDOM | UNFILTERED
    mode: str = "matched"          # matched | overhead (C/controls only)
    triples: int | None = None
    hops: int | None = None
    alloc: float | None = None

    @property
    def name(self) -> str:
        if self.config in ("0", "B"):
            return self.config
        if self.config == "A":
            return "A_matched"
        return f"{self.config}_t{self.triples}_h{self.hops}_g{int(round(self.alloc * 100))}_{self.mode}"

    @property
    def uses_graph(self) -> bool:
        return self.config in ("C", "D", "RANDOM", "UNFILTERED")


def default_condition(project: Project, config: str = "C", mode: str = "matched") -> Condition:
    d = project.study["experiment"]["default_variant"]
    return Condition(config, mode, d["triples"], d["hops"], d["alloc"])


def ablation_grid(project: Project) -> list[Condition]:
    e = project.study["experiment"]
    return [Condition("C", "matched", t, h, a) for t in e["triple_budgets"] for h in e["hop_depths"]
            for a in e["graph_allocations"]]


class Pipeline:
    def __init__(self, project: Project, passages: dict, retriever, reranker, threshold: float, kg, count, llm,
                 gold: dict | None = None, groups: dict | None = None, env_id: str = ""):
        self.p, self.passages, self.retriever, self.reranker = project, passages, retriever, reranker
        self.threshold, self.kg, self.count, self.llm = threshold, kg, count, llm
        self.gold, self.groups, self.env_id = gold or {}, groups or {}, env_id
        rr = project.study["reranking"]
        self.max_k, self.min_k = rr["max_passages"], rr["min_passages"]
        self.tol = project.study["experiment"]["budget_tolerance"]
        self.refill = project.study["experiment"]["refill_unused_graph_allocation"]
        self.header_tokens = count(GRAPH_HEADER) if count else 0
        self.b_cache_path = project.path("interim", "b_rankings.jsonl")
        self._b_cache = {r["question_id"]: r for r in iter_jsonl(self.b_cache_path)}
        self._lock = threading.Lock()

    # ------------------------------------------------------------------ shared retrieval state (recomputed each call)
    def b_state(self, q: dict) -> dict:
        qt = question_text(q)
        cands, rl = self.retriever.retrieve(qt)
        sel, scored, kl = do_rerank(self.reranker, qt, cands, self.passages, self.threshold, self.max_k, self.min_k)
        st = {"candidates": cands, "scored": scored, "selected": sel, "retrieval_latency_s": rl, "rerank_latency_s": kl,
              "qvec": None}
        ids = [s["passage_id"] for s in sel]
        with self._lock:
            cached = self._b_cache.get(q["question_id"])
            if cached is None:
                cached = {"question_id": q["question_id"], "final_passage_ids": ids,
                          "rerank_scores": [s["rerank_score"] for s in sel],
                          "retrieved_passage_ids": [c["passage_id"] for c in cands], "threshold": self.threshold,
                          "utc": utc_now()}
                self._b_cache[q["question_id"]] = cached
                append_jsonl(self.b_cache_path, cached)
        st["b_cached_ids"] = cached["final_passage_ids"]
        st["ranking_invariance_ok"] = ids == cached["final_passage_ids"]
        return st

    def prefetch_counts(self, questions: list[dict]) -> None:
        """API-count mode only: count rendered A/B passages in parallel once (labels depend on rank)."""
        if not hasattr(self.count, "prefetch") or getattr(self.count, "local_fn", None) is not None:
            return
        texts = []
        for q in questions:
            st = self.b_state(q)
            texts += [render_passage(self.passages[c["passage_id"]], f"P{k + 1}") for k, c in enumerate(st["candidates"])]
            texts += [render_passage(self.passages[i], f"P{k + 1}") for k, i in enumerate(st["b_cached_ids"])]
        self.count.prefetch(texts)

    def qvec(self, q: dict):
        return self.retriever.encoder.encode_queries([question_text(q)])[0]

    # ------------------------------------------------------------------ context builders
    def _b_items(self, st: dict) -> list[Item]:
        ids = st["b_cached_ids"]  # B ranking reused verbatim (cached identifiers)
        sc = {s["passage_id"]: s["rerank_score"] for s in st["scored"]}
        return passage_items(ids, self.passages, self.count, None, [sc.get(i) for i in ids])

    def build(self, q: dict, cond: Condition, st: dict | None) -> tuple[Context, dict]:
        lat = {"kg_lookup_s": 0.0, "extra_rerank_s": 0.0}
        if cond.config == "0":
            return Context([], meta={"closed_book": True}), lat
        b_items = self._b_items(st)
        T = sum(i.tokens for i in b_items)
        b_ids = [i.ref for i in b_items]
        if cond.config == "B":
            ctx = Context(b_items, T_q=T)
            ctx.checks = {"T_q": T, "evidence_tokens": T, "within_tolerance": True}
            return ctx, lat
        if cond.config == "A":
            ids = [c["passage_id"] for c in st["candidates"]]
            ctx = Context(passage_items(ids, self.passages, self.count, T, [c["score"] for c in st["candidates"]]), T_q=T)
            ctx.checks = check_matched(ctx, self.tol)
            return ctx, lat
        cap = int(math.floor(cond.alloc * T))
        qv = self.qvec(q)
        if cond.config == "D":
            return self._build_d(q, cond, st, T, cap, qv, lat)
        seed_text = "\n".join(self.passages[i]["text"] for i in b_ids)  # seeds ONLY from B's final passages
        gcap = max(0, cap - self.header_tokens)
        kw = dict(qvec=qv, seed_text=seed_text, hops=cond.hops, count=self.count)
        c_sel = self.kg.select(**kw, triple_budget=cond.triples, token_cap=gcap, mode="selective")
        if cond.config == "C":
            sel = c_sel
        elif cond.config == "UNFILTERED":
            sel = self.kg.select(**kw, triple_budget=None, token_cap=gcap, mode="unfiltered")
        else:  # RANDOM: same count as C's realised triples, same allocation, random over the whole validated KG
            sel = self.kg.select(**kw, triple_budget=None, token_cap=gcap, mode="random",
                                 rng_seed=int(stable_hash([q["question_id"], cond.name], 8), 16),
                                 n_random=len(c_sel["items"]))
        lat["kg_lookup_s"] = sel["lookup_latency_s"] + (c_sel["lookup_latency_s"] if sel is not c_sel else 0)
        g_items = graph_items(sel, self.header_tokens, GRAPH_HEADER)
        g_tok = sum(i.tokens for i in g_items)
        if cond.mode == "overhead":
            ctx = Context(b_items + g_items, T_q=T, graph_cap=cap)
            ctx.checks = {"T_q": T, "evidence_tokens": ctx.evidence_tokens, "graph_tokens": g_tok,
                          "graph_cap_ok": g_tok <= cap, "added_tokens": g_tok, "passage_prefix_of_B": True}
        else:
            p_budget = T - (g_tok if self.refill else cap)
            p_items = passage_items(b_ids, self.passages, self.count, p_budget)
            ctx = Context(p_items + g_items, T_q=T, graph_cap=cap)
            ctx.checks = check_matched(ctx, self.tol, b_ids)
            ctx.checks["underfill_due_to_unused_graph_allocation"] = (not ctx.checks["within_tolerance"]
                                                                      and g_tok < cap and not self.refill)
        ctx.meta = {"requested_triples": cond.triples, "realised_triples": len(sel["items"]), "n_seeds": sel["n_seeds"],
                    "n_candidate_triples": sel["n_candidates"], "stop_reason": sel["stop_reason"],
                    "triple_hops": [g["hop"] for g in sel["items"]], "c_reference_triples": len(c_sel["items"])}
        return ctx, lat

    def _build_d(self, q, cond, st, T, cap, qv, lat):
        seed_text = "\n".join(self.passages[c["passage_id"]]["text"] for c in st["candidates"])  # all 30 candidates
        sel = self.kg.select(qvec=qv, seed_text=seed_text, hops=cond.hops, count=self.count,
                             triple_budget=cond.triples, token_cap=None, mode="selective")
        lat["kg_lookup_s"] = sel["lookup_latency_s"]
        trip = {g["triple_id"]: self.kg.by_id[g["triple_id"]] for g in sel["items"]}
        g_texts = [render_statement(t, "G") for t in trip.values()]
        g_scores, kl = self.reranker.score(question_text(q), g_texts) if g_texts else ([], 0.0)
        lat["extra_rerank_s"] = kl  # passages already scored in st["scored"]; graph statements scored jointly here
        pool = [("P", s["passage_id"], s["rerank_score"], s["rank"]) for s in st["scored"]] + \
               [("G", tid, sc, 10**6 + k) for k, (tid, sc) in enumerate(zip(trip, g_scores))]
        pool.sort(key=lambda x: (-x[2], x[3]))
        items, used, gused, np_, ng = [], 0, 0, 0, 0
        for kind, ref, sc, _ in pool:
            if used >= T:
                break
            if kind == "G":
                txt = render_statement(trip[ref], f"G{ng + 1}")
                n = self.count(txt)
                if gused + n <= cap and used + n <= T:
                    ng += 1
                    items.append(Item(f"G{ng}", "G", ref, txt, n, source_passage_ids=[p["passage_id"] for p in trip[ref]["provenance"]], score=sc))
                    used += n
                    gused += n
                continue
            lab = f"P{np_ + 1}"
            txt = render_passage(self.passages[ref], lab)
            n = self.count(txt)
            if used + n <= T:
                np_ += 1
                items.append(Item(lab, "P", ref, txt, n, source_passage_ids=[ref], score=sc))
                used += n
            else:
                it = trim_to(self.passages[ref], lab, T - used, self.count)
                if it is not None:
                    np_ += 1
                    it.score = sc
                    items.append(it)
                    used += it.tokens
                break
        ctx = Context(items, T_q=T, graph_cap=cap)
        ctx.checks = check_matched(ctx, self.tol)
        b_ids = st["b_cached_ids"]
        ctx.checks["passages_differ_from_B"] = [i.ref for i in ctx.passage_items] != b_ids
        ctx.meta = {"requested_triples": cond.triples, "realised_triples": ng, "n_seeds": sel["n_seeds"],
                    "n_candidate_triples": sel["n_candidates"], "graph_pool_size": len(trip), "stop_reason": "joint_budget_fill",
                    "triple_hops": [sel_i["hop"] for sel_i in sel["items"] if sel_i["triple_id"] in {i.ref for i in items}]}
        return ctx, lat

    # ------------------------------------------------------------------ one record
    def prepare(self, q: dict, cond: Condition) -> tuple:
        """Local stages (retrieval, reranking, KG lookup, context) — run sequentially on the main thread so that
        stage latencies are not distorted by concurrent GPU work."""
        t_start = time.perf_counter()
        st = None if cond.config == "0" else self.b_state(q)
        ctx, lat = self.build(q, cond, st)
        return st, ctx, lat, time.perf_counter() - t_start

    def run_one(self, q: dict, cond: Condition, run_id: int, seed: int, exp_id: str, split: str,
                prepared: tuple | None = None) -> dict:
        st, ctx, lat, local_s = prepared or self.prepare(q, cond)
        gen = generator.generate(self.llm, q, ctx.text(), seed)
        ref = q["answer_option_id"]
        labels = {l: {"kind": i.kind, "ref": i.ref, "source_passage_ids": i.source_passage_ids} for l, i in ctx.labels().items()}
        rl = st["retrieval_latency_s"] if st else 0.0
        kl = (st["rerank_latency_s"] if st else 0.0) + lat["extra_rerank_s"]
        if cond.config == "A":
            kl = 0.0  # A does not rerank; its latency excludes the (shared) reranking computation
        total = rl + kl + lat["kg_lookup_s"] + (gen["generation_latency_s"] or 0)
        rec = {
            "key": f"{exp_id}|{cond.name}|{q['question_id']}|{run_id}", "experiment_id": exp_id, "run_id": run_id,
            "seed": seed, "question_id": q["question_id"], "split": split, "configuration": cond.config,
            "condition": cond.name, "budget_mode": cond.mode if cond.uses_graph or cond.config == "A" else "n/a",
            "triple_budget": cond.triples, "hop_depth": cond.hops, "graph_alloc": cond.alloc,
            "hop_type": q.get("hop_type"), "question_type": q.get("question_type"),
            "retriever_group": self.groups.get(q["question_id"]),
            "retrieved_passage_ids": [c["passage_id"] for c in st["candidates"]] if st else [],
            "b_passage_ids": st["b_cached_ids"] if st else [],
            "ranking_list_ids": ([c["passage_id"] for c in st["candidates"]] if cond.config == "A" else
                                 (st["b_cached_ids"] if st else [])),
            "final_passage_ids": [i.ref for i in ctx.passage_items], "final_passage_trimmed": [i.trimmed for i in ctx.passage_items],
            "kg_triple_ids": [i.ref for i in ctx.graph_items], "context_labels": labels,
            "graph_tokens": ctx.graph_tokens, "evidence_tokens": ctx.evidence_tokens, "T_q": ctx.T_q,
            "graph_cap": ctx.graph_cap, "budget_checks": ctx.checks, "kg_meta": ctx.meta,
            "ranking_invariance_ok": st["ranking_invariance_ok"] if st else None,
            "context_text": ctx.text(), "context_text_passages_only": ctx.text(passages_only=True),
            "context_items": [i.text for i in ctx.items if i.kind in ("P", "G")],
            "parsed_option": gen["parsed_option"], "reference_option": ref,
            "correct": int(gen["parsed_option"] == ref), "answer_valid": gen["answer_valid"],
            "justification": gen["justification"], "citations": gen["citations"],
            "retrieval_latency_s": rl, "rerank_latency_s": kl, "kg_lookup_latency_s": lat["kg_lookup_s"],
            "generation_latency_s": gen["generation_latency_s"], "total_latency_s": total,
            "local_pipeline_wall_s": local_s,
            "input_tokens": gen["input_tokens"], "output_tokens": gen["output_tokens"], "thought_tokens": gen["thought_tokens"],
            "cost_usd": gen["generation_cost_usd"], "generator_model": gen["generator_model"],
            "generator_model_version": gen["generator_model_version"], "prompt_sha": gen["prompt_sha"],
            "template_hash": gen["template_hash"], "embedding_model": getattr(self.retriever.encoder, "name", None) if st else None,
            "reranker_model": getattr(self.reranker, "name", None) if st else None, "rerank_threshold": self.threshold,
            "config_hash": self.p.config_hash(), "env_id": self.env_id,
            "cache_policy": self.p.study["experiment"]["latency_cache_policy"],
            "status": "ok" if gen["generation_error"] is None else "generation_error",
            "error": gen["generation_error"], "utc": utc_now()}
        return rec


def run(pipe: Pipeline, questions: list[dict], conditions: list[Condition], exp_id: str, split: str,
        run_ids: list[int] | None = None, out_dir: str = "dev", workers: int | None = None) -> str:
    """Resumable: a (experiment, condition, question, run) key with status ok is never recomputed."""
    project = pipe.p
    out = project.path(out_dir, f"{exp_id}.jsonl")
    done = {r["key"] for r in iter_jsonl(out) if r.get("status") == "ok"}
    e = project.study["experiment"]
    run_ids = run_ids or list(range(1, int(_run_policy(project)["n_runs"]) + 1))
    order = sorted(questions, key=lambda q: q["question_id"])
    random.Random(e["question_order_seed"]).shuffle(order)  # one fixed permutation for all conditions
    sem = threading.Semaphore((workers or e["workers"]) * 2)

    def fail(q, c, rid, ex):  # recorded, never replaced by an invented answer
        append_jsonl(out, {"key": f"{exp_id}|{c.name}|{q['question_id']}|{rid}", "experiment_id": exp_id,
                           "condition": c.name, "question_id": q["question_id"], "run_id": rid, "status": "exception",
                           "error": f"{type(ex).__name__}: {ex}", "utc": utc_now()})

    def job(q, c, rid, seed, prepared):
        try:
            append_jsonl(out, pipe.run_one(q, c, rid, seed, exp_id, split, prepared))
        except Exception as ex:
            fail(q, c, rid, ex)
        finally:
            sem.release()

    with ThreadPoolExecutor(workers or e["workers"]) as ex:
        for rid in run_ids:
            seed = e["run_seeds"][rid - 1]
            for c in conditions:
                for q in order:
                    if f"{exp_id}|{c.name}|{q['question_id']}|{rid}" in done:
                        continue
                    try:
                        prepared = pipe.prepare(q, c)  # local stages: main thread, sequential
                    except Exception as err:
                        fail(q, c, rid, err)
                        continue
                    sem.acquire()
                    ex.submit(job, q, c, rid, seed, prepared)  # hosted generation: parallel
    return str(out)


def _run_policy(project: Project) -> dict:
    p = project.path("reports", "run_policy.json")
    return read_json(p) if p.exists() else {"n_runs": project.study["experiment"]["n_runs"], "source": "study.yaml default"}


def determinism_pilot(pipe: Pipeline, dev_questions: list[dict], n: int = 30) -> dict:
    """Sec. 3.7: run B twice on the same development questions with the same seed; any disagreement -> 3 runs."""
    qs = sorted(dev_questions, key=lambda q: q["question_id"])[:n]
    for r in (1, 2):
        run(pipe, qs, [Condition("B")], f"pilot_determinism_r{r}", "development", [1], "dev")
    from ..core import read_jsonl
    a = {x["question_id"]: x.get("parsed_option") for x in read_jsonl(pipe.p.path("dev", "pilot_determinism_r1.jsonl")) if x.get("status") == "ok"}
    b = {x["question_id"]: x.get("parsed_option") for x in read_jsonl(pipe.p.path("dev", "pilot_determinism_r2.jsonl")) if x.get("status") == "ok"}
    common = sorted(set(a) & set(b))
    diff = [k for k in common if a[k] != b[k]]
    res = {"n_compared": len(common), "n_different": len(diff), "different_ids": diff,
           "n_runs": 3 if diff else 1, "decided_utc": utc_now(), "source": "determinism pilot (development only)"}
    write_json(pipe.p.path("reports", "run_policy.json"), res)
    return res
''')

In [ ]:
#@title kgrag/experiments/selection.py
put('src/kgrag/experiments/selection.py', r'''"""Stage 13: development-only variant selection and parameter freezing (Sec. 3.5, Table 4.5)."""
from __future__ import annotations

import pandas as pd

from ..core import Project, read_jsonl, utc_now, write_json, read_json
from .runner import Condition, default_condition


def select_variants(project: Project, dev_log: str) -> dict:
    lock = project.path("reports", "frozen_parameters.json")
    if lock.exists():
        raise RuntimeError("Parameters already frozen; development selection must not be repeated after test evaluation.")
    df = pd.DataFrame([r for r in read_jsonl(dev_log) if r.get("status") == "ok" and r["configuration"] == "C"])
    if df.empty:
        raise RuntimeError("No successful development ablation records")
    per_q = df.groupby(["condition", "question_id"]).agg(correct=("correct", "mean"), graph_tokens=("graph_tokens", "mean"),
                                                          t=("triple_budget", "first"), h=("hop_depth", "first"),
                                                          a=("graph_alloc", "first")).reset_index()
    per_q["correct"] = (per_q.correct >= 0.5).astype(int)  # majority over runs if 3 runs
    agg = per_q.groupby("condition").agg(accuracy=("correct", "mean"), n=("question_id", "nunique"),
                                         mean_graph_tokens=("graph_tokens", "mean"), triples=("t", "first"),
                                         hops=("h", "first"), alloc=("a", "first")).reset_index()
    if agg.n.nunique() != 1:
        raise RuntimeError(f"Unequal question coverage across variants: {agg.n.unique().tolist()} — finish the ablation first")
    if len(agg) != 18:
        raise RuntimeError(f"Expected 18 variants, found {len(agg)}")
    agg = agg.sort_values(["accuracy", "mean_graph_tokens", "alloc", "triples", "hops"],
                          ascending=[False, True, True, True, True]).reset_index(drop=True)
    agg["selection_rank"] = range(1, len(agg) + 1)
    n_top = project.study["experiment"]["n_top_variants"]
    top = agg.head(n_top)
    d = default_condition(project)
    chosen = [{"triples": int(r.triples), "hops": int(r.hops), "alloc": float(r.alloc), "reason": f"top-{r.selection_rank}"}
              for r in top.itertuples()]
    if d.name not in set(top.condition):
        chosen.append({"triples": d.triples, "hops": d.hops, "alloc": d.alloc, "reason": "predefined default"})
    else:
        for c in chosen:
            if (c["triples"], c["hops"], c["alloc"]) == (d.triples, d.hops, d.alloc):
                c["reason"] += " + predefined default"
    agg.to_csv(project.path("dev", "ablation_development_summary.csv"), index=False)
    thr = read_json(project.path("reports", "rerank_threshold.json"))
    pol = read_json(project.path("reports", "run_policy.json")) if project.path("reports", "run_policy.json").exists() else {}
    frozen = {"frozen_utc": utc_now(), "rule": project.study["experiment"]["variant_selection_rule"],
              "selected_variants": chosen, "rerank_threshold": thr["threshold"], "n_runs": pol.get("n_runs", 1),
              "config_hash": project.config_hash(), "development_log": str(dev_log)}
    write_json(lock, frozen)
    return frozen


def heldout_conditions(project: Project) -> list[Condition]:
    fz = read_json(project.path("reports", "frozen_parameters.json"))
    v = [Condition("C", "matched", c["triples"], c["hops"], c["alloc"]) for c in fz["selected_variants"]]
    d = default_condition(project)
    conds = [Condition("0"), Condition("A"), Condition("B")] + v
    conds += [Condition("C", "overhead", c.triples, c.hops, c.alloc) for c in v]
    conds += [Condition("D", "matched", d.triples, d.hops, d.alloc),
              Condition("RANDOM", "matched", d.triples, d.hops, d.alloc),
              Condition("UNFILTERED", "matched", d.triples, d.hops, d.alloc)]
    return list(dict.fromkeys(conds))
''')

In [ ]:
#@title kgrag/evaluation/metrics.py
put('src/kgrag/evaluation/metrics.py', r'''"""Stage 16 (script-computed constructs, Table 3.6). Each construct is computed independently."""
from __future__ import annotations


def retrieval_metrics(ranking_ids: list[str], context_ids: list[str], gold: set[str], ks=(1, 3, 5)) -> dict:
    """Recall@k / MRR on the configuration's passage RANKING (B and C share it by design) and full-coverage recall on
    the passages actually supplied. Gold = independently validated passages (never the experimental retriever)."""
    out = {}
    for k in ks:
        out[f"recall_at_{k}"] = int(any(p in gold for p in ranking_ids[:k])) if ranking_ids else 0
    rr = 0.0
    for i, p in enumerate(ranking_ids, 1):
        if p in gold:
            rr = 1.0 / i
            break
    out["mrr"] = rr if ranking_ids else 0.0
    out["context_recall_any"] = int(bool(gold & set(context_ids)))
    out["full_coverage"] = int(bool(gold) and gold <= set(context_ids))
    return out


def citation_metrics(citations: list[str], labels: dict, gold: set[str]) -> dict:
    """Validity: cited label exists in the supplied context. Support: cited item's source passage is gold.
    Graph statements count as citations of their source clause and are flagged graph-sourced. None if no citations."""
    cits = [c for c in dict.fromkeys(citations) if c]
    if not cits:
        return {"n_citations": 0, "citation_validity": None, "citation_support": None,
                "n_graph_citations": 0, "graph_citation_support": None, "passage_citation_support": None}
    valid = [c for c in cits if c in labels]
    sup = lambda c: int(bool(set(labels[c]["source_passage_ids"]) & gold))
    g = [c for c in valid if labels[c]["kind"] == "G"]
    p = [c for c in valid if labels[c]["kind"] == "P"]
    return {"n_citations": len(cits), "citation_validity": len(valid) / len(cits),
            "citation_support": sum(sup(c) for c in valid) / len(cits),
            "n_graph_citations": len(g), "graph_citation_support": (sum(sup(c) for c in g) / len(g)) if g else None,
            "passage_citation_support": (sum(sup(c) for c in p) / len(p)) if p else None}


def triple_metrics(labels: dict, gold: set[str], valid_passage_ids: set[str], judged: dict[str, int] | None = None,
                   release_of: dict | None = None, release: str = "17") -> dict:
    g = [v for v in labels.values() if v["kind"] == "G"]
    if not g:
        return {"n_triples": 0, "provenance_validity": None, "gold_linked_precision": None, "judged_relevant_precision": None}
    prov = [int(all(s in valid_passage_ids and (release_of or {}).get(s, release) == release for s in v["source_passage_ids"])
                and bool(v["source_passage_ids"])) for v in g]
    linked = [int(bool(set(v["source_passage_ids"]) & gold)) for v in g]
    jr = None
    if judged is not None:
        vals = [judged.get(v["ref"]) for v in g]
        vals = [x for x in vals if x is not None]
        jr = sum(vals) / len(vals) if vals else None
    return {"n_triples": len(g), "provenance_validity": sum(prov) / len(g), "gold_linked_precision": sum(linked) / len(g),
            "judged_relevant_precision": jr}
''')

In [ ]:
#@title kgrag/evaluation/ragas_eval.py
put('src/kgrag/evaluation/ragas_eval.py', r'''"""RAGAs-protocol evaluator (Sec. 3.6): faithfulness (full context and passages-only), answer relevance, context
precision and context recall, computed with a FIXED evaluator model (role `evaluator`, distinct from generator and
verifier).

Implementation note (documented deviation, compliance matrix C-24): the metric definitions and two-step procedures of
RAGAs (Es et al., 2024) are re-implemented here with fixed prompts so that the evaluator can be a Gemini model through
the same audited client. The `ragas` package API changes frequently and is not pinned for Gemini 3.x; scores are
therefore labelled "RAGAs-protocol". Missing scores are recorded as None and NEVER imputed (Sec. 3.7).
"""
from __future__ import annotations

import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np

from ..core import Project, append_jsonl, iter_jsonl, utc_now
from ..llm import LLMClient

PROMPT_VERSION = "ragas-protocol-v1.0"

ST_SCHEMA = {"type": "object", "properties": {"statements": {"type": "array", "items": {"type": "string"}}},
             "required": ["statements"]}
NLI_SCHEMA = {"type": "object", "properties": {"verdicts": {"type": "array", "items": {"type": "object", "properties": {
    "n": {"type": "integer"}, "verdict": {"type": "integer", "enum": [0, 1]}, "reason": {"type": "string"}},
    "required": ["n", "verdict"]}}}, "required": ["verdicts"]}
AR_SCHEMA = {"type": "object", "properties": {"questions": {"type": "array", "items": {"type": "string"}},
                                              "noncommittal": {"type": "integer", "enum": [0, 1]}},
             "required": ["questions", "noncommittal"]}
CP_SCHEMA = {"type": "object", "properties": {"verdicts": {"type": "array", "items": {"type": "object", "properties": {
    "n": {"type": "integer"}, "useful": {"type": "integer", "enum": [0, 1]}}, "required": ["n", "useful"]}}},
    "required": ["verdicts"]}

P_STATEMENTS = """Break the ANSWER into simple, self-contained factual statements (resolve pronouns). Return JSON {{"statements": [...]}}.
QUESTION: {question}
ANSWER: {answer}"""
P_NLI = """For each STATEMENT decide whether it can be directly inferred from the CONTEXT (1) or not (0).
CONTEXT:
{context}

STATEMENTS:
{statements}"""
P_AR = """Generate {n} questions that the ANSWER below would answer. Also set noncommittal = 1 if the answer is evasive,
vague or says it does not know, else 0.
ANSWER: {answer}"""
P_CP = """REFERENCE ANSWER to the QUESTION: {reference}
QUESTION: {question}
For each numbered CONTEXT item decide whether it was useful in arriving at the reference answer (1) or not (0).
{items}"""
P_CR = """For each STATEMENT of the REFERENCE ANSWER decide whether it can be attributed to the CONTEXT (1) or not (0).
CONTEXT:
{context}

STATEMENTS:
{statements}"""


def _verdicts(r, n: int, key: str = "verdict") -> list[int] | None:
    if not r.ok:
        return None
    d = {int(v.get("n", 0)): int(v.get(key, 0)) for v in r.data.get("verdicts", [])}
    return [d.get(i + 1, 0) for i in range(n)] if n else []


def answer_text(rec: dict, q: dict) -> str:
    opt = rec.get("parsed_option")
    chosen = f"Selected option {opt}: {q['options'].get(opt, '')}. " if opt else ""
    return (chosen + (rec.get("justification") or "")).strip()


def evaluate_record(llm: LLMClient, emb, rec: dict, q: dict, n_q: int = 3) -> dict:
    t0 = time.perf_counter()
    out = {"key": rec["key"], "prompt_version": PROMPT_VERSION, "evaluator_model": llm.model_of("evaluator"),
           "costs": [], "errors": []}
    ans = answer_text(rec, q)
    log = lambda r: (out["costs"].append(r.cost_usd or 0.0), r.error and out["errors"].append(r.error))
    # --- faithfulness (two context versions)
    stm = None
    if rec.get("context_text") and ans:
        r = llm.generate_json("evaluator", P_STATEMENTS.format(question=q["question"], answer=ans), ST_SCHEMA)
        log(r)
        stm = [s for s in (r.data or {}).get("statements", []) if s.strip()] if r.ok else None
    for name, ctx in (("faithfulness_full", rec.get("context_text")), ("faithfulness_passages", rec.get("context_text_passages_only"))):
        if not ctx or stm is None or len(stm) == 0:
            out[name] = None
            continue
        r = llm.generate_json("evaluator", P_NLI.format(context=ctx, statements="\n".join(f"{i + 1}. {s}" for i, s in enumerate(stm))), NLI_SCHEMA)
        log(r)
        v = _verdicts(r, len(stm))
        out[name] = (sum(v) / len(v)) if v else None
    out["n_statements"] = len(stm) if stm else 0
    # --- answer relevance (mean cosine of generated questions to the original; 0 if noncommittal)
    if ans:
        r = llm.generate_json("evaluator", P_AR.format(n=n_q, answer=ans), AR_SCHEMA)
        log(r)
        if r.ok and r.data.get("questions"):
            gq = [x for x in r.data["questions"] if x.strip()][:n_q]
            E = emb.encode_queries([q["question"]] + gq)
            cos = float(np.mean(E[1:] @ E[0]))
            out["answer_relevance"] = 0.0 if int(r.data.get("noncommittal", 0)) else cos
        else:
            out["answer_relevance"] = None
    else:
        out["answer_relevance"] = None
    # --- context precision / recall against the TeleQnA reference
    reference = f"{q['options'].get(q['answer_option_id'], '')}. {q.get('explanation', '')}".strip()
    items = rec.get("context_items") or []
    if items:
        r = llm.generate_json("evaluator", P_CP.format(reference=reference, question=q["question"],
                              items="\n\n".join(f"{i + 1}. {t}" for i, t in enumerate(items))), CP_SCHEMA)
        log(r)
        v = _verdicts(r, len(items), "useful")
        if v is not None:
            num, hits = 0.0, 0
            for k, u in enumerate(v, 1):
                if u:
                    hits += 1
                    num += hits / k
            out["context_precision"] = (num / hits) if hits else 0.0
        else:
            out["context_precision"] = None
        r = llm.generate_json("evaluator", P_STATEMENTS.format(question=q["question"], answer=reference), ST_SCHEMA)
        log(r)
        rs = [s for s in (r.data or {}).get("statements", []) if s.strip()] if r.ok else []
        if rs:
            r2 = llm.generate_json("evaluator", P_CR.format(context=rec["context_text"], statements="\n".join(
                f"{i + 1}. {s}" for i, s in enumerate(rs))), NLI_SCHEMA)
            log(r2)
            v = _verdicts(r2, len(rs))
            out["context_recall"] = (sum(v) / len(v)) if v else None
        else:
            out["context_recall"] = None
    else:
        out["context_precision"] = out["context_recall"] = None
    out["evaluation_latency_s"] = time.perf_counter() - t0
    out["evaluation_cost_usd"] = sum(out.pop("costs"))
    out["status"] = "ok" if not out["errors"] else "partial_error"
    out["utc"] = utc_now()
    return out


def run(project: Project, llm: LLMClient, emb, gen_log: str, questions: dict, conditions: set[str] | None,
        out_name: str, workers: int | None = None) -> str:
    out = project.path("test" if "heldout" in out_name else "dev", out_name)
    done = {r["key"] for r in iter_jsonl(out) if r.get("status") == "ok"}
    recs = [r for r in iter_jsonl(gen_log) if r.get("status") == "ok" and r["key"] not in done
            and (conditions is None or r["condition"] in conditions)]
    with ThreadPoolExecutor(workers or project.study["experiment"]["workers"]) as ex:
        futs = [ex.submit(evaluate_record, llm, emb, r, questions[r["question_id"]]) for r in recs]
        for f in as_completed(futs):
            append_jsonl(out, f.result())
    return str(out)
''')

In [ ]:
#@title kgrag/evaluation/triple_relevance.py
put('src/kgrag/evaluation/triple_relevance.py', r'''"""Judged triple relevance (Sec. 3.6): an independent verifier (role `verifier`), given the question, keyed option and
the triple's source passage, judges whether each appended triple helps establish the keyed answer. One judgement per
(question, triple) — identical across conditions, so it cannot favour any configuration."""
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed

from ..core import Project, append_jsonl, iter_jsonl, utc_now
from ..knowledge_graph.fidelity import statement
from ..llm import LLMClient

PROMPT_VERSION = "triple-relevance-v1.0"
SCHEMA = {"type": "object", "properties": {"judgements": {"type": "array", "items": {"type": "object", "properties": {
    "n": {"type": "integer"}, "helpful": {"type": "integer", "enum": [0, 1]}}, "required": ["n", "helpful"]}}},
    "required": ["judgements"]}
TEMPLATE = """QUESTION: {question}
KEYED (correct) OPTION: {key}

For each numbered knowledge-graph statement (with its source passage), decide whether it helps establish that the keyed
option is correct (1) or not (0).

{items}"""


def judge_question(llm: LLMClient, q: dict, triples: list[dict], passages: dict) -> list[dict]:
    items = "\n\n".join(f"{i + 1}. STATEMENT: {statement(t['subject'], t['predicate'], t['object'])}\n   SOURCE "
                        f"[{t['provenance'][0]['passage_id']}]: {passages[t['provenance'][0]['passage_id']]['text'][:1500]}"
                        for i, t in enumerate(triples))
    r = llm.generate_json("verifier", TEMPLATE.format(question=q["question"],
                                                      key=q["options"].get(q["answer_option_id"], ""), items=items), SCHEMA)
    d = {int(j.get("n", 0)): int(j.get("helpful", 0)) for j in (r.data or {}).get("judgements", [])} if r.ok else {}
    return [{"question_id": q["question_id"], "triple_id": t["triple_id"], "helpful": d.get(i + 1),
             "status": "ok" if (i + 1) in d else "error", "model": r.model, "prompt_version": PROMPT_VERSION,
             "cost_usd": r.cost_usd if i == 0 else 0.0, "utc": utc_now()} for i, t in enumerate(triples)]


def run(project: Project, llm: LLMClient, gen_logs: list[str], questions: dict, kg_by_id: dict, passages: dict,
        workers: int | None = None, batch: int = 15) -> str:
    out = project.path("test", "triple_relevance.jsonl")
    done = {(r["question_id"], r["triple_id"]) for r in iter_jsonl(out) if r.get("status") == "ok"}
    need: dict[str, set] = {}
    for lg in gen_logs:
        for r in iter_jsonl(lg):
            if r.get("status") == "ok":
                for tid in r.get("kg_triple_ids", []):
                    if (r["question_id"], tid) not in done:
                        need.setdefault(r["question_id"], set()).add(tid)
    jobs = [(questions[qid], [kg_by_id[t] for t in sorted(ts)[i:i + batch]])
            for qid, ts in need.items() for i in range(0, len(ts), batch)]
    with ThreadPoolExecutor(workers or project.study["experiment"]["workers"]) as ex:
        for f in as_completed([ex.submit(judge_question, llm, q, ts, passages) for q, ts in jobs]):
            for row in f.result():
                append_jsonl(out, row)
    return str(out)


def load(project: Project) -> dict[tuple[str, str], int]:
    return {(r["question_id"], r["triple_id"]): r["helpful"] for r in iter_jsonl(project.path("test", "triple_relevance.jsonl"))
            if r.get("status") == "ok"}
''')

In [ ]:
#@title kgrag/statistics/tests.py
put('src/kgrag/statistics/tests.py', r'''"""Appendix 6 statistical functions, hardened for edge cases without changing the design (Sec. 3.7, Tables 3.7-3.8)."""
from __future__ import annotations

import numpy as np
from scipy.stats import rankdata, wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

SEED, B_RESAMPLES = 42, 10_000


def _pairs(ref, alt) -> tuple[np.ndarray, np.ndarray, int]:
    ref, alt = np.asarray(ref, float), np.asarray(alt, float)
    ok = ~(np.isnan(ref) | np.isnan(alt))  # pairwise exclusion; never impute (Sec. 3.7)
    return ref[ok], alt[ok], int((~ok).sum())


def mcnemar_exact(ref, alt) -> dict:
    """ref, alt: 0/1 over the same questions. b = correct only in reference, c = correct only in compared."""
    ref, alt, n_excl = _pairs(ref, alt)
    ref, alt = ref.astype(int), alt.astype(int)
    n11 = int(((ref == 1) & (alt == 1)).sum())
    b = int(((ref == 1) & (alt == 0)).sum())
    c = int(((ref == 0) & (alt == 1)).sum())
    n00 = int(((ref == 0) & (alt == 0)).sum())
    p = 1.0 if b + c == 0 else float(mcnemar([[n11, b], [c, n00]], exact=True).pvalue)
    n = len(ref)
    return {"test": "mcnemar_exact", "n": n, "n_excluded_pairs": n_excl, "b": b, "c": c,
            "ref_rate": ref.mean() if n else np.nan, "alt_rate": alt.mean() if n else np.nan,
            "diff_pp": 100 * (alt.mean() - ref.mean()) if n else np.nan, "p_raw": p,
            "note": "no discordant pairs" if b + c == 0 else ""}


def rank_biserial(d: np.ndarray) -> float:
    d = d[d != 0]
    if not len(d):
        return 0.0
    r = rankdata(np.abs(d))
    wp, wm = r[d > 0].sum(), r[d < 0].sum()
    return float((wp - wm) / (wp + wm))


def wilcoxon_paired(ref, alt) -> dict:
    ref, alt, n_excl = _pairs(ref, alt)
    d = alt - ref
    nz = int((d != 0).sum())
    res = {"test": "wilcoxon_signed_rank", "n": len(d), "n_excluded_pairs": n_excl, "n_nonzero": nz,
           "mean_diff": float(d.mean()) if len(d) else np.nan, "median_diff": float(np.median(d)) if len(d) else np.nan,
           "rank_biserial_r": rank_biserial(d) if len(d) else np.nan}
    if len(d) == 0:
        res.update(p_raw=np.nan, note="no valid pairs")
    elif nz == 0:
        res.update(p_raw=1.0, note="all differences zero")
    else:
        res.update(p_raw=float(wilcoxon(alt, ref, zero_method="wilcox", alternative="two-sided", method="auto").pvalue),
                   note="" if nz >= 6 else "fewer than 6 non-zero differences: low power")
    return res


def paired_bootstrap_ci(ref, alt, stat=np.mean, n_boot: int = B_RESAMPLES, seed: int = SEED, scale: float = 1.0):
    ref, alt, _ = _pairs(ref, alt)
    d = (alt - ref) * scale
    if len(d) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))  # question-level resampling, same for both configurations
    boots = stat(d[idx], axis=1)
    return tuple(float(x) for x in np.percentile(boots, [2.5, 97.5]))


def bootstrap_mean_ci(x, n_boot: int = B_RESAMPLES, seed: int = SEED, scale: float = 1.0):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)] * scale
    if not len(x):
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    b = x[rng.integers(0, len(x), size=(n_boot, len(x)))].mean(axis=1)
    return tuple(float(v) for v in np.percentile(b, [2.5, 97.5]))


def holm(p_values) -> list[float]:
    p = np.asarray(p_values, float)
    out = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    if ok.any():
        out[ok] = multipletests(p[ok], alpha=0.05, method="holm")[1]
    return out.tolist()
''')

In [ ]:
#@title kgrag/statistics/analysis.py
put('src/kgrag/statistics/analysis.py', r'''"""Consolidated per-question log and every pre-specified test family (Sec. 3.7, Table 3.8, Appendix 6).

Raw logs (append-only JSONL) are never modified; this module derives analysis tables from them.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..core import Project, iter_jsonl, read_json, write_json, utc_now
from ..evaluation.metrics import citation_metrics, retrieval_metrics, triple_metrics
from .tests import B_RESAMPLES, SEED, bootstrap_mean_ci, holm, mcnemar_exact, paired_bootstrap_ci, wilcoxon_paired

CONT = ["faithfulness_full", "faithfulness_passages", "answer_relevance", "context_precision", "context_recall",
        "mrr", "citation_validity", "citation_support", "gold_linked_precision", "judged_relevant_precision",
        "latency_total_s", "retrieval_latency_s", "rerank_latency_s", "kg_lookup_latency_s", "generation_latency_s",
        "input_tokens", "output_tokens", "graph_tokens", "evidence_tokens", "cost_usd", "provenance_validity",
        "evaluation_latency_s", "evaluation_cost_usd"]
BIN = ["correct", "answer_valid", "recall_at_1", "recall_at_3", "recall_at_5", "full_coverage", "context_recall_any"]


def per_run_table(gen_logs: list[str], eval_logs: list[str], gold: dict, valid_pids: set, judged: dict) -> pd.DataFrame:
    ev = {}
    for lg in eval_logs:
        for r in iter_jsonl(lg):
            if r.get("status") in ("ok", "partial_error"):
                ev[r["key"]] = r
    rows = []
    for lg in gen_logs:
        for r in iter_jsonl(lg):
            if r.get("status") != "ok":
                continue
            g = set(gold[r["question_id"]]["reference_passage_ids"])
            labels = r.get("context_labels", {})
            row = {k: r.get(k) for k in ("key", "experiment_id", "run_id", "question_id", "split", "configuration",
                                         "condition", "budget_mode", "triple_budget", "hop_depth", "graph_alloc",
                                         "hop_type", "question_type", "retriever_group", "correct", "answer_valid",
                                         "retrieval_latency_s", "rerank_latency_s", "kg_lookup_latency_s",
                                         "generation_latency_s", "input_tokens", "output_tokens", "graph_tokens",
                                         "evidence_tokens", "T_q", "cost_usd", "ranking_invariance_ok")}
            row["latency_total_s"] = r.get("total_latency_s")
            bc = r.get("budget_checks") or {}
            row.update({f"chk_{k}": v for k, v in bc.items() if isinstance(v, (bool, int, float)) or v is None})
            km = r.get("kg_meta") or {}
            row.update(realised_triples=km.get("realised_triples"), requested_triples=km.get("requested_triples"),
                       kg_stop_reason=km.get("stop_reason"))
            if r["configuration"] != "0":
                row.update(retrieval_metrics(r.get("ranking_list_ids", []), r.get("final_passage_ids", []), g))
            row.update(citation_metrics(r.get("citations", []), labels, g))
            row.update(triple_metrics(labels, g, valid_pids, {t: judged.get((r["question_id"], t)) for t in r.get("kg_triple_ids", [])}))
            e = ev.get(r["key"], {})
            for k in ("faithfulness_full", "faithfulness_passages", "answer_relevance", "context_precision",
                      "context_recall", "evaluation_latency_s", "evaluation_cost_usd"):
                row[k] = e.get(k)
            rows.append(row)
    return pd.DataFrame(rows)


def per_question_table(runs: pd.DataFrame) -> pd.DataFrame:
    """Repeated runs: majority correctness for binary accuracy, mean for continuous measures (Sec. 3.7)."""
    keys = ["experiment_id", "condition", "question_id"]
    first = runs.groupby(keys, as_index=False).first()[keys + ["split", "configuration", "budget_mode", "triple_budget",
                                                              "hop_depth", "graph_alloc", "hop_type", "question_type",
                                                              "retriever_group"]]
    num = runs.copy()
    for c in CONT + BIN:
        if c in num:
            num[c] = pd.to_numeric(num[c], errors="coerce")
    agg = num.groupby(keys).agg({**{c: "mean" for c in CONT if c in num}, **{c: "mean" for c in BIN if c in num},
                                 "run_id": "nunique"}).reset_index().rename(columns={"run_id": "n_runs"})
    agg["correct"] = (agg["correct"] >= 0.5).astype(int)  # majority of 3 (or the single run)
    return first.merge(agg, on=keys)


def _pivot(pq: pd.DataFrame, metric: str, a: str, b: str, sub: pd.Series | None = None):
    d = pq if sub is None else pq[sub]
    w = d.pivot_table(index="question_id", columns="condition", values=metric, aggfunc="first")
    if a not in w or b not in w:
        return None, None
    return w[a].to_numpy(float), w[b].to_numpy(float)


def compare(pq, family, label, ref, alt, metric, test, sub=None, scale_ci=1.0) -> dict:
    x, y = _pivot(pq, metric, ref, alt, sub)
    base = {"family": family, "comparison": label, "reference": ref, "compared": alt, "metric": metric}
    if x is None:
        return {**base, "note": "condition missing", "p_raw": np.nan}
    res = mcnemar_exact(x, y) if test == "mcnemar_exact" else wilcoxon_paired(x, y)
    scale = 100.0 if test == "mcnemar_exact" else scale_ci
    lo, hi = paired_bootstrap_ci(x, y, n_boot=B_RESAMPLES, seed=SEED, scale=scale)
    return {**base, **res, "ci95_low": lo, "ci95_high": hi, "ci_units": "percentage points" if scale == 100 else "metric units"}


def run_families(project: Project, pq: pd.DataFrame) -> pd.DataFrame:
    fz = read_json(project.path("reports", "frozen_parameters.json"))
    d = project.study["experiment"]["default_variant"]
    tag = lambda t, h, a: f"t{t}_h{h}_g{int(round(a * 100))}"
    dflt = tag(d["triples"], d["hops"], d["alloc"])
    C, D = f"C_{dflt}_matched", f"D_{dflt}_matched"
    R, U, CO = f"RANDOM_{dflt}_matched", f"UNFILTERED_{dflt}_matched", f"C_{dflt}_overhead"
    variants = list(dict.fromkeys(f"C_{tag(v['triples'], v['hops'], v['alloc'])}_matched" for v in fz["selected_variants"]))
    rows = [compare(pq, "1_confirmatory", "B vs C(default), matched", "B", C, "correct", "mcnemar_exact")]
    fam = lambda name, items: [{**compare(pq, name, *it), } for it in items]
    rows += fam("2_variants", [(f"B vs {v}", "B", v, "correct", "mcnemar_exact") for v in variants])
    rows += fam("3_controls", [("C(default) vs random-triple", C, R, "correct", "mcnemar_exact"),
                               ("C(default) vs unfiltered-expansion", C, U, "correct", "mcnemar_exact")])
    rows += fam("4_placement", [("C(default) vs D(default)", C, D, "correct", "mcnemar_exact")])
    subs = {"single-hop": pq.hop_type == "single-hop", "multi-hop": pq.hop_type == "multi-hop",
            "retriever-found": pq.retriever_group == "retriever-found",
            "independent-only": pq.retriever_group == "independent-only"}
    rows += [compare(pq, "5_subgroups", f"B vs C(default) [{k}]", "B", C, "correct", "mcnemar_exact", s) for k, s in subs.items()]
    rows += fam("6_secondary", [(f"B vs C(default): {m}", "B", C, m, "wilcoxon") for m in
                                ["faithfulness_full", "faithfulness_passages", "answer_relevance", "context_precision",
                                 "context_recall", "citation_validity", "citation_support"]])
    rows += fam("7_cost", [("B vs C(default): end-to-end latency", "B", C, "latency_total_s", "wilcoxon")])
    # exploratory, outside Table 3.8 (reported with Holm within family, clearly labelled)
    rows += fam("8_exploratory_retrieval", [("A vs B: Recall@5", "A_matched", "B", "recall_at_5", "mcnemar_exact"),
                                            ("A vs B: full coverage", "A_matched", "B", "full_coverage", "mcnemar_exact"),
                                            ("B vs D: full coverage", "B", D, "full_coverage", "mcnemar_exact"),
                                            ("C vs D: full coverage", C, D, "full_coverage", "mcnemar_exact")])
    rows += fam("9_exploratory_overhead_mode", [(f"B vs {v.replace('_matched', '_overhead')}", "B",
                                                 v.replace("_matched", "_overhead"), "correct", "mcnemar_exact") for v in variants])
    rows += fam("10_exploratory_references", [("0 (closed-book) vs B", "0", "B", "correct", "mcnemar_exact"),
                                              ("A vs B", "A_matched", "B", "correct", "mcnemar_exact")])
    df = pd.DataFrame(rows)
    df["p_adjusted"] = np.nan
    for f, g in df.groupby("family"):
        if f == "1_confirmatory":
            df.loc[g.index, "p_adjusted"] = g.p_raw  # single prespecified test: not adjusted
        else:
            df.loc[g.index, "p_adjusted"] = holm(g.p_raw.tolist())
    df["adjustment"] = np.where(df.family == "1_confirmatory", "none", "holm within family")
    sig = (df.p_adjusted < 0.05) & ((df.ci95_low > 0) | (df.ci95_high < 0))
    df["interpretation"] = np.where(sig, np.where(df.ci95_low > 0, "improvement (compared > reference)",
                                                  "decrease (compared < reference)"),
                                    "no supported difference (not evidence of equivalence)")
    return df


def descriptive(pq: pd.DataFrame) -> pd.DataFrame:
    out = []
    for cond, g in pq.groupby("condition"):
        r = {"condition": cond, "configuration": g.configuration.iloc[0], "budget_mode": g.budget_mode.iloc[0], "n": len(g)}
        for m in ["correct", "recall_at_1", "recall_at_3", "recall_at_5", "full_coverage", "mrr", "faithfulness_full",
                  "faithfulness_passages", "answer_relevance", "context_precision", "context_recall",
                  "citation_validity", "citation_support", "gold_linked_precision", "judged_relevant_precision",
                  "provenance_validity"]:
            if m in g:
                v = pd.to_numeric(g[m], errors="coerce")
                r[m], r[f"{m}_n"] = v.mean(), int(v.notna().sum())
        lo, hi = bootstrap_mean_ci(g.correct, scale=100)
        r.update(accuracy_pct=100 * g.correct.mean(), accuracy_ci95_low=lo, accuracy_ci95_high=hi)
        for m in ["latency_total_s", "retrieval_latency_s", "rerank_latency_s", "kg_lookup_latency_s", "generation_latency_s",
                  "input_tokens", "output_tokens", "graph_tokens", "evidence_tokens", "cost_usd"]:
            if m in g:
                v = pd.to_numeric(g[m], errors="coerce").dropna()
                if len(v):
                    r.update({f"{m}_median": v.median(), f"{m}_iqr": v.quantile(.75) - v.quantile(.25),
                              f"{m}_p95": v.quantile(.95), f"{m}_mean": v.mean()})
        if "cost_usd" in g:
            c = pd.to_numeric(g.cost_usd, errors="coerce")
            r["generation_cost_usd_per_1000_questions"] = 1000 * c.mean() if c.notna().any() else None
        if "evaluation_cost_usd" in g:
            c = pd.to_numeric(g.evaluation_cost_usd, errors="coerce")
            r["evaluation_cost_usd_per_1000_questions"] = 1000 * c.mean() if c.notna().any() else None
        out.append(r)
    return pd.DataFrame(out)


def quality_cost(desc: pd.DataFrame, reference: str = "B") -> pd.DataFrame:
    """Accuracy gain per +1k tokens and per +100 ms vs B; Pareto efficiency; 'limited value' flags (Sec. 3.6)."""
    ref = desc.set_index("condition").loc[reference]
    rows = []
    for _, r in desc.iterrows():
        if r.condition == reference:
            continue
        dacc = r.accuracy_pct - ref.accuracy_pct
        dtok = r.get("input_tokens_median", np.nan) - ref.get("input_tokens_median", np.nan)
        dlat = 1000 * (r.get("latency_total_s_median", np.nan) - ref.get("latency_total_s_median", np.nan))
        rows.append({"condition": r.condition, "budget_mode": r.budget_mode, "delta_accuracy_pp": dacc,
                     "delta_input_tokens_median": dtok, "delta_latency_ms_median": dlat,
                     "acc_pp_per_1k_extra_tokens": dacc / (dtok / 1000) if dtok and dtok > 0 else None,
                     "acc_pp_per_100ms_extra": dacc / (dlat / 100) if dlat and dlat > 0 else None,
                     "ratio_note": "undefined: no additional cost" if not (dtok and dtok > 0) else "",
                     "limited_value_flag": bool(dacc > 0 and any(
                         pd.notna(r.get(m)) and pd.notna(ref.get(m)) and r.get(m) < ref.get(m)
                         for m in ("faithfulness_full", "citation_support"))),
                     "accuracy_pct": r.accuracy_pct, "input_tokens_median": r.get("input_tokens_median"),
                     "latency_total_s_median": r.get("latency_total_s_median")})
    qc = pd.DataFrame(rows)
    allc = pd.concat([qc, pd.DataFrame([{"condition": reference, "accuracy_pct": ref.accuracy_pct,
                                          "input_tokens_median": ref.get("input_tokens_median"),
                                          "latency_total_s_median": ref.get("latency_total_s_median")}])])
    eff = []
    for _, r in allc.iterrows():
        dom = ((allc.accuracy_pct > r.accuracy_pct) & (allc.input_tokens_median <= r.input_tokens_median)).any() or \
              ((allc.accuracy_pct >= r.accuracy_pct) & (allc.input_tokens_median < r.input_tokens_median)).any()
        eff.append({"condition": r.condition, "pareto_efficient_tokens": not dom})
    return allc.merge(pd.DataFrame(eff), on="condition")


def budget_compliance(pq_runs: pd.DataFrame) -> pd.DataFrame:
    m = pq_runs[pq_runs.budget_mode == "matched"]
    cols = [c for c in ["chk_within_tolerance", "chk_not_over_budget", "chk_graph_cap_ok", "chk_passage_prefix_of_B",
                        "chk_underfill_due_to_unused_graph_allocation", "ranking_invariance_ok"] if c in m]
    return m.groupby("condition")[cols].agg(lambda s: pd.to_numeric(s, errors="coerce").mean()).reset_index()
''')

In [ ]:
#@title kgrag/reporting/figures.py
put('src/kgrag/reporting/figures.py', r'''"""Stage 18: Chapter 5 figures and tables, computed ONLY from the saved logs, plus a results manifest."""
from __future__ import annotations

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ..core import Project, utc_now, write_json

SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
# Categorical identity follows the configuration family (fixed order, never cycled).
FAMILY_COLOR = {"0": "#52514e", "A": "#2a78d6", "B": "#eb6834", "C": "#1baf7a", "D": "#6250d6",
                "RANDOM": "#eda100", "UNFILTERED": "#e87ba4"}
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]


def _style(ax, title: str, xlabel: str = "", ylabel: str = ""):
    ax.set_facecolor(SURFACE)
    ax.figure.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_title(title, color=INK, fontsize=11, loc="left")
    ax.set_xlabel(xlabel, color=INK2, fontsize=9)
    ax.set_ylabel(ylabel, color=INK2, fontsize=9)


def _save(fig, project: Project, name: str, manifest: list, sources: list[str], script: str = "kgrag.reporting.figures"):
    path = project.path("figures", name)
    fig.tight_layout()
    fig.savefig(path, dpi=220, facecolor=SURFACE)
    fig.savefig(path.with_suffix(".svg"), facecolor=SURFACE)
    plt.close(fig)
    manifest.append({"artifact": str(path.relative_to(project.root)), "script": script, "source_logs": sources,
                     "created_utc": utc_now()})


def _table(df: pd.DataFrame, project: Project, name: str, manifest: list, sources: list[str]):
    path = project.path("tables", name)
    df.to_csv(path, index=False)
    manifest.append({"artifact": str(path.relative_to(project.root)), "script": "kgrag.reporting.figures",
                     "source_logs": sources, "created_utc": utc_now()})


def make_all(project: Project, desc: pd.DataFrame, tests: pd.DataFrame, qc: pd.DataFrame, pq: pd.DataFrame,
             dev_summary: pd.DataFrame | None, extra_tables: dict[str, pd.DataFrame], sources: list[str]) -> list:
    man: list = []
    order = sorted(desc.condition, key=lambda c: (list(FAMILY_COLOR).index(c.split("_")[0]), c))
    d = desc.set_index("condition").loc[order].reset_index()
    col = [FAMILY_COLOR[c.split("_")[0]] for c in d.condition]

    # accuracy_comparison.png — bars with bootstrap 95% CIs
    fig, ax = plt.subplots(figsize=(10, 4.6))
    x = np.arange(len(d))
    ax.bar(x, d.accuracy_pct, color=col, width=0.62, edgecolor=SURFACE, linewidth=2)
    ax.errorbar(x, d.accuracy_pct, yerr=[d.accuracy_pct - d.accuracy_ci95_low, d.accuracy_ci95_high - d.accuracy_pct],
                fmt="none", ecolor=INK, elinewidth=1, capsize=3)
    ax.set_xticks(x, d.condition, rotation=40, ha="right")
    for xi, v in zip(x, d.accuracy_pct):
        ax.text(xi, v + 1, f"{v:.1f}", ha="center", va="bottom", fontsize=8, color=INK2)
    _style(ax, "Answer accuracy by condition (held-out test set; 95% bootstrap CI)", "", "Accuracy (%)")
    ax.set_ylim(0, 105)
    _save(fig, project, "accuracy_comparison.png", man, sources)

    # retrieval_mrr.csv + figure
    rcols = ["condition", "recall_at_1", "recall_at_3", "recall_at_5", "mrr", "full_coverage"]
    _table(d[[c for c in rcols if c in d]], project, "retrieval_mrr.csv", man, sources)

    # faithfulness: full vs passages-only
    if "faithfulness_full" in d:
        f = d.dropna(subset=["faithfulness_full"])
        if len(f):
            fig, ax = plt.subplots(figsize=(9, 4.2))
            x = np.arange(len(f))
            ax.bar(x - 0.18, f.faithfulness_full, 0.34, color="#2a78d6", label="Full context (passages + graph)")
            ax.bar(x + 0.18, f.faithfulness_passages, 0.34, color="#eb6834", label="Passages only")
            ax.set_xticks(x, f.condition, rotation=40, ha="right")
            ax.legend(frameon=False, fontsize=8)
            _style(ax, "Faithfulness: full context vs passages only (RAGAs protocol)", "", "Mean faithfulness")
            ax.set_ylim(0, 1.05)
            _save(fig, project, "faithfulness_full_vs_passages.png", man, sources)

    # latency_comparison.png — median with p95 whisker per stage-sum
    if "latency_total_s_median" in d:
        fig, ax = plt.subplots(figsize=(10, 4.4))
        x = np.arange(len(d))
        ax.bar(x, 1000 * d.latency_total_s_median, color=col, width=0.62, edgecolor=SURFACE, linewidth=2)
        ax.scatter(x, 1000 * d.latency_total_s_p95, marker="_", s=180, color=INK, label="95th percentile")
        ax.set_xticks(x, d.condition, rotation=40, ha="right")
        ax.legend(frameon=False, fontsize=8)
        _style(ax, "End-to-end latency per question (median bar, p95 tick)", "", "Milliseconds")
        _save(fig, project, "latency_comparison.png", man, sources)

    # tokens
    if "input_tokens_median" in d:
        fig, ax = plt.subplots(figsize=(10, 4.2))
        x = np.arange(len(d))
        ax.bar(x, d.input_tokens_median, color=col, width=0.62)
        ax.set_xticks(x, d.condition, rotation=40, ha="right")
        _style(ax, "Median input tokens per question (generator tokenizer)", "", "Tokens")
        _save(fig, project, "input_tokens.png", man, sources)

    # quality_cost_pareto.png
    q = qc.dropna(subset=["input_tokens_median"])
    if len(q):
        fig, ax = plt.subplots(figsize=(8, 5))
        for _, r in q.iterrows():
            c = FAMILY_COLOR[r.condition.split("_")[0]]
            ax.scatter(r.input_tokens_median, r.accuracy_pct, s=60, color=c,
                       edgecolor=INK if r.pareto_efficient_tokens else SURFACE, linewidth=1.5, zorder=3)
            ax.annotate(r.condition, (r.input_tokens_median, r.accuracy_pct), fontsize=7, color=INK2,
                        xytext=(4, 3), textcoords="offset points")
        ef = q[q.pareto_efficient_tokens].sort_values("input_tokens_median")
        ax.plot(ef.input_tokens_median, ef.accuracy_pct, color=INK2, linewidth=1, linestyle="--", zorder=2,
                label="Pareto frontier (dark outline = efficient)")
        ax.legend(frameon=False, fontsize=8)
        _style(ax, "Quality-cost trade-off", "Median input tokens per question", "Accuracy (%)")
        _save(fig, project, "quality_cost_pareto.png", man, sources)

    # subgroup accuracy
    subs = []
    for name, mask in {"single-hop": pq.hop_type == "single-hop", "multi-hop": pq.hop_type == "multi-hop",
                       "retriever-found": pq.retriever_group == "retriever-found",
                       "independent-only": pq.retriever_group == "independent-only"}.items():
        for cond, g in pq[mask].groupby("condition"):
            subs.append({"subgroup": name, "condition": cond, "n": len(g), "accuracy_pct": 100 * g.correct.mean()})
    sdf = pd.DataFrame(subs)
    _table(sdf, project, "subgroup_accuracy.csv", man, sources)
    key = [c for c in ["B", next((c for c in order if c.startswith("C_") and c.endswith("matched")), None)] if c]
    s2 = sdf[sdf.condition.isin(key)] if len(sdf) else sdf
    if len(s2):
        fig, ax = plt.subplots(figsize=(8, 4))
        groups = list(dict.fromkeys(s2.subgroup))
        x = np.arange(len(groups))
        for k, cnd in enumerate(key):
            vals = [s2[(s2.subgroup == g) & (s2.condition == cnd)].accuracy_pct.mean() for g in groups]
            ax.bar(x + (k - 0.5) * 0.36, vals, 0.34, color=FAMILY_COLOR[cnd.split("_")[0]], label=cnd)
        ax.set_xticks(x, groups)
        ax.legend(frameon=False, fontsize=8)
        _style(ax, "Accuracy by subgroup: B vs C (default, matched)", "", "Accuracy (%)")
        _save(fig, project, "subgroup_accuracy.png", man, sources)

    # kg_ablation_heatmap.png (development only)
    if dev_summary is not None and len(dev_summary):
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
        vmin, vmax = 100 * dev_summary.accuracy.min(), 100 * dev_summary.accuracy.max()
        for ax, h in zip(axes, sorted(dev_summary.hops.unique())):
            g = dev_summary[dev_summary.hops == h].pivot_table(index="triples", columns="alloc", values="accuracy") * 100
            im = ax.imshow(g.values, cmap=matplotlib.colors.LinearSegmentedColormap.from_list("seq", SEQ),
                           vmin=vmin, vmax=vmax, aspect="auto")
            ax.set_xticks(range(len(g.columns)), [f"{int(a * 100)}%" for a in g.columns])
            ax.set_yticks(range(len(g.index)), g.index)
            for i in range(g.shape[0]):
                for j in range(g.shape[1]):
                    v = g.values[i, j]
                    ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8,
                            color="#ffffff" if v > (vmin + vmax) / 2 else INK)
            ax.set_title(f"{h}-hop", fontsize=10, color=INK, loc="left")
            ax.set_xlabel("Graph-token allocation", color=INK2, fontsize=9)
        axes[0].set_ylabel("Triple budget", color=INK2, fontsize=9)
        fig.colorbar(im, ax=axes, label="Development accuracy (%)", shrink=0.85)
        fig.suptitle("KG ablation on the development split (18 variants; selection only, not tested)", fontsize=10,
                     color=INK, x=0.02, ha="left")
        fig.set_facecolor(SURFACE)
        path = project.path("figures", "kg_ablation_heatmap.png")
        fig.savefig(path, dpi=220, facecolor=SURFACE, bbox_inches="tight")
        plt.close(fig)
        man.append({"artifact": str(path.relative_to(project.root)), "script": "kgrag.reporting.figures",
                    "source_logs": [str(project.path("dev", "ablation_development_summary.csv"))], "created_utc": utc_now()})
        _table(dev_summary, project, "kg_ablation_development.csv", man, sources)

    # allocation sensitivity (test, selected variants) + matched vs overhead
    c = d[d.condition.str.startswith("C_")]
    if len(c):
        _table(c[["condition", "budget_mode", "accuracy_pct", "accuracy_ci95_low", "accuracy_ci95_high"] +
                 [x for x in ["graph_tokens_median", "input_tokens_median"] if x in c]], project,
               "matched_vs_overhead.csv", man, sources)

    _table(desc, project, "descriptive_by_condition.csv", man, sources)
    _table(tests, project, "statistical_tests.csv", man, sources)
    _table(qc, project, "quality_cost.csv", man, sources)
    for n, t in extra_tables.items():
        _table(t, project, n, man, sources)
    write_json(project.root / "results" / "results_manifest.json", {"created_utc": utc_now(), "artifacts": man})
    return man
''')

### 2.8 Workflow: the stage functions that Part 4 calls
`workflow.py` is a thin orchestration layer. Each function (`gate1`, `stage3_4`, `pools`, `verify`, …) reloads its inputs
from disk, does one stage, saves its outputs and updates the gate tracker. It is imported as **`W`** in Part 3.

In [ ]:
#@title kgrag/workflow.py: stage functions (imported as W)
put('src/kgrag/workflow.py', r'''"""Stage-by-stage orchestration used by notebooks/KG_RAG_Raju_KSR_v2.ipynb. Every function reloads its inputs from
disk (Colab runtimes restart), is resumable, and updates the gate tracker only from actual outputs."""
from __future__ import annotations

import json
import os
import subprocess
import sys

import pandas as pd

from .core import Project, Tracker, read_json, read_jsonl, write_json, utc_now
from . import env as envmod


# ------------------------------------------------------------------ helpers
def project(root: str) -> Project:
    return Project.load(root)


def llm(p: Project):
    from .llm import GeminiClient
    return GeminiClient(p.models)


def passages_dict(p: Project, retrieval_only: bool = False) -> dict:
    from .data.corpus import load_passages
    return {x["passage_id"]: x for x in load_passages(p, retrieval_only)}


def candidates_dict(p: Project) -> dict:
    return {q["question_id"]: q for q in read_jsonl(p.path("interim", "teleqna_candidates.jsonl"))}


def cost_of(path, key="cost_usd") -> float:
    tot = 0.0
    for r in read_jsonl(path):
        if "calls" in r:
            tot += sum((c.get("cost_usd") or 0) for c in r["calls"])
        else:
            tot += r.get(key) or r.get("batch_cost_usd") or 0
    return tot


# ------------------------------------------------------------------ Gate 1
def gate1(p: Project) -> dict:
    t = Tracker(p)
    e = envmod.capture(p)
    tests = subprocess.run([sys.executable, "-m", "pytest", "-q", str(p.root / "tests")], capture_output=True, text=True)
    (p.path("reports", "unit_tests.txt")).write_text(tests.stdout[-5000:] + tests.stderr[-2000:])
    client = llm(p)
    avail = set(client.list_models())
    roles = {r: (c["model"], c["model"] in avail) for r, c in p.models["roles"].items()}
    probe = {}
    for r, (m, ok) in roles.items():
        if ok:
            res = client.generate_json(r, 'Return JSON {"ok": true}', {"type": "object", "properties": {"ok": {"type": "boolean"}}, "required": ["ok"]})
            probe[r] = {"model": m, "ok": res.ok, "model_version": res.model_version, "error": res.error}
        else:
            probe[r] = {"model": m, "ok": False, "error": "model id not listed for this key"}
    out = {"tests_passed": tests.returncode == 0, "tests_tail": tests.stdout.strip().splitlines()[-1:],
           "role_probes": probe, "gpu": e["gpu"]}
    write_json(p.path("reports", "gate1_check.json"), out)
    ok = out["tests_passed"] and all(v["ok"] for v in probe.values())
    t.set(1, "VALIDATED" if ok else "BLOCKED", ["reports/environment.json", "reports/gate1_check.json"],
          "" if ok else "see gate1_check.json")
    return out


# ------------------------------------------------------------------ Gates 2-3
def stage3_4(p: Project, hf_token: str | None, check_archive: bool = True) -> dict:
    from .data import teleqna, tspec, corpus
    t = Tracker(p)
    teleqna.download(p)
    cand = teleqna.load_candidates(p)
    dl = tspec.download(p, token=hf_token)
    man = tspec.build_manifest(p, check_archive=check_archive)
    docs = man["documents"]
    verified = sum(d["status"] == "VERIFIED" for d in docs)
    t.set(2, "VALIDATED" if verified == len(p.study["corpus"]["specs"]) and not man["mixed_releases"] else "TESTED",
          "data/manifests/corpus_manifest.json", f"{verified}/{len(docs)} specs verified against 3GPP archive")
    stats = corpus.build(p)
    ok = (stats.duplicate_ids.sum() == 0 and stats.empty_passages.sum() == 0 and (stats.normative_lost <= 0).all()
          and stats.version_consistent.all())
    t.set(3, "VALIDATED" if ok else "TESTED", ["reports/corpus_quality_report.md", "results/corpus_statistics.csv"])
    return {"candidates": len(cand), "download": dl, "verified_specs": verified,
            "passages": int(stats.n_passages.sum()), "corpus_checks_ok": bool(ok)}


# ------------------------------------------------------------------ Gate 4
def pools(p: Project) -> int:
    from .retrieval.encoders import e5
    from .validation.pooling import build_pools
    from .data.corpus import load_passages
    out = build_pools(p, list(candidates_dict(p).values()), load_passages(p), e5())
    return len(out)


def verify(p: Project, limit: int | None = None) -> dict:
    from .validation import verifier
    pl = {r["question_id"]: r["pool"] for r in read_jsonl(p.path("interim", "candidate_pools.jsonl"))}
    path = verifier.run(p, llm(p), list(candidates_dict(p).values()), pl, passages_dict(p), limit=limit)
    recs = verifier.latest_records(p)
    n_ok = sum(r.get("status") == "ok" for r in recs.values())
    spent = cost_of(path)
    res = {"verified_ok": n_ok, "errors": sum(r.get("status") != "ok" for r in recs.values()), "cost_usd_so_far": spent,
           "projected_total_cost_usd": spent / max(1, n_ok) * len(candidates_dict(p)),
           "decisions": pd.Series([r.get("decision") for r in recs.values() if r.get("status") == "ok"]).value_counts().to_dict(),
           "reason_codes": pd.Series([r.get("reason_code") for r in recs.values() if r.get("status") == "ok"]).value_counts().to_dict()}
    if limit is None and res["errors"] == 0 and n_ok == len(candidates_dict(p)):
        Tracker(p).set(4, "VALIDATED", "data/interim/verifier_results.jsonl")
    return res


def register_and_audit(p: Project, round_no: int = 1) -> dict:
    from .validation import register, audit, verifier
    recs = verifier.latest_records(p)
    cand = candidates_dict(p)
    reg = register.build(p, list(cand.values()), recs)
    register.save(p, reg, register.flow_table(reg, recs))
    pl = {r["question_id"]: r["pool"] for r in read_jsonl(p.path("interim", "candidate_pools.jsonl"))}
    sheet, key = audit.make_question_audit(p, reg, cand, passages_dict(p), recs, pl, round_no)
    Tracker(p).set(5, "BLOCKED", sheet, "waiting for blind manual audit")
    return {"sheet": sheet, "included": int((reg.decision == "include").sum()), "excluded": int((reg.decision == "exclude").sum())}


# ------------------------------------------------------------------ Gate 5
def score_and_freeze(p: Project, round_no: int = 1) -> dict:
    from .validation import register, audit, verifier
    from .benchmark import freeze
    sc = audit.score_question_audit(p, round_no)
    if sc["status"] != "PASS":
        Tracker(p).set(5, "BLOCKED", f"reports/audit/question_audit_round{round_no}_score.json",
                       f"audit {sc['status']}: revise verifier prompt and audit a fresh sample (Sec. 3.3c)")
        return sc
    corr = pd.read_json(p.path("audit", f"question_audit_round{round_no}_corrections.json"))
    recs = verifier.latest_records(p)
    cand = candidates_dict(p)
    reg = register.build(p, list(cand.values()), recs, corr)
    register.save(p, reg, register.flow_table(reg, recs))
    n = int((reg.decision == "include").sum())
    if n < p.study["benchmark"]["min_questions"]:
        Tracker(p).set(5, "BLOCKED", "reports/inclusion_exclusion_log.csv",
                       f"only {n} questions qualify (< {p.study['benchmark']['min_questions']}): apply corpus expansion (TS 23.401, TS 33.501) and repeat Stages 3-6")
        return {**sc, "n_included": n, "status": "BELOW_MINIMUM"}
    man = freeze.freeze(p, reg, cand, passages_dict(p))
    Tracker(p).set(5, "VALIDATED", ["data/processed/FROZEN.json", f"reports/audit/question_audit_round{round_no}_score.json"])
    return {**sc, **man}


# ------------------------------------------------------------------ Gate 6
def retrieval_and_threshold(p: Project) -> dict:
    from .benchmark import freeze
    from .retrieval.dense import Retriever, run_and_label, question_text
    from .retrieval.encoders import bge
    from .reranking.reranker import CrossEncoderReranker, choose_threshold
    from .data.corpus import load_passages
    gq, ge, dev, test = freeze.load(p)
    ps = load_passages(p)
    pd_ = {x["passage_id"]: x for x in ps}
    enc = bge(p.study["retrieval"]["query_instruction"])
    ret = Retriever.build(p, enc, ps)
    groups = run_and_label(p, ret, gq, ge)
    rr = CrossEncoderReranker(p.study["reranking"]["model"], p.study["reranking"]["max_length"])
    top = {r["question_id"]: r["candidates"] for r in read_jsonl(p.path("interim", "bge_top30.jsonl"))}
    qd = {q["question_id"]: q for q in gq}
    dev_scored = {}
    for qid in dev:
        sc, _ = rr.score(question_text(qd[qid]), [pd_[c["passage_id"]]["text"] for c in top[qid]])
        dev_scored[qid] = [{**c, "rerank_score": s} for c, s in zip(top[qid], sc)]
    thr = choose_threshold(dev_scored, {q: set(ge[q]["reference_passage_ids"]) for q in dev},
                           p.study["reranking"]["max_passages"], p.study["reranking"]["min_passages"])
    thr.update(model=rr.name, chosen_utc=utc_now())
    write_json(p.path("reports", "rerank_threshold.json"), thr)
    share = pd.Series(groups).value_counts(normalize=True).to_dict()
    write_json(p.path("reports", "retrieval_diagnostics.json"), {"retriever_group_share": share, "n": len(groups)})
    Tracker(p).set(6, "VALIDATED", ["data/manifests/retrieval_index.json", "reports/rerank_threshold.json",
                                    "data/processed/retriever_groups.json"])
    return {"threshold": thr, "retriever_group_share": share}


# ------------------------------------------------------------------ Gate 7
def kg_extract(p: Project, limit: int | None = None) -> dict:
    from .knowledge_graph import extract
    from .data.corpus import load_passages
    path = extract.run(p, llm(p), load_passages(p), limit=limit)
    recs = read_jsonl(path)
    ok = {r["passage_id"] for r in recs if r.get("status") == "ok"}
    spent = cost_of(path)
    n = len(load_passages(p))
    return {"passages_done": len(ok), "of": n, "cost_usd_so_far": spent,
            "projected_total_cost_usd": spent / max(1, len(ok)) * n,
            "triples_so_far": sum(len(r["triples"]) for r in recs if r.get("status") == "ok")}


def kg_verify_and_build(p: Project) -> dict:
    from .knowledge_graph import fidelity, graph
    from .validation import audit
    ps = passages_dict(p)
    inst = []
    for rec in read_jsonl(p.path("interim", "kg_extraction.jsonl")):
        if rec.get("status") == "ok":
            for t in rec["triples"]:
                if t["schema_valid"] and t["quote_found"]:
                    inst.append({**t, "instance_id": f"{rec['passage_id']}#{t['local_id']}", "passage_id": rec["passage_id"]})
    fidelity.run(p, llm(p), inst, ps)
    all_tr, validated, stats = graph.assemble(p, ps)
    verified = [t for t in all_tr if t["fidelity_verdict"] in ("entailed", "not_entailed")]  # went through the verifier
    sheet, _ = audit.make_triple_audit(p, pd.DataFrame(verified), ps)
    Tracker(p).set(7, "BLOCKED", ["reports/kg_statistics.json", sheet], "waiting for blind 200-triple audit")
    return {**stats, "audit_sheet": sheet}


def kg_audit_score(p: Project) -> dict:
    from .validation import audit
    sc = audit.score_triple_audit(p)
    if sc["status"] == "SCORED":
        Tracker(p).set(7, "VALIDATED", "reports/audit/triple_audit_score.json",
                       f"fidelity precision {sc['fidelity_precision']:.3f}")
    return sc


# ------------------------------------------------------------------ pipeline (Gates 8-10)
def pipeline(p: Project):
    from .benchmark import freeze
    from .data.corpus import load_passages
    from .knowledge_graph import graph
    from .llm import TokenCounter, local_gemini_tokenizer, calibrate_local_tokenizer
    from .reranking.reranker import CrossEncoderReranker
    from .retrieval.dense import Retriever
    from .retrieval.encoders import bge
    from .experiments.runner import Pipeline
    import numpy as np
    gq, ge, dev, test = freeze.load(p)
    ps = load_passages(p)
    enc = bge(p.study["retrieval"]["query_instruction"])
    V = np.load(p.path("interim", "bge_passage_embeddings.npy"))
    ret = Retriever(enc, ps, V, p.study["retrieval"]["top_k"])
    rr = CrossEncoderReranker(p.study["reranking"]["model"], p.study["reranking"]["max_length"])
    thr = read_json(p.path("reports", "rerank_threshold.json"))["threshold"]
    kg = graph.load(p, enc)
    client = llm(p)
    calp = p.path("reports", "tokenizer_calibration.json")
    fn, name = local_gemini_tokenizer(client.model_of("generator"))
    if fn is not None:
        if not calp.exists():
            import random
            sample = random.Random(p.seed).sample([x["text"] for x in ps], 200)
            write_json(calp, {**calibrate_local_tokenizer(client, fn, sample), "local_tokenizer": name,
                              "generator": client.model_of("generator")})
        if not read_json(calp)["use_local"]:
            fn = None
    else:
        write_json(calp, {"use_local": False, "reason": name})
    count = TokenCounter(client, p.path("cache", "token_counts.jsonl"), local_fn=fn, method="local" if fn else "api")
    groups = read_json(p.path("processed", "retriever_groups.json"))
    env_id = read_json(p.path("reports", "environment.json"))["config_hash"]
    pipe = Pipeline(p, {x["passage_id"]: x for x in ps}, ret, rr, thr, kg, count, client, ge, groups, env_id)
    write_json(p.path("reports", "kg_query_index.json"), {"triples": len(kg.triples), "build_seconds": kg.build_seconds,
                                                           "peak_graph_memory_mb": kg.peak_memory_mb})
    qd = {q["question_id"]: q for q in gq}
    return pipe, [qd[i] for i in dev], [qd[i] for i in test]


def gate8_smoke(p: Project, pipe, dev_q: list, n: int = 15) -> dict:
    """Development-only smoke test of every configuration; NOT a thesis finding."""
    from .experiments.runner import Condition, default_condition, run, determinism_pilot
    d = default_condition(p)
    conds = [Condition("0"), Condition("A"), Condition("B"), d, Condition("C", "overhead", d.triples, d.hops, d.alloc),
             Condition("D", "matched", d.triples, d.hops, d.alloc), Condition("RANDOM", "matched", d.triples, d.hops, d.alloc),
             Condition("UNFILTERED", "matched", d.triples, d.hops, d.alloc)]
    out = run(pipe, dev_q[:n], conds, "smoke_dev", "development", [1], "dev")
    recs = [r for r in read_jsonl(out)]
    ok = [r for r in recs if r.get("status") == "ok"]
    c = [r for r in ok if r["configuration"] == "C" and r["budget_mode"] == "matched"]
    chk = {"records": len(recs), "ok": len(ok), "errors": [r.get("error") for r in recs if r.get("status") != "ok"][:5],
           "invalid_answers": sum(not r["answer_valid"] for r in ok),
           "C_prefix_ok": all(r["budget_checks"].get("passage_prefix_of_B") for r in c),
           "ranking_invariance_ok": all(r["ranking_invariance_ok"] for r in ok if r["ranking_invariance_ok"] is not None),
           "graph_cap_ok": all(r["budget_checks"].get("graph_cap_ok", True) for r in ok),
           "within_tolerance_share_C": sum(r["budget_checks"]["within_tolerance"] for r in c) / max(1, len(c)),
           "underfill_share_C": sum(r["budget_checks"].get("underfill_due_to_unused_graph_allocation", False) for r in c) / max(1, len(c)),
           "mean_cost_per_generation_usd": sum((r.get("cost_usd") or 0) for r in ok) / max(1, len(ok))}
    chk["determinism"] = determinism_pilot(pipe, dev_q)
    write_json(p.path("reports", "gate8_smoke.json"), chk)
    passed = chk["ok"] == chk["records"] and chk["C_prefix_ok"] and chk["ranking_invariance_ok"] and chk["graph_cap_ok"]
    Tracker(p).set(8, "VALIDATED" if passed else "BLOCKED", "reports/gate8_smoke.json")
    return chk


def dev_ablation(p: Project, pipe, dev_q: list) -> dict:
    from .experiments.runner import Condition, ablation_grid, run
    from .experiments.selection import select_variants
    out = run(pipe, dev_q, [Condition("B")] + ablation_grid(p), "dev_ablation", "development", out_dir="dev")
    fz = select_variants(p, out)
    Tracker(p).set(9, "VALIDATED", ["reports/frozen_parameters.json", "results/development/ablation_development_summary.csv"])
    return fz


def heldout(p: Project, pipe, test_q: list) -> str:
    from .experiments.selection import heldout_conditions
    from .experiments.runner import run
    Tracker(p).require(5)
    Tracker(p).require(7)
    Tracker(p).require(8)
    Tracker(p).require(9)
    out = run(pipe, test_q, heldout_conditions(p), "heldout_main", "test", out_dir="test")
    recs = read_jsonl(out)
    bad = sum(r.get("status") != "ok" for r in recs)
    Tracker(p).set(10, "VALIDATED" if bad == 0 else "IN PROGRESS", "results/heldout/heldout_main.jsonl",
                   f"{len(recs) - bad} ok, {bad} failed (re-run this cell to retry)")
    return out


def evaluate(p: Project, pipe) -> dict:
    from .evaluation import ragas_eval, triple_relevance
    from .retrieval.encoders import e5
    cand = candidates_dict(p)
    gen = str(p.path("test", "heldout_main.jsonl"))
    rv = ragas_eval.run(p, llm(p), e5(), gen, cand, None, "ragas_heldout.jsonl")
    tv = triple_relevance.run(p, llm(p), [gen], cand, pipe.kg.by_id, pipe.passages)
    return {"ragas_cost_usd": cost_of(rv, "evaluation_cost_usd"), "triple_relevance_cost_usd": cost_of(tv)}


def analyse(p: Project) -> dict:
    from .benchmark import freeze
    from .evaluation.triple_relevance import load as load_tr
    from .reporting.figures import make_all
    from .statistics import analysis as A
    gq, ge, dev, test = freeze.load(p)
    gen = [str(p.path("test", "heldout_main.jsonl"))]
    ev = [str(p.path("test", "ragas_heldout.jsonl"))]
    runs = A.per_run_table(gen, ev, ge, set(passages_dict(p, True)), load_tr(p))
    runs.to_csv(p.path("test", "per_run_log.csv"), index=False)
    pq = A.per_question_table(runs)
    pq.to_csv(p.path("test", "per_question_log.csv"), index=False)  # the consolidated log of Sec. 3.7
    tests = A.run_families(p, pq)
    desc = A.descriptive(pq)
    qc = A.quality_cost(desc)
    comp = A.budget_compliance(runs)
    devsum = pd.read_csv(p.path("dev", "ablation_development_summary.csv"))
    kgs = read_json(p.path("reports", "kg_statistics.json"))
    extra = {"budget_compliance.csv": comp,
             "kg_relation_types.csv": pd.DataFrame(list(kgs["relation_type_frequencies"].items()), columns=["predicate", "triples"]),
             "question_flow.csv": pd.read_csv(p.path("reports", "question_flow_table.csv")),
             "corpus_statistics.csv": pd.read_csv(p.root / "results" / "corpus_statistics.csv")}
    make_all(p, desc, tests, qc, pq, devsum, extra, gen + ev)
    conf = tests[tests.family == "1_confirmatory"].iloc[0].to_dict()
    Tracker(p).set(11, "VALIDATED", ["artifacts/tables/statistical_tests.csv", "results/results_manifest.json"])
    return {"confirmatory": conf, "n_test_questions": int(pq.question_id.nunique())}


def package(p: Project) -> str:
    import shutil
    envmod.capture(p)
    base = p.root.parent / "kg_rag_thesis_evidence"
    keep = ["reports", "artifacts", "configs", "results", "literature_review", "src", "tests", "notebooks",
            "README.md", "requirements.txt", "requirements.lock.txt", "pyproject.toml"]
    tmp = p.root.parent / "_pkg"
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir()
    for k in keep:
        s = p.root / k
        if s.is_dir():
            shutil.copytree(s, tmp / k)
        elif s.exists():
            shutil.copy2(s, tmp / k)
    zp = shutil.make_archive(str(base), "zip", tmp)  # no raw TeleQnA / TSpec text (licence: IDs + results only)
    shutil.rmtree(tmp)
    Tracker(p).set(12, "VALIDATED", zp)
    return zp
''')

### 2.9 Unit tests
28 tests on **synthetic** data only (nothing here is empirical evidence). Stage 4.1 runs them; they check the chunker,
parsing, the X1–X7 rules, the matched-context budget, graph selection, the statistics against known values and an end-to-end dry run.

In [ ]:
#@title tests/conftest.py
put('tests/conftest.py', r'''"""Synthetic fixtures for UNIT TESTS ONLY. Nothing here is ever written to results/ or used as empirical evidence."""
import shutil
import sys
from pathlib import Path

import numpy as np
import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

from kgrag.core import Project  # noqa: E402


@pytest.fixture
def project(tmp_path):
    shutil.copytree(ROOT / "configs", tmp_path / "configs")
    return Project.load(tmp_path)


def wcount(s: str) -> int:
    return len(s.split())


def make_passage(pid, spec="23501", clause="5.1", text="The AMF shall send the message.", heading="General"):
    return {"passage_id": pid, "specification": f"TS {spec[:2]}.{spec[2:]}", "spec_id": spec, "release": "17",
            "version": "17.10.0", "clause_id": clause, "clause_ids": [clause], "heading": heading, "text": text,
            "token_count": wcount(text), "chunk_flags": []}


class HashEncoder:
    """Deterministic bag-of-words hashing encoder (unit tests only)."""
    name = "fake-hash-encoder"

    def _enc(self, texts):
        M = np.zeros((len(texts), 64), dtype="float32")
        for i, t in enumerate(texts):
            for w in t.lower().split():
                M[i, hash(w) % 64] += 1
        return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)

    def encode_queries(self, texts):
        return self._enc(texts)

    def encode_passages(self, texts, show_progress=False):
        return self._enc(texts)


class LenReranker:
    """Scores by word overlap with the query (unit tests only)."""
    name = "fake-reranker"

    def score(self, query, texts):
        qw = set(query.lower().split())
        return [len(qw & set(t.lower().split())) / (1 + len(qw)) for t in texts], 0.001
''')

In [ ]:
#@title tests/test_data_and_validation.py
put('tests/test_data_and_validation.py', r'''import pandas as pd

from conftest import make_passage, wcount
from kgrag.benchmark.freeze import stratified_split
from kgrag.data.chunker import make_passages, quality_checks, is_spec_ref
from kgrag.data.tspec import version_from_filename
from kgrag.data.teleqna import parse
from kgrag.llm import FakeLLM
from kgrag.validation import verifier as V
from kgrag.validation.audit import cohen_kappa, wilson_ci, stratified_sample
from kgrag.validation.pooling import assert_not_forbidden, pool_for_question
from kgrag.validation.register import near_duplicates

DOC = """3GPP TS 23.501 V17.10.0 (2023-09)
Contents
1 Scope 20
2 References 21
1 Scope
The present document defines the 5G System. The AMF shall support registration.
2 References
The following documents apply.
### 5.1 General
The SMF may select a UPF. See 38.214 \\[19\\] for details.
### 5.1.1 Detail
Tiny.
### 5.2 Registration
The UE shall send a Registration Request to the AMF when it powers on. The AMF should respond.
#### -- *BWP*
ASN.1 field description text.
"""


def test_chunker_structure_and_normative_preservation():
    lines = DOC.splitlines()
    ps, st = make_passages(lines, "23501", "17", "23501-ha0.md", wcount, cap=500, min_body=3)
    assert st["scope_anchor_found"] and st["version"] == "17.10.0"
    clauses = [p["clause_id"] for p in ps]
    assert clauses[0] == "1" and "5.2" in clauses
    assert not any(is_spec_ref(c) for c in clauses)              # 38.214 is a cross-reference, not a heading
    q = quality_checks(ps, DOC)
    assert q["duplicate_ids"] == 0 and q["empty_passages"] == 0
    front = sum(l.count("shall") for l in lines[:5])
    assert q["normative_terms_in_passages"] == q["normative_terms_in_source"] - front
    assert any("BWP" in p["text"] for p in ps)                   # ASN.1 kept inside its parent clause


def test_tiny_sections_merge_forward_within_top_clause():
    ps, _ = make_passages(DOC.splitlines(), "23501", "17", "f.md", wcount, cap=500, min_body=6)
    merged = [p for p in ps if "5.1.1" in p["clause_ids"]]
    assert merged and merged[0]["clause_id"] == "5.2"


def test_version_code_and_teleqna_parse():
    assert version_from_filename("23501-ha0.md") == "17.10.0"
    assert version_from_filename("24501-hc0.md") == "17.12.0"
    rows = parse({"question 1": {"question": "What? [3GPP Release 17]", "option 1": "a", "option 2": "b",
                                 "answer": "option 2: b", "category": "Standards specifications"}})
    assert rows[0]["answer_option_id"] == "2" and rows[0]["answer_parsed"] and rows[0]["source_tag"] == "3GPP Release 17"


def test_pooling_union_and_forbidden_model():
    import numpy as np
    pool = pool_for_question(np.array([3.0, 2, 1, 0]), np.array([0.0, 1, 2, 3]), ["a", "b", "c", "d"], depth=2)
    assert {r["passage_id"] for r in pool} == {"a", "b", "c", "d"}
    try:
        assert_not_forbidden("BAAI/bge-base-en-v1.5")
        raise AssertionError("BGE must be rejected")
    except ValueError:
        pass


def test_quote_check_normalises_layout_only():
    text = "The AMF \\*shall\\* send the | Registration Accept | message to the UE."
    assert V.quote_in_passage("The AMF shall send the Registration Accept message", text)
    assert not V.quote_in_passage("The SMF shall send the Registration Accept message", text)
    assert not V.quote_in_passage("AMF shall", text)  # too short to count as evidence


Q = {"question_id": "question 7", "question": "Which entity sends Registration Accept?", "category": "Standards specifications",
     "options": {"1": "SMF", "2": "AMF"}, "answer_option_id": "2", "explanation": ""}
PASS = {"p1": make_passage("p1", text="After authentication the AMF sends the Registration Accept message to the UE."),
        "p2": make_passage("p2", clause="6.1", text="Unrelated text about the PCF and policy control decisions here.")}
POOL = [{"passage_id": "p1", "sources": ["bm25"], "rrf": 0.1}, {"passage_id": "p2", "sources": ["e5"], "rrf": 0.05}]
CFG = {"window_size": 25, "max_windows": 4, "non_3gpp_tag_pattern": r"\[(IEEE|ITU)[^\]]*\]"}


def responder(quote, entail="entails", verdict="supports", **flags):
    def r(role, prompt):
        if "VERIFIED QUOTATIONS" in prompt:
            return {"entailment": entail, "reason": "x"}
        return {"verdict": verdict, "evidence": [{"passage_id": "p1", "quote": quote}], "ambiguous": False,
                "general_knowledge": False, "out_of_corpus": False, "question_type": "factual_lookup", "reason": "x", **flags}
    return r


def test_verifier_include_requires_quote_and_entailment():
    rec = V.verify_question(FakeLLM(responder("the AMF sends the Registration Accept message to the UE")), Q, POOL, PASS, CFG)
    assert rec["decision"] == "include" and rec["reference_passage_ids"] == ["p1"] and rec["hop_type"] == "single-hop"
    rec = V.verify_question(FakeLLM(responder("the SMF sends the Registration Accept message to the UE")), Q, POOL, PASS, CFG)
    assert rec["reason_code"] == "X7"                                   # hallucinated quotation
    rec = V.verify_question(FakeLLM(responder("the AMF sends the Registration Accept message to the UE", "partial")), Q, POOL, PASS, CFG)
    assert rec["reason_code"] == "X3"                                   # separate entailment check
    rec = V.verify_question(FakeLLM(responder("the AMF sends the Registration Accept message to the UE", ambiguous=True)), Q, POOL, PASS, CFG)
    assert rec["reason_code"] == "X4"
    rec = V.verify_question(FakeLLM(responder("x", verdict="no_support")), Q, POOL, PASS, CFG)
    assert rec["reason_code"] == "X1"


def test_verifier_non3gpp_tag_excluded_without_llm():
    llm = FakeLLM(responder("x"))
    rec = V.verify_question(llm, {**Q, "question": "What is GTS? [IEEE 802.15.4]"}, POOL, PASS, CFG)
    assert rec["reason_code"] == "X2" and not llm.calls


def test_multihop_classification_rule():
    ps = {"a": make_passage("a", clause="5.1"), "b": make_passage("b", clause="5.2"), "c": make_passage("c", spec="38331", clause="5.3")}
    assert V.classify(["a"], ps, "procedure_association") == ("single-hop", "procedure_association")
    assert V.classify(["a", "b"], ps, "factual_lookup") == ("multi-hop", "cross_section_dependency")
    assert V.classify(["a", "c"], ps, "factual_lookup") == ("multi-hop", "multi_hop_reasoning")


def test_near_duplicates_keep_earliest():
    df = pd.DataFrame({"question_id": ["question 10", "question 2", "question 5"],
                       "question": ["What does the AMF send to the UE?", "What does the AMF send to the UE ?", "Totally different item"],
                       "reference_passage_ids": [["p1"], ["p1"], ["p9"]], "answer_key_text": ["x", "x", "y"]})
    assert near_duplicates(df, 0.9) == {"question 10": "question 2"}


def test_split_deterministic_and_stratified():
    df = pd.DataFrame({"question_id": [f"q{i}" for i in range(100)], "question_type": ["a"] * 60 + ["b"] * 40,
                       "hop_type": ["single-hop"] * 100})
    d1, t1 = stratified_split(df, ["question_type", "hop_type"], 0.2, 42)
    d2, _ = stratified_split(df, ["question_type", "hop_type"], 0.2, 42)
    assert d1 == d2 and len(d1) == 20 and not set(d1) & set(t1)
    assert sum(x in d1 for x in df.question_id[:60]) == 12


def test_audit_statistics():
    assert abs(cohen_kappa([1, 1, 0, 0], [1, 1, 0, 0]) - 1) < 1e-12
    assert abs(cohen_kappa([1, 0, 1, 0], [1, 1, 0, 0]) - 0.0) < 1e-12
    lo, hi = wilson_ci(9, 10)
    assert 0.59 < lo < 0.60 and 0.98 < hi < 0.99
    df = pd.DataFrame({"category": ["s"] * 70 + ["o"] * 30, "x": range(100)})
    s = stratified_sample(df, 10, "category", 1)
    assert len(s) == 10 and (s.category == "s").sum() == 7
''')

In [ ]:
#@title tests/test_kg_and_eval.py
put('tests/test_kg_and_eval.py', r'''import numpy as np

from conftest import HashEncoder, make_passage
from kgrag.core import append_jsonl, read_jsonl, write_jsonl
from kgrag.evaluation import ragas_eval
from kgrag.evaluation.triple_relevance import judge_question
from kgrag.knowledge_graph import extract, fidelity, graph
from kgrag.llm import FakeLLM

P1 = make_passage("p1", text="After authentication the AMF sends the Registration Accept message to the UE over N1.")


def test_extract_verify_assemble(project):
    kg_cfg = project.study["knowledge_graph"]
    ext = FakeLLM(lambda r, pr: {"triples": [
        {"subject": "AMF", "subject_type": "NetworkFunction", "predicate": "SENDS", "object": "Registration Accept",
         "object_type": "Message", "quote": "the AMF sends the Registration Accept message to the UE"},
        {"subject": "SMF", "subject_type": "NetworkFunction", "predicate": "SENDS", "object": "PDU Session Accept",
         "object_type": "Message", "quote": "the SMF sends a PDU Session Accept to everyone"},      # quote not in passage
        {"subject": "AMF", "subject_type": "NetworkFunction", "predicate": "LIKES", "object": "UE",
         "object_type": "Other", "quote": "the AMF sends the Registration Accept message to the UE"}]})  # bad predicate
    rec = extract.extract_passage(ext, P1, kg_cfg)
    assert [t["quote_found"] for t in rec["triples"]] == [True, False, True]
    assert [t["schema_valid"] for t in rec["triples"]] == [True, True, False]
    append_jsonl(project.path("interim", "kg_extraction.jsonl"), rec)
    inst = [{**t, "instance_id": f"p1#{t['local_id']}", "passage_id": "p1"} for t in rec["triples"]
            if t["schema_valid"] and t["quote_found"]]
    ver = FakeLLM(lambda r, pr: {"judgements": [{"n": 1, "verdict": "entailed"}]})
    for row in fidelity.verify_passage(ver, P1, inst):
        append_jsonl(project.path("interim", "kg_fidelity.jsonl"), row)
    all_tr, validated, stats = graph.assemble(project, {"p1": P1})
    assert stats["validated_triples"] == 1 and validated[0]["subject"] == "AMF"
    assert stats["instances_quote_verified"] == 2 and stats["nodes"] == 2
    assert graph.load(project).by_id[validated[0]["triple_id"]]["provenance"][0]["passage_id"] == "p1"


def test_ragas_protocol_scores_and_no_imputation():
    def r(role, prompt):
        if prompt.startswith("Break the ANSWER"):
            return {"statements": ["The AMF sends Registration Accept.", "It uses N2."]}
        if "can be directly inferred" in prompt or "attributed to the CONTEXT" in prompt:
            return {"verdicts": [{"n": 1, "verdict": 1}, {"n": 2, "verdict": 0}]}
        if prompt.startswith("Generate"):
            return {"questions": ["Which message does the AMF send?"], "noncommittal": 0}
        return {"verdicts": [{"n": 1, "useful": 0}, {"n": 2, "useful": 1}]}
    q = {"question": "Which message does the AMF send?", "options": {"1": "Reject", "2": "Registration Accept"},
         "answer_option_id": "2", "explanation": "Clause 4.2.2."}
    rec = {"key": "k", "parsed_option": "2", "justification": "The AMF sends it.", "context_text": "[P1] x\n\n[G1] y",
           "context_text_passages_only": "[P1] x", "context_items": ["[P1] x", "[G1] y"]}
    out = ragas_eval.evaluate_record(FakeLLM(r), HashEncoder(), rec, q)
    assert out["faithfulness_full"] == 0.5 and out["faithfulness_passages"] == 0.5
    assert out["context_precision"] == 0.5 and out["context_recall"] == 0.5
    assert 0.99 < out["answer_relevance"] <= 1.0001
    closed = ragas_eval.evaluate_record(FakeLLM(r), HashEncoder(), {**rec, "context_text": "", "context_text_passages_only": "",
                                                                   "context_items": []}, q)
    assert closed["faithfulness_full"] is None and closed["context_precision"] is None  # undefined, never imputed


def test_triple_relevance_one_judgement_per_triple():
    t = {"triple_id": "T1", "subject": "AMF", "predicate": "SENDS", "object": "Registration Accept",
         "provenance": [{"passage_id": "p1"}]}
    q = {"question_id": "q", "question": "?", "options": {"1": "a", "2": "Registration Accept"}, "answer_option_id": "2"}
    rows = judge_question(FakeLLM(lambda r, p: {"judgements": [{"n": 1, "helpful": 1}]}), q, [t], {"p1": P1})
    assert rows[0]["helpful"] == 1 and rows[0]["status"] == "ok"
''')

In [ ]:
#@title tests/test_pipeline_and_stats.py
put('tests/test_pipeline_and_stats.py', r'''import json

import numpy as np
import pytest

from conftest import HashEncoder, LenReranker, make_passage, wcount
from kgrag.core import read_jsonl
from kgrag.evaluation.metrics import citation_metrics, retrieval_metrics, triple_metrics
from kgrag.experiments import runner as R
from kgrag.generation.generator import parse_option
from kgrag.knowledge_graph.graph import KGIndex
from kgrag.knowledge_graph.normalize import build_alias_table, canonical, deduplicate
from kgrag.llm import FakeLLM
from kgrag.reranking.reranker import choose_threshold, select
from kgrag.retrieval.dense import Retriever
from kgrag.statistics.tests import holm, mcnemar_exact, paired_bootstrap_ci, rank_biserial, wilcoxon_paired

# ------------------------------------------------------------------ KG
def test_alias_and_canonical():
    ab = make_passage("ab", clause="3.2", heading="Abbreviations",
                      text="AMF Access and Mobility Management Function\nSMF Session Management Function")
    al = build_alias_table([ab])
    assert al["access and mobility management function"] == "AMF"
    assert canonical("the Access and Mobility Management Function", al) == ("AMF", "AMF")
    assert canonical("Session Management Function (SMF)", al)[0] == "SMF"
    assert canonical("Registration procedure", al)[0] == "registration procedure"


def _inst(i, s, p, o, pid="p1"):
    return {"instance_id": f"{pid}#{i}", "passage_id": pid, "subject": s, "predicate": p, "object": o,
            "subject_type": "NetworkFunction", "object_type": "Message", "quote": "q", "specification": "TS 23.501",
            "clause_id": "5.1", "version": "17.10.0", "release": "17", "fidelity_verdict": "entailed"}


def test_dedup_preserves_provenance():
    tr = deduplicate([_inst(0, "AMF", "SENDS", "Registration Accept"), _inst(1, "the AMF", "SENDS", "registration accept", "p2")], {})
    assert len(tr) == 1 and tr[0]["n_sources"] == 2 and {p["passage_id"] for p in tr[0]["provenance"]} == {"p1", "p2"}


def triple(tid, s, p, o, pid):
    return {"triple_id": tid, "subject": s, "subject_key": s if s.isupper() else s.lower(), "subject_type": "NetworkFunction",
            "predicate": p, "object": o, "object_key": o if o.isupper() else o.lower(), "object_type": "Message",
            "provenance": [{"passage_id": pid, "specification": "TS 23.501", "clause_id": "5.1"}]}


KG = [triple("T1", "AMF", "SENDS", "Registration Accept", "p1"), triple("T2", "UE", "RECEIVES", "Registration Accept", "p1"),
      triple("T3", "SMF", "SELECTS", "UPF", "p3"), triple("T4", "UPF", "STORES", "usage report", "p4"),
      triple("T5", "PCF", "PROVIDES", "policy control", "p2")]


def test_kg_seeds_hops_and_budgets():
    kg = KGIndex(KG, HashEncoder())
    assert kg.seeds("The SMF and the AMF") == {"SMF", "AMF"}
    assert set(kg.expand({"SMF"}, 1)) == {"T3"}
    assert kg.expand({"SMF"}, 2) == {"T3": 1, "T4": 2}
    qv = HashEncoder().encode_queries(["registration accept"])[0]
    s = kg.select(qvec=qv, seed_text="AMF", hops=2, triple_budget=1, token_cap=1000, count=wcount)
    assert len(s["items"]) == 1 and s["stop_reason"] == "triple_budget"
    s = kg.select(qvec=qv, seed_text="AMF", hops=2, triple_budget=10, token_cap=5, count=wcount)
    assert s["items"] == [] and s["stop_reason"] == "token_cap"
    r1 = kg.select(qvec=qv, seed_text="AMF", hops=1, triple_budget=None, token_cap=1000, count=wcount, mode="random", rng_seed=3, n_random=2)
    r2 = kg.select(qvec=qv, seed_text="AMF", hops=1, triple_budget=None, token_cap=1000, count=wcount, mode="random", rng_seed=3, n_random=2)
    assert [i["triple_id"] for i in r1["items"]] == [i["triple_id"] for i in r2["items"]] and len(r1["items"]) == 2


# ------------------------------------------------------------------ reranker
def test_select_and_threshold():
    sc = [{"passage_id": f"p{i}", "rank": i, "rerank_score": s} for i, s in enumerate([0.9, 0.8, 0.2, 0.1, 0.05, 0.01, 0.6])]
    sel = select(sc, 0.5)
    assert [s["passage_id"] for s in sel] == ["p0", "p1", "p6"] and sel[0]["final_rank"] == 1
    assert len(select(sc, 0.99)) == 1  # min one passage
    thr = choose_threshold({"q": sc}, {"q": {"p0", "p1"}})
    assert 0.6 < thr["threshold"] <= 0.8


# ------------------------------------------------------------------ pipeline invariants
WORDS = "registration accept message sent by the AMF to the UE after successful authentication procedure"


@pytest.fixture
def pipe(project):
    ps = {}
    for i in range(12):
        txt = (f"Clause {i}. " + WORDS + ". ") * (1 + i % 3) if i < 6 else f"Clause {i}. The SMF selects a UPF for the PDU session. The PCF provides policy."
        ps[f"p{i}"] = make_passage(f"p{i}", clause=f"5.{i}", text=txt)
    plist = list(ps.values())
    enc = HashEncoder()
    ret = Retriever(enc, plist, enc.encode_passages([p["text"] for p in plist]), top_k=10)
    kg = KGIndex(KG, enc)
    llm = FakeLLM(lambda role, prompt: {"answer_option": "2", "justification": "Because P1.", "citations": ["P1", "G1", "X9"]})
    return R.Pipeline(project, ps, ret, LenReranker(), 0.0, kg, wcount, llm,
                      gold={"q1": {"reference_passage_ids": ["p0"]}}, groups={"q1": "retriever-found"})


QQ = {"question_id": "q1", "question": "Which message does the AMF send after registration?",
      "options": {"1": "Registration Reject", "2": "Registration Accept"}, "answer_option_id": "2",
      "hop_type": "single-hop", "question_type": "factual_lookup"}


def test_c_preserves_b_ranking_and_budget(pipe):
    st = pipe.b_state(QQ)
    b_ctx, _ = pipe.build(QQ, R.Condition("B"), st)
    b_ids = [i.ref for i in b_ctx.passage_items]
    for alloc in (0.1, 0.2, 0.3):
        c_ctx, _ = pipe.build(QQ, R.Condition("C", "matched", 10, 1, alloc), pipe.b_state(QQ))
        ids = [i.ref for i in c_ctx.passage_items]
        assert ids == b_ids[: len(ids)]                                    # in-order prefix, never reordered
        assert c_ctx.checks["passage_prefix_of_B"] and c_ctx.checks["graph_cap_ok"]
        assert c_ctx.evidence_tokens <= c_ctx.T_q * 1.05
        assert c_ctx.graph_tokens <= int(alloc * c_ctx.T_q)
    o_ctx, _ = pipe.build(QQ, R.Condition("C", "overhead", 10, 1, 0.3), pipe.b_state(QQ))
    assert [i.ref for i in o_ctx.passage_items] == b_ids                  # overhead keeps B's full context
    assert o_ctx.evidence_tokens == b_ctx.evidence_tokens + o_ctx.graph_tokens


def test_c_seeds_only_from_b_passages(pipe, monkeypatch):
    seen = {}
    orig = pipe.kg.select

    def spy(**kw):
        seen.setdefault("seed_text", kw["seed_text"])
        return orig(**kw)
    monkeypatch.setattr(pipe.kg, "select", spy)
    st = pipe.b_state(QQ)
    pipe.build(QQ, R.Condition("C", "matched", 10, 1, 0.2), st)
    assert seen["seed_text"] == "\n".join(pipe.passages[i]["text"] for i in st["b_cached_ids"])


def test_a_d_controls_and_closed_book(pipe):
    st = pipe.b_state(QQ)
    a, _ = pipe.build(QQ, R.Condition("A"), st)
    assert a.checks["not_over_budget"]
    d, _ = pipe.build(QQ, R.Condition("D", "matched", 10, 1, 0.2), st)
    assert d.evidence_tokens <= d.T_q and d.graph_tokens <= d.graph_cap and "passages_differ_from_B" in d.checks
    for cfg in ("RANDOM", "UNFILTERED"):
        c, _ = pipe.build(QQ, R.Condition(cfg, "matched", 10, 1, 0.2), st)
        assert c.graph_tokens <= c.graph_cap and c.checks["passage_prefix_of_B"]
    z, _ = pipe.build(QQ, R.Condition("0"), None)
    assert z.items == [] and z.text() == ""


def test_run_is_resumable_and_logs_invariants(pipe, project):
    conds = [R.Condition("0"), R.Condition("B"), R.Condition("C", "matched", 10, 1, 0.2)]
    out = R.run(pipe, [QQ], conds, "unit", "test", [1], "dev", workers=2)
    recs = read_jsonl(out)
    assert len(recs) == 3 and all(r["status"] == "ok" for r in recs)
    c = next(r for r in recs if r["configuration"] == "C")
    assert c["ranking_invariance_ok"] and c["correct"] == 1 and c["budget_checks"]["passage_prefix_of_B"]
    zero = next(r for r in recs if r["configuration"] == "0")
    assert zero["context_text"] == "" and zero["retrieved_passage_ids"] == []
    R.run(pipe, [QQ], conds, "unit", "test", [1], "dev", workers=2)
    assert len(read_jsonl(out)) == 3                                      # nothing recomputed


def test_malformed_answer_is_incorrect():
    assert parse_option({"answer_option": "option 2"}, ["1", "2"]) == "2"
    assert parse_option({"answer_option": "7"}, ["1", "2"]) is None
    assert parse_option(None, ["1", "2"]) is None


# ------------------------------------------------------------------ metrics
def test_metrics():
    m = retrieval_metrics(["a", "b", "c"], ["a", "b"], {"b", "c"})
    assert m["recall_at_1"] == 0 and m["recall_at_3"] == 1 and m["mrr"] == 0.5 and m["full_coverage"] == 0
    labels = {"P1": {"kind": "P", "ref": "a", "source_passage_ids": ["a"]},
              "G1": {"kind": "G", "ref": "T1", "source_passage_ids": ["b"]}}
    c = citation_metrics(["P1", "G1", "Z9"], labels, {"b"})
    assert c["citation_validity"] == 2 / 3 and c["citation_support"] == 1 / 3 and c["graph_citation_support"] == 1.0
    assert citation_metrics([], labels, {"b"})["citation_validity"] is None
    t = triple_metrics(labels, {"b"}, {"a", "b"}, {"T1": 1})
    assert t["provenance_validity"] == 1 and t["gold_linked_precision"] == 1 and t["judged_relevant_precision"] == 1


# ------------------------------------------------------------------ statistics (Appendix 6)
def test_mcnemar_exact_known_value():
    ref = [0] * 6 + [1] * 10
    alt = [1] * 6 + [1] * 10
    r = mcnemar_exact(ref, alt)
    assert (r["b"], r["c"]) == (0, 6) and abs(r["p_raw"] - 0.03125) < 1e-9 and abs(r["diff_pp"] - 37.5) < 1e-9
    assert mcnemar_exact([1, 0], [1, 0])["p_raw"] == 1.0


def test_wilcoxon_edge_cases_and_effect():
    assert wilcoxon_paired([1, 2, 3], [1, 2, 3])["note"] == "all differences zero"
    r = wilcoxon_paired([0.1, np.nan, 0.3, 0.2, 0.5, 0.4, 0.6, 0.2], [0.4, 0.9, 0.6, 0.5, 0.9, 0.8, 0.9, 0.7])
    assert r["n"] == 7 and r["n_excluded_pairs"] == 1 and r["rank_biserial_r"] == 1.0
    assert rank_biserial(np.array([1.0, -1.0])) == 0.0


def test_bootstrap_and_holm():
    ci1 = paired_bootstrap_ci([0, 1, 0, 1, 1], [1, 1, 1, 1, 0], scale=100)
    ci2 = paired_bootstrap_ci([0, 1, 0, 1, 1], [1, 1, 1, 1, 0], scale=100)
    assert ci1 == ci2 and ci1[0] <= 20 <= ci1[1]
    assert holm([0.01, 0.04, 0.03]) == pytest.approx([0.03, 0.06, 0.06])
''')

In [ ]:
#@title tests/test_analysis_e2e.py
put('tests/test_analysis_e2e.py', r'''"""End-to-end smoke test of runner -> per-question table -> all statistical families -> figures (fake models only)."""
import pandas as pd

from conftest import HashEncoder, LenReranker, make_passage, wcount
from kgrag.core import write_json
from kgrag.experiments import runner as R
from kgrag.experiments.selection import heldout_conditions
from kgrag.knowledge_graph.graph import KGIndex
from kgrag.llm import FakeLLM
from kgrag.reporting.figures import make_all
from kgrag.retrieval.dense import Retriever
from kgrag.statistics import analysis as A
from test_pipeline_and_stats import KG, WORDS


def test_full_analysis_path(project):
    ps = {f"p{i}": make_passage(f"p{i}", clause=f"5.{i}", text=(f"Clause {i}. " + WORDS + ". ") * (1 + i % 3))
          for i in range(10)}
    enc = HashEncoder()
    ret = Retriever(enc, list(ps.values()), enc.encode_passages([p["text"] for p in ps.values()]), top_k=8)
    qs = [{"question_id": f"q{i}", "question": f"Which message does the AMF send {i}?", "options": {"1": "Reject", "2": "Accept"},
           "answer_option_id": "2", "hop_type": "single-hop" if i % 2 else "multi-hop", "question_type": "factual_lookup"}
          for i in range(12)]
    gold = {q["question_id"]: {"reference_passage_ids": ["p0"]} for q in qs}
    groups = {q["question_id"]: ("retriever-found" if i % 3 else "independent-only") for i, q in enumerate(qs)}

    def resp(role, prompt):
        return {"answer_option": "2" if "Accept" in prompt and "EVIDENCE:\nNo evidence" not in prompt else "1",
                "justification": "Because of P1.", "citations": ["P1"]}
    pipe = R.Pipeline(project, ps, ret, LenReranker(), 0.0, KGIndex(KG, enc), wcount, FakeLLM(resp), gold, groups)
    write_json(project.path("reports", "rerank_threshold.json"), {"threshold": 0.0})
    write_json(project.path("reports", "frozen_parameters.json"),
               {"selected_variants": [{"triples": 10, "hops": 1, "alloc": 0.2}, {"triples": 5, "hops": 2, "alloc": 0.1}]})
    conds = heldout_conditions(project)
    assert any(c.name == "C_t10_h1_g20_matched" for c in conds) and any(c.config == "D" for c in conds)
    out = R.run(pipe, qs, conds, "heldout_main", "test", [1], "test", workers=2)
    runs = A.per_run_table([out], [], gold, set(ps), {})
    pq = A.per_question_table(runs)
    tests = A.run_families(project, pq)
    assert set(tests.family) >= {"1_confirmatory", "2_variants", "3_controls", "4_placement", "5_subgroups", "7_cost"}
    conf = tests[tests.family == "1_confirmatory"].iloc[0]
    assert conf.adjustment == "none" and conf.n == 12
    desc = A.descriptive(pq)
    qc = A.quality_cost(desc)
    comp = A.budget_compliance(runs)
    assert (comp[comp.condition.str.startswith("C_") & comp.condition.str.endswith("matched")].chk_passage_prefix_of_B == 1).all()
    dev = pd.DataFrame([{"condition": f"c{t}{h}{a}", "accuracy": 0.5 + 0.01 * t, "triples": t, "hops": h, "alloc": a}
                        for t in (5, 10, 20) for h in (1, 2) for a in (0.1, 0.2, 0.3)])
    man = make_all(project, desc, tests, qc, pq, dev, {"budget_compliance.csv": comp}, [out])
    names = {m["artifact"].split("/")[-1] for m in man}
    assert {"accuracy_comparison.png", "quality_cost_pareto.png", "kg_ablation_heatmap.png", "statistical_tests.csv",
            "latency_comparison.png", "retrieval_mrr.csv"} <= names
''')

# Part 3 · Load the code
Creates the package markers, puts `src/` on the import path and loads the project.
`W` is the workflow module (the stage functions); `P` is the project (paths + configuration);
`Tracker(P).table()` prints the 12-gate status. On a fresh run every gate reads **NOT STARTED**.

In [ ]:
# ===== Part 3 | Load project (run every session, after Part 2)
for d in [ROOT/'src'/'kgrag', *[p for p in (ROOT/'src'/'kgrag').iterdir() if p.is_dir()]]:
    (d / '__init__.py').touch()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
import importlib
for m in [m for m in list(sys.modules) if m.startswith('kgrag')]:
    del sys.modules[m]                      # pick up files re-written in Part 2
from kgrag import workflow as W
from kgrag.core import Tracker
P = W.project(ROOT)
print(Tracker(P).table())

| Gate | Title | Status | Note |
|---|---|---|---|
| 1 | Environment and dependencies verified | BLOCKED | see gate1_check.json |
| 2 | Dataset files obtained and versions confirmed | NOT STARTED |  |
| 3 | Corpus preprocessing validated | NOT STARTED |  |
| 4 | Independent answerability verification completed | NOT STARTED |  |
| 5 | Manual audit meets criteria and gold benchmark frozen | NOT STARTED |  |
| 6 | BGE retrieval and cross-encoder reranking validated | NOT STARTED |  |
| 7 | Knowledge Graph extracted and fidelity audited | NOT STARTED |  |
| 8 | Configurations 0-D implemented and invariance checks passed | NOT STARTED |  |
| 9 | Development ablation completed and parameters frozen | NOT STARTED |  |
| 10 | Held-out experimental runs completed | NOT STARTED |  |
| 11 | Metrics and statistical tests validated | NOT STARTED |  |
| 12 | Final artifacts, diagrams and reproducibility package verified | NOT STARTED |  |


# Part 4 · The experiment
Eleven stages. Each shows **what it does**, the **thesis section**, **what it saves** and **what must hold** before the next stage.
Cells that call the paid API print a projected cost on a small pilot first, so you can stop before spending the full amount.

## 4.1 Gate 1 · Environment, tests and model check  *(Sec. 4.3)*
Runs the unit tests, records hardware and library versions, and sends one tiny request to **each of the four Gemini
roles** to prove the key works and every model id is available.

* **Saves:** `reports/environment.json`, `reports/gate1_check.json`, `reports/unit_tests.txt`
* **Must hold:** `tests_passed: True` and every role `ok: True`. If a model shows *model id not listed for this key*, run
  `W.llm(P).list_models()`, pick the right name, edit `configs/models.yaml` (Part 2.1) and re-run Parts 2–3.

## OpenAI local results run · exploratory only
The default OpenAI configuration uses `gpt-4o-mini` and a $5 local API budget cap. In VS Code, run Parts 1–3, run the safe preflight to confirm the Colab/GPU/key/data state, and only then run the next cell **instead of** the full thesis stages. The sampled pipeline uses 50 Release 17 questions, 300 corpus passages and 20 KG-extraction passages; runs B/C on the full sample, 0/A/D on 10 questions, and evaluates five questions. It writes isolated outputs under `data/interim/smoke50_openai/`. All LLM roles share one model, so judgments are not independent; report these as exploratory results only. The local cap is a safeguard, not a substitute for checking OpenAI billing/usage.

In [ ]:
# ===== SAFE PREFLIGHT | no OpenAI requests
import os
import torch
print('Colab runtime:', bool(os.environ.get('COLAB_RELEASE_TAG')))
print('CUDA GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not detected')
print('OpenAI key loaded:', bool(os.environ.get('OPENAI_API_KEY')))
print('Project root:', P.root)
print('TeleQnA available:', P.path('raw_q', 'TeleQnA.json').is_file())
print('Release 17 Markdown files:', len(list(P.path('raw_md').rglob('*.md'))))
print('OpenAI smoke module:', (P.root / 'src' / 'kgrag' / 'smoke.py').is_file())

Colab runtime: True
CUDA GPU: not detected
OpenAI key loaded: False


NameError: name 'P' is not defined

In [ ]:
# ===== FAST OPENAI RESULTS PROFILE | exploratory only
import torch
print('CUDA GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not detected; this run will use CPU')
from kgrag.smoke import run as run_smoke50
smoke50_result = run_smoke50(P, n_questions=20, n_passages=150, n_kg_passages=10, n_eval_questions=5)
display(smoke50_result['summary'])
smoke50_result['report']

In [ ]:
# ===== 4.1 | GATE 1
r = W.gate1(P)
r

{'tests_passed': True,
 'tests_tail': ['28 passed in 26.58s'],
 'role_probes': {'generator': {'model': 'gemini-3.8-flash',
   'ok': True,
   'model_version': 'gemini-3.8-flash',
   'error': None},
  'verifier': {'model': 'gemini-3.5-flash',
   'ok': True,
   'model_version': 'gemini-3.6-flash',
   'error': None},
  'evaluator': {'model': 'gemini-3.1-flash-lite',
   'ok': True,
   'model_version': 'gemini-3.1-flash-lite',
   'error': None},
  'kg_extractor': {'model': 'gemini-3-flash',
   'ok': False,
   'error': 'model id not listed for this key'}},
 'gpu': 'NVIDIA RTX PRO 6000 Blackwell Server Edition'}

## 4.2 Gates 2–3 · Datasets, version check and chunking  *(Sec. 3.3, 4.4)*
1. Downloads **TeleQnA** (password-protected archive) and keeps the Standards-specifications and Standards-overview questions.
2. Downloads **TSpec-LLM**, Release 17, as Markdown and builds a manifest (file, series, release, hash).
3. Compares specification versions with the **official 3GPP archive** (version check only, not a corpus).
4. **Chunks** the corpus clause by clause (chunker v2.4) into `passages.jsonl` and writes a corpus-quality report.

* **Saves:** `data/…`, `results/corpus_statistics.csv`, `reports/corpus_quality_report.md`
* **Must hold:** the quality report shows no unresolved chunking failures (oversize, empty or broken-table passages).

In [ ]:
# ===== 4.2 | GATES 2-3
W.stage3_4(P, os.environ.get('HF_TOKEN'))

3GPP-clean/Rel-17/23_series/23501-ha0.md: reconstructing file:   0%|          |  0.00B / 3.62MB            

3GPP-clean/Rel-17/23_series/23501-ha0.md: downloading bytes:           |  0.00B            

3GPP-clean/Rel-17/23_series/23502-ha0.md: reconstructing file:   0%|          |  0.00B / 2.80MB            

3GPP-clean/Rel-17/23_series/23502-ha0.md: downloading bytes:           |  0.00B            

3GPP-clean/Rel-17/24_series/24501-hc0.md: reconstructing file:   0%|          |  0.00B / 7.32MB            

3GPP-clean/Rel-17/24_series/24501-hc0.md: downloading bytes:           |  0.00B            

3GPP-clean/Rel-17/38_series/38300-h60.md: reconstructing file:   0%|          |  0.00B /  875kB            

3GPP-clean/Rel-17/38_series/38300-h60.md: downloading bytes:           |  0.00B            

3GPP-clean/Rel-17/38_series/38331-h60.md: reconstructing file:   0%|          |  0.00B / 4.47MB            

3GPP-clean/Rel-17/38_series/38331-h60.md: downloading bytes:           |  0.00B            

{'candidates': 3000,
 'download': {'23501': {'status': 'DOWNLOADED',
   'hf_path': '3GPP-clean/Rel-17/23_series/23501-ha0.md',
   'alternatives': ['3GPP-clean/Rel-17/23_series/23501-ha0.md']},
  '23502': {'status': 'DOWNLOADED',
   'hf_path': '3GPP-clean/Rel-17/23_series/23502-ha0.md',
   'alternatives': ['3GPP-clean/Rel-17/23_series/23502-ha0.md']},
  '24501': {'status': 'DOWNLOADED',
   'hf_path': '3GPP-clean/Rel-17/24_series/24501-hc0.md',
   'alternatives': ['3GPP-clean/Rel-17/24_series/24501-hc0.md']},
  '38300': {'status': 'DOWNLOADED',
   'hf_path': '3GPP-clean/Rel-17/38_series/38300-h60.md',
   'alternatives': ['3GPP-clean/Rel-17/38_series/38300-h60.md']},
  '38331': {'status': 'DOWNLOADED',
   'hf_path': '3GPP-clean/Rel-17/38_series/38331-h60.md',
   'alternatives': ['3GPP-clean/Rel-17/38_series/38331-h60.md']}},
 'verified_specs': 5,
 'passages': 13250,
 'corpus_checks_ok': True}

## 4.3 Stage 5a · Independent candidate pooling  *(Sec. 3.3a)*
For every TeleQnA question, finds up to 100 candidate passages with **BM25** and **E5** (top-50 each). The BGE retriever that
is being evaluated is *not* used here, so the benchmark cannot be biased toward it. About 10 minutes on a GPU.

* **Saves:** `data/interim/pools…`

In [ ]:
# ===== 4.3 | Stage 5a POOLING
W.pools(P)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/142 [00:00<?, ?it/s]

3000

## 4.4 Stage 5b · Answerability verification  *(Sec. 3.3b)*
The verifier model reads each question with its candidate passages and decides whether the corpus **supports the keyed
answer**. It must quote the passage verbatim (checked by code), and a *separate* entailment call must agree.
Failures get an exclusion code **X1–X7**.

**4.4a Pilot (30 questions)** prints a **projected total cost**. Look at it before running 4.4b.

In [ ]:
# ===== 4.4a | PILOT: 30 questions + projected cost
W.verify(P, limit=30)

{'verified_ok': 8,
 'errors': 22,
 'cost_usd_so_far': 0.15186599999999997,
 'projected_total_cost_usd': 56.94974999999999,
 'decisions': {'exclude': 8},
 'reason_codes': {'X2': 7, 'X7': 1}}

**4.4b Full run** over all candidate questions. Resumable: if it stops, run the same cell again.

In [ ]:
# ===== 4.4b | FULL verification
W.verify(P)

In [ ]:
import json
from pathlib import Path
from kgrag.core import read_jsonl

# Define paths using Project instance
p_obj = W.project(ROOT)
candidates_path = p_obj.path('interim', 'teleqna_candidates.jsonl')
pilot_results_path = p_obj.path('interim', 'verifier_results.jsonl')

# Count total candidates
total_candidates = len(read_jsonl(candidates_path))

# Calculate average latency from pilot
pilot_runs = [r for r in read_jsonl(pilot_results_path) if r.get('status') == 'ok']

if pilot_runs:
    total_calls = sum(len(r.get('calls', [])) for r in pilot_runs)
    total_latency = sum(sum(call.get('latency_s', 0) for call in r.get('calls', [])) for r in pilot_runs)
    avg_latency_per_question = total_latency / len(pilot_runs) if len(pilot_runs) > 0 else 0

    workers = p_obj.study['experiment']['workers']
    estimated_seq_time_s = total_candidates * avg_latency_per_question
    estimated_parallel_time_m = (estimated_seq_time_s / workers) / 60

    print(f"--- Verification Execution Estimate ---")
    print(f"Total Questions to Verify: {total_candidates}")
    print(f"Parallel Workers Configured: {workers}")
    print(f"Average Latency Per Question (Pilot): {avg_latency_per_question:.2f} seconds")
    print(f"Estimated Total Parallel Run Time: {estimated_parallel_time_m:.2f} minutes")
else:
    print("No successful pilot runs found in verifier_results.jsonl to calculate average latency.")

## 4.5 Stage 5c · Register and **your blind question audit**  *(Sec. 3.3c)*
Builds the inclusion/exclusion register, the question-flow table (Table 3.4) and a **blind audit sheet** of 200 questions
(100 included + 100 excluded) in random order, with the verifier's verdict hidden.

In [ ]:
# ===== 4.5 | Register, flow table, blind audit sheet
W.register_and_audit(P)

### ⛔ STOP: human step (≈ 200 rows)
Open `MyDrive/kg-rag-thesis/reports/audit/question_audit_round1_BLIND.csv` in Google Sheets. **Do not open the
`_KEY_do_not_open` file.** For each row read the question, keyed option, TeleQnA explanation and the candidate evidence
(check the official 3GPP text if needed), then fill in:

* `manual_answerable`: **Y** if the Release-17 text supports the keyed option, else **N**
* `manual_passage_ids`: only for an *excluded* question you mark **Y**: the supporting passage id(s)
* `manual_hop_type` / `manual_question_type`: only for such newly included rows

Download as CSV with the **same file name**, replace the old file, then run Stage 4.6.

## 4.6 Gate 5 · Score the audit, freeze the benchmark, split  *(Sec. 3.3c, 3.3d)*
Scores your audit against the verifier: **precision ≥ 0.90** and **Cohen's κ ≥ 0.70** are required. Applies your corrections,
removes near-duplicates (X6), **freezes the gold set** and makes a stratified **20 % development / 80 % held-out** split,
recording SHA-256 hashes. After this point the test questions are not touched until Stage 4.10.

* **Saves:** `data/processed/FROZEN.json`, `reports/inclusion_exclusion_log.csv` (IDs and codes only)
* **Must hold:** both thresholds met. Otherwise the gate shows BLOCKED and the notebook asks for a second audit round (`round_no=2`).

In [ ]:
# ===== 4.6 | GATE 5
W.score_and_freeze(P, round_no=1)

## 4.7 Gate 6 · Retrieval index and reranker threshold  *(Sec. 3.4, 4.7)*
Embeds all passages with BGE, builds the FAISS index, runs top-30 retrieval for every question, marks which gold passages
the retriever actually found, and fixes the **reranker threshold using development questions only**.

* **Saves:** `data/processed/index…`, retrieval and rerank logs, the frozen threshold

In [ ]:
# ===== 4.7 | GATE 6
W.retrieval_and_threshold(P)

## 4.8 Gate 7 · Build the knowledge graph  *(Sec. 3.4 Comp. 1, 3.6, 4.6)*
**4.8a Pilot** extracts triples from 60 passages (fixed prompt, 15 predicates, 10 entity types) and prints a projected cost.

In [ ]:
# ===== 4.8a | KG extraction pilot (60 passages)
W.kg_extract(P, limit=60)

**4.8b Full extraction, fidelity verification and graph.** Extracts from all passages; every triple must carry its source
passage and a quote that really occurs in it. A *different* model then verifies each triple. Only entailed triples enter the
NetworkX graph. A **blind triple-audit sheet** (200 triples) is written.

In [ ]:
# ===== 4.8b | KG full extraction + fidelity + graph + audit sheet
print(W.kg_extract(P))
W.kg_verify_and_build(P)

### ⛔ STOP: human step (200 rows)
Open `reports/audit/triple_audit_BLIND.csv`. For each triple set `manual_entailed` = **Y** if the source passage text
entails the triple (subject, relation, object **and direction** all correct), else **N**. Save it with the same file name,
then run Stage 4.9.

## 4.9 Gate 7 · Score the triple audit  *(Sec. 3.6)*
Reports the precision of the graph's triples with a Wilson 95 % confidence interval and the agreement between you and the
verifier. This is the number to quote for KG quality in Chapter 5.

In [ ]:
# ===== 4.9 | GATE 7: score the triple audit
W.kg_audit_score(P)

## 4.10 Gates 8–10 · The controlled experiment  *(Sec. 3.5, 4.8–4.9)*
Everything is now frozen except the configuration under test.

**4.10a Build the pipeline and smoke-test it (Gate 8).** Calibrates the local tokenizer against Gemini's `count_tokens`,
runs a small number of development questions through all configurations, checks the **invariance rules** (B's ranking is
identical inside C; A, B, C use the same prompt and decoding) and runs a **determinism pilot** (same question, repeated).
If the model is non-deterministic, the protocol switches to three runs per question.

Check `underfill_share_C` in the output: the share of matched-context C prompts that fall below 0.95 × T_q because unused
graph allocation is *not* refilled (the thesis rule). Report it with the results.

In [ ]:
# ===== 4.10a | GATE 8: build pipeline + smoke test + determinism pilot
pipe, DEV, TEST = W.pipeline(P)
print(len(DEV), 'development /', len(TEST), 'test questions')
W.gate8_smoke(P, pipe, DEV)

**4.10b Development ablation (Gate 9).** Runs B and the **18 KG variants** (triples {5,10,20} × hops {1,2} × allocation
{10,20,30 %}) on the **development questions only**, ranks them, and **freezes** the top-3 plus the default
(10 triples, 1 hop, 20 %). The test questions are not used for any choice.

In [ ]:
# ===== 4.10b | GATE 9: development ablation + selection lock
W.dev_ablation(P, pipe, DEV)

**4.10c Held-out test run (Gate 10).** Runs configurations 0, A, B, the frozen C variants (matched and overhead-inclusive),
D, RANDOM and UNFILTERED on the held-out questions. Resumable; it refuses to start unless Gates 5, 7, 8 and 9 are validated.

In [ ]:
# ===== 4.10c | GATE 10: held-out test run
W.heldout(P, pipe, TEST)

## 4.11 Stage 16 · Evaluation  *(Sec. 3.6)*
Scores the test-set answers with the **RAGAs-protocol** (faithfulness over full context and over passages only, answer
relevance, context precision/recall; the metric definitions are re-implemented with fixed Gemini evaluator prompts rather
than imported from the `ragas` package, so report them as *RAGAs-protocol scores*) and judges the relevance of every appended triple.

In [ ]:
# ===== 4.11 | RAGAs-protocol evaluation + judged triple relevance
W.evaluate(P, pipe)

## 4.12 Gate 11 · Statistics, figures and tables  *(Sec. 3.7, Appendix 6)*
Builds the per-question log and runs every pre-specified test family:

| Family | Comparison |
|---|---|
| 1 | **Confirmatory:** B vs C (default, matched-context), unadjusted |
| 2–7 | Secondary families, Holm-adjusted within each (A vs B, C variants, matched vs overhead, D vs C, controls, …) |

Tests: exact McNemar (accuracy), Wilcoxon (continuous measures), paired bootstrap (10,000 resamples, seed 42), with effect
sizes and 95 % CIs. Also draws every Chapter 5 figure/table and the quality–cost (Pareto) plot, and writes
`results/results_manifest.json` that lists every number with its source file.

In [ ]:
# ===== 4.12 | GATE 11
res = W.analyse(P)
import pandas as pd
display(pd.read_csv(ROOT/'artifacts/tables/statistical_tests.csv')[['family','comparison','metric','n','b','c','diff_pp','mean_diff','ci95_low','ci95_high','p_raw','p_adjusted','interpretation']])
res

# Part 5 · Evidence package  *(Gate 12)*
Records the final environment and saves a ZIP containing the code, configs, reports, figures, tables and result logs —
**without** raw TeleQnA or TSpec-LLM text (licences). Cite figures and tables in Chapter 5 only from this package.

In [ ]:
# ===== Part 5 | GATE 12
print(W.package(P))
print(Tracker(P).table())

---
### Notes for the reader
* **Reproducibility:** all seeds, prompts, model ids (with the exact `model_version` returned by the API), thresholds, hashes and
  prices are written next to the results. Prices in `configs/models.yaml` are dated assumptions; confirm them before quoting cost.
* **What is human-validated:** the 200-question answerability audit and the 200-triple fidelity audit (Sections 4.5, 4.8).
* **What is not claimed:** no model is trained; the RAGAs numbers are *protocol-equivalent* re-implementations; results apply to
  3GPP Release 17 and the TeleQnA Standards questions only.

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU attached")